In [ ]:
# @title
# CELL 0 — Repair Colab numerical-package compatibility

!pip install -q --force-reinstall --no-cache-dir \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "pandas==2.2.2" \
    "scikit-learn==1.5.2"

print("Packages repaired. Restarting the runtime now...")

import os
os.kill(os.getpid(), 9)

In [ ]:
# ============================================================
# CELL 1 — Environment, Drive, GPU, and project paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

# Install only the NLP-specific packages.
# Do not upgrade NumPy, SciPy, pandas, or scikit-learn.
!pip install -q \
    "transformers>=4.44,<5" \
    "datasets>=2.20,<4" \
    "accelerate>=0.33,<2" \
    evaluate \
    seqeval

import os
import sys
import platform
from pathlib import Path

import torch
import transformers
import datasets
import pandas as pd
import numpy as np
import scipy
import sklearn

# Preserve the path used in your original collapse notebook.
BASE_DIR = Path("/content/drive/MyDrive/MDPI Information/DETECH2026")

# Existing DETECH data paths.
TASKA_TRAIN_DIR = BASE_DIR / "data" / "taskA" / "training"
TASKA_TRAIN_ABS_DIR = TASKA_TRAIN_DIR / "abstracts"
TASKA_TRAIN_CSV_DIR = TASKA_TRAIN_DIR / "csv"

# New journal-experiment paths.
JOURNAL_DIR = BASE_DIR / "journal_experiments" / "taskA_holdout"
SPLIT_DIR = JOURNAL_DIR / "splits"
MODEL_OUTPUT_DIR = JOURNAL_DIR / "models"
PREDICTION_DIR = JOURNAL_DIR / "predictions"
CACHE_DIR = JOURNAL_DIR / "cache"
RESULTS_DIR = JOURNAL_DIR / "results"

# Create output directories only.
for path in [
    JOURNAL_DIR,
    SPLIT_DIR,
    MODEL_OUTPUT_DIR,
    PREDICTION_DIR,
    CACHE_DIR,
    RESULTS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"

print("=" * 72)
print("RUNTIME")
print("=" * 72)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Platform:", platform.platform())

print("\n" + "=" * 72)
print("GPU")
print("=" * 72)

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    DEVICE = torch.device("cuda")

    print("CUDA available: Yes")
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GB")
else:
    DEVICE = torch.device("cpu")

    print("CUDA available: No")
    print(
        "Select Runtime > Change runtime type > "
        "T4 GPU or A100 GPU before training."
    )

print("\n" + "=" * 72)
print("PROJECT PATHS")
print("=" * 72)

paths_to_check = {
    "BASE_DIR": BASE_DIR,
    "Task A training directory": TASKA_TRAIN_DIR,
    "Task A abstract directory": TASKA_TRAIN_ABS_DIR,
    "Task A CSV directory": TASKA_TRAIN_CSV_DIR,
    "Journal experiment directory": JOURNAL_DIR,
}

for label, path in paths_to_check.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{status:7} | {label}: {path}")

assert BASE_DIR.exists(), (
    f"\nBASE_DIR was not found:\n{BASE_DIR}\n"
    "Edit BASE_DIR to match the folder in Google Drive."
)

assert TASKA_TRAIN_DIR.exists(), (
    f"\nTask A training directory was not found:\n{TASKA_TRAIN_DIR}"
)

assert TASKA_TRAIN_ABS_DIR.exists(), (
    f"\nTask A abstract directory was not found:\n{TASKA_TRAIN_ABS_DIR}"
)

assert TASKA_TRAIN_CSV_DIR.exists(), (
    f"\nTask A CSV directory was not found:\n{TASKA_TRAIN_CSV_DIR}"
)

print("\nCell 1 completed successfully.")

In [ ]:
# ============================================================
# CELL 2 — Inventory DETECH Task A training files
# ============================================================

from pathlib import Path
import pandas as pd

DOMAINS = {
    "parkinson": {
        "prefix": "pa_",
        "abstract_dir": TASKA_TRAIN_ABS_DIR / "parkinson",
    },
    "mental_health": {
        "prefix": "mh_",
        "abstract_dir": TASKA_TRAIN_ABS_DIR / "mental_health",
    },
}

print("=" * 76)
print("CSV FILES")
print("=" * 76)

csv_files = sorted(TASKA_TRAIN_CSV_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files were found in:\n{TASKA_TRAIN_CSV_DIR}"
    )

csv_inventory = {}

for csv_path in csv_files:
    try:
        df_preview = pd.read_csv(csv_path, nrows=5)

        csv_inventory[csv_path.name] = {
            "path": csv_path,
            "columns": list(df_preview.columns),
        }

        print(f"\nFILE: {csv_path.name}")
        print("Columns:", list(df_preview.columns))
        print("Preview rows:", len(df_preview))
        display(df_preview)

    except Exception as exc:
        print(f"\nCOULD NOT READ: {csv_path.name}")
        print(type(exc).__name__, str(exc))


print("\n" + "=" * 76)
print("ABSTRACT FILES")
print("=" * 76)

abstract_inventory = {}

for domain, config in DOMAINS.items():
    abstract_dir = config["abstract_dir"]

    if not abstract_dir.exists():
        print(f"\nMISSING | {domain}: {abstract_dir}")
        abstract_inventory[domain] = []
        continue

    txt_files = sorted(abstract_dir.glob("*.txt"))
    abstract_inventory[domain] = txt_files

    print(f"\nDOMAIN: {domain}")
    print("Directory:", abstract_dir)
    print("Number of .txt files:", len(txt_files))

    if txt_files:
        print("First five filenames:")
        for path in txt_files[:5]:
            print("  ", path.name)

        # Inspect one document without printing the entire abstract.
        example_path = txt_files[0]
        example_text = example_path.read_text(
            encoding="utf-8",
            errors="replace"
        )

        print("Example file:", example_path.name)
        print("Character count:", len(example_text))
        print("First 300 characters:")
        print(repr(example_text[:300]))


print("\n" + "=" * 76)
print("EXPECTED GOLD FILE CHECK")
print("=" * 76)

EXPECTED_GOLD_PATHS = {
    "parkinson": (
        TASKA_TRAIN_CSV_DIR /
        "taskA_parkinson_terms_train.csv"
    ),
    "mental_health": (
        TASKA_TRAIN_CSV_DIR /
        "taskA_mental_health_terms_train.csv"
    ),
}

gold_schema = {}

for domain, gold_path in EXPECTED_GOLD_PATHS.items():
    print(f"\nDOMAIN: {domain}")
    print("Expected file:", gold_path.name)
    print("Exists:", gold_path.exists())

    if gold_path.exists():
        gold_df = pd.read_csv(gold_path)

        gold_schema[domain] = {
            "path": gold_path,
            "columns": list(gold_df.columns),
            "rows": len(gold_df),
            "unique_articles": (
                gold_df["num_article"].nunique()
                if "num_article" in gold_df.columns
                else None
            ),
        }

        print("Columns:", list(gold_df.columns))
        print("Rows:", len(gold_df))

        if "num_article" in gold_df.columns:
            print(
                "Unique annotated articles:",
                gold_df["num_article"].nunique()
            )

        if "term" in gold_df.columns:
            print("Unique term strings:", gold_df["term"].nunique())
            print(
                "Single-word rows:",
                int(
                    gold_df["term"]
                    .astype(str)
                    .str.split()
                    .str.len()
                    .eq(1)
                    .sum()
                )
            )
            print(
                "Multiword rows:",
                int(
                    gold_df["term"]
                    .astype(str)
                    .str.split()
                    .str.len()
                    .gt(1)
                    .sum()
                )
            )

        print("Missing values by column:")
        print(gold_df.isna().sum())

        print("First five records:")
        display(gold_df.head())


print("\n" + "=" * 76)
print("BASIC CONSISTENCY CHECK")
print("=" * 76)

for domain, config in DOMAINS.items():
    txt_files = abstract_inventory.get(domain, [])
    gold_info = gold_schema.get(domain)

    if not gold_info:
        print(f"{domain}: gold file unavailable")
        continue

    gold_df = pd.read_csv(gold_info["path"])
    gold_ids = set(gold_df["num_article"].astype(str))

    # Extract article IDs from filenames such as pa_123.txt or mh_123.txt.
    prefix = config["prefix"]
    file_ids = {
        path.stem[len(prefix):]
        for path in txt_files
        if path.stem.startswith(prefix)
    }

    gold_without_text = sorted(gold_ids - file_ids)
    text_without_gold = sorted(file_ids - gold_ids)

    print(f"\nDOMAIN: {domain}")
    print("Gold article IDs:", len(gold_ids))
    print("Abstract file IDs:", len(file_ids))
    print("Gold IDs without abstract file:", len(gold_without_text))
    print("Abstract files without gold annotation:", len(text_without_gold))

    if gold_without_text:
        print("First missing abstract IDs:", gold_without_text[:10])

    if text_without_gold:
        print("First abstracts without gold:", text_without_gold[:10])


print("\nCell 2 completed successfully.")

In [ ]:
# ============================================================
# CELL 3 — Document populations and gold-term alignment audit
# ============================================================

import re
from collections import Counter
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# 1. Load manifests and gold term files
# ------------------------------------------------------------

PA_MANIFEST_PATH = TASKA_TRAIN_CSV_DIR / "taskA_parkinson_train.csv"
MH_MANIFEST_PATH = TASKA_TRAIN_CSV_DIR / "taskA_mental_health_train.csv"

PA_GOLD_PATH = TASKA_TRAIN_CSV_DIR / "taskA_parkinson_terms_train.csv"
MH_GOLD_PATH = TASKA_TRAIN_CSV_DIR / "taskA_mental_health_terms_train.csv"

pa_manifest = pd.read_csv(PA_MANIFEST_PATH)
mh_manifest = pd.read_csv(MH_MANIFEST_PATH)

gold_pa = pd.read_csv(PA_GOLD_PATH)
gold_mh = pd.read_csv(MH_GOLD_PATH)

for df in [pa_manifest, mh_manifest, gold_pa, gold_mh]:
    df["num_article"] = df["num_article"].astype(str)

gold_pa["term"] = gold_pa["term"].astype(str)
gold_mh["term"] = gold_mh["term"].astype(str)


# ------------------------------------------------------------
# 2. Read abstracts
# ------------------------------------------------------------

def read_taska_abstract(domain, article_id):
    prefix = "pa_" if domain == "parkinson" else "mh_"
    abstract_dir = (
        TASKA_TRAIN_ABS_DIR / "parkinson"
        if domain == "parkinson"
        else TASKA_TRAIN_ABS_DIR / "mental_health"
    )

    path = abstract_dir / f"{prefix}{article_id}.txt"

    return path.read_text(
        encoding="utf-8",
        errors="replace"
    ).replace("\r\n", "\n")


def abstract_file_ids(domain):
    prefix = "pa_" if domain == "parkinson" else "mh_"
    abstract_dir = (
        TASKA_TRAIN_ABS_DIR / "parkinson"
        if domain == "parkinson"
        else TASKA_TRAIN_ABS_DIR / "mental_health"
    )

    return {
        path.stem[len(prefix):]
        for path in abstract_dir.glob(f"{prefix}*.txt")
    }


# ------------------------------------------------------------
# 3. Compare manifest, abstract, and gold populations
# ------------------------------------------------------------

print("=" * 78)
print("DOCUMENT POPULATION COMPARISON")
print("=" * 78)

population_objects = {
    "parkinson": {
        "manifest": set(pa_manifest["num_article"]),
        "abstracts": abstract_file_ids("parkinson"),
        "gold": set(gold_pa["num_article"]),
    },
    "mental_health": {
        "manifest": set(mh_manifest["num_article"]),
        "abstracts": abstract_file_ids("mental_health"),
        "gold": set(gold_mh["num_article"]),
    },
}

for domain, sets in population_objects.items():
    manifest_ids = sets["manifest"]
    abstract_ids = sets["abstracts"]
    gold_ids = sets["gold"]

    print(f"\nDOMAIN: {domain}")
    print("Manifest rows:", len(manifest_ids))
    print("Abstract files:", len(abstract_ids))
    print("Articles represented in gold:", len(gold_ids))

    print(
        "Manifest ∩ gold:",
        len(manifest_ids & gold_ids)
    )
    print(
        "Manifest without gold:",
        len(manifest_ids - gold_ids)
    )
    print(
        "Gold without manifest:",
        len(gold_ids - manifest_ids)
    )
    print(
        "Abstracts outside manifest:",
        len(abstract_ids - manifest_ids)
    )

    print(
        "First manifest IDs:",
        sorted(manifest_ids, key=lambda x: int(x) if x.isdigit() else x)[:15]
    )

    if manifest_ids - gold_ids:
        print(
            "First manifest IDs without gold:",
            sorted(
                manifest_ids - gold_ids,
                key=lambda x: int(x) if x.isdigit() else x
            )[:15]
        )

    if gold_ids - manifest_ids:
        print(
            "First gold IDs outside manifest:",
            sorted(
                gold_ids - manifest_ids,
                key=lambda x: int(x) if x.isdigit() else x
            )[:15]
        )


# ------------------------------------------------------------
# 4. Count literal surface occurrences
# ------------------------------------------------------------

def literal_spans(text, term, ignore_case=False):
    flags = re.IGNORECASE if ignore_case else 0

    return [
        (match.start(), match.end())
        for match in re.finditer(
            re.escape(term),
            text,
            flags=flags
        )
    ]


def audit_gold_alignment(gold_df, domain):
    records = []

    for article_id, sub in gold_df.groupby("num_article", sort=False):
        text = read_taska_abstract(domain, article_id)

        # Gold may contain duplicate rows for repeated occurrences.
        gold_counts = Counter(sub["term"].astype(str))

        for term, gold_row_count in gold_counts.items():
            exact_spans = literal_spans(
                text,
                term,
                ignore_case=False
            )

            insensitive_spans = literal_spans(
                text,
                term,
                ignore_case=True
            )

            records.append({
                "domain": domain,
                "num_article": article_id,
                "term": term,
                "gold_row_count": gold_row_count,
                "exact_text_occurrences": len(exact_spans),
                "case_insensitive_occurrences": len(insensitive_spans),
                "exact_enough_for_gold_rows": (
                    len(exact_spans) >= gold_row_count
                ),
                "case_insensitive_enough_for_gold_rows": (
                    len(insensitive_spans) >= gold_row_count
                ),
            })

    return pd.DataFrame(records)


alignment_pa = audit_gold_alignment(gold_pa, "parkinson")
alignment_mh = audit_gold_alignment(gold_mh, "mental_health")

alignment_all = pd.concat(
    [alignment_pa, alignment_mh],
    ignore_index=True
)


# ------------------------------------------------------------
# 5. Alignment summary
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("GOLD-TERM ALIGNMENT SUMMARY")
print("=" * 78)

for domain, audit_df in [
    ("parkinson", alignment_pa),
    ("mental_health", alignment_mh),
]:
    total = len(audit_df)

    exact_ok = int(
        audit_df["exact_enough_for_gold_rows"].sum()
    )

    insensitive_ok = int(
        audit_df["case_insensitive_enough_for_gold_rows"].sum()
    )

    print(f"\nDOMAIN: {domain}")
    print("Unique document–surface-form pairs:", total)
    print(
        f"Exact case-sensitive alignment: "
        f"{exact_ok}/{total} ({exact_ok / total:.2%})"
    )
    print(
        f"Case-insensitive alignment: "
        f"{insensitive_ok}/{total} ({insensitive_ok / total:.2%})"
    )

    print(
        "Pairs with more gold rows than exact text occurrences:",
        int((~audit_df["exact_enough_for_gold_rows"]).sum())
    )

    print(
        "Pairs still unmatched case-insensitively:",
        int(
            (~audit_df["case_insensitive_enough_for_gold_rows"]).sum()
        )
    )


# ------------------------------------------------------------
# 6. Show problematic examples
# ------------------------------------------------------------

problematic = alignment_all[
    ~alignment_all["exact_enough_for_gold_rows"]
].copy()

problematic = problematic.sort_values(
    [
        "domain",
        "case_insensitive_enough_for_gold_rows",
        "num_article",
        "term",
    ]
)

print("\n" + "=" * 78)
print("FIRST 30 NON-EXACT ALIGNMENT CASES")
print("=" * 78)

display(problematic.head(30))


# ------------------------------------------------------------
# 7. Save the audit for later reproducibility
# ------------------------------------------------------------

ALIGNMENT_AUDIT_PATH = RESULTS_DIR / "gold_alignment_audit.csv"
alignment_all.to_csv(ALIGNMENT_AUDIT_PATH, index=False)

print("\nSaved alignment audit to:")
print(ALIGNMENT_AUDIT_PATH)

print("\nCell 3 completed successfully.")

In [ ]:
# ============================================================
# CELL 4 — Gold occurrence multiplicity and overlap audit
# ============================================================

import re
from collections import Counter

import pandas as pd


# ------------------------------------------------------------
# 1. Compare gold-row counts with literal occurrence counts
# ------------------------------------------------------------

alignment_all["exact_count_equal"] = (
    alignment_all["gold_row_count"]
    == alignment_all["exact_text_occurrences"]
)

alignment_all["surplus_text_occurrences"] = (
    alignment_all["exact_text_occurrences"]
    - alignment_all["gold_row_count"]
)

print("=" * 78)
print("GOLD ROWS VERSUS LITERAL TEXT OCCURRENCES")
print("=" * 78)

for domain, audit_df in alignment_all.groupby("domain"):
    equal_count = int(audit_df["exact_count_equal"].sum())
    total = len(audit_df)
    unequal = total - equal_count

    print(f"\nDOMAIN: {domain}")
    print(
        f"Exact count equality: "
        f"{equal_count}/{total} ({equal_count / total:.2%})"
    )
    print("Pairs with surplus text occurrences:", unequal)

    if unequal:
        print(
            "Total surplus text occurrences:",
            int(
                audit_df.loc[
                    ~audit_df["exact_count_equal"],
                    "surplus_text_occurrences"
                ].sum()
            )
        )


surplus_cases = alignment_all[
    ~alignment_all["exact_count_equal"]
].copy()

surplus_cases = surplus_cases.sort_values(
    [
        "surplus_text_occurrences",
        "domain",
        "num_article",
        "term",
    ],
    ascending=[False, True, True, True]
)

print("\n" + "=" * 78)
print("FIRST 30 SURPLUS-OCCURRENCE CASES")
print("=" * 78)

display(surplus_cases.head(30))


# ------------------------------------------------------------
# 2. Reconstruct provisional exact spans
#
# This is unambiguous only where the number of gold rows equals
# the number of literal surface occurrences.
# ------------------------------------------------------------

def reconstruct_provisional_spans(gold_df, domain):
    records = []

    for article_id, sub in gold_df.groupby(
        "num_article",
        sort=False
    ):
        text = read_taska_abstract(domain, article_id)
        gold_counts = Counter(sub["term"].astype(str))

        for term, gold_count in gold_counts.items():
            matches = list(
                re.finditer(
                    re.escape(term),
                    text
                )
            )

            # At this stage, preserve every gold row.
            # If the text has surplus occurrences, mark the mapping
            # as ambiguous rather than silently treating it as certain.
            for occurrence_index, match in enumerate(
                matches[:gold_count]
            ):
                records.append({
                    "domain": domain,
                    "num_article": str(article_id),
                    "term": term,
                    "start_char": match.start(),
                    "end_char": match.end(),
                    "occurrence_index": occurrence_index,
                    "text_occurrence_count": len(matches),
                    "gold_row_count": gold_count,
                    "alignment_ambiguous": (
                        len(matches) != gold_count
                    ),
                })

    return pd.DataFrame(records)


provisional_pa = reconstruct_provisional_spans(
    gold_pa,
    "parkinson"
)

provisional_mh = reconstruct_provisional_spans(
    gold_mh,
    "mental_health"
)

provisional_gold = pd.concat(
    [provisional_pa, provisional_mh],
    ignore_index=True
)


# ------------------------------------------------------------
# 3. Validate reconstructed text exactly
# ------------------------------------------------------------

validation_failures = []

for (
    domain,
    article_id
), sub in provisional_gold.groupby(
    ["domain", "num_article"]
):
    text = read_taska_abstract(domain, article_id)

    for row in sub.itertuples(index=False):
        recovered = text[row.start_char:row.end_char]

        if recovered != row.term:
            validation_failures.append({
                "domain": domain,
                "num_article": article_id,
                "term": row.term,
                "recovered": recovered,
                "start_char": row.start_char,
                "end_char": row.end_char,
            })


print("\n" + "=" * 78)
print("PROVISIONAL SPAN RECONSTRUCTION")
print("=" * 78)

print("Expected Parkinson gold rows:", len(gold_pa))
print("Reconstructed Parkinson spans:", len(provisional_pa))

print("Expected mental-health gold rows:", len(gold_mh))
print("Reconstructed mental-health spans:", len(provisional_mh))

print(
    "Provisional spans marked ambiguous:",
    int(provisional_gold["alignment_ambiguous"].sum())
)

print(
    "Exact substring validation failures:",
    len(validation_failures)
)

if validation_failures:
    display(pd.DataFrame(validation_failures).head(20))


# ------------------------------------------------------------
# 4. Measure exact nested/overlapping gold spans
# ------------------------------------------------------------

def classify_span_relations(sub):
    spans = list(
        sub[
            ["start_char", "end_char", "term"]
        ].itertuples(index=False, name=None)
    )

    overlapping_rows = set()
    nested_rows = set()
    relation_pairs = []

    for i in range(len(spans)):
        s1, e1, t1 = spans[i]

        for j in range(i + 1, len(spans)):
            s2, e2, t2 = spans[j]

            overlaps = max(s1, s2) < min(e1, e2)

            if not overlaps:
                continue

            identical = s1 == s2 and e1 == e2

            nested = (
                not identical
                and (
                    (s1 <= s2 and e2 <= e1)
                    or
                    (s2 <= s1 and e1 <= e2)
                )
            )

            overlapping_rows.update([i, j])

            if nested:
                nested_rows.update([i, j])

            relation_pairs.append({
                "span_1": t1,
                "span_2": t2,
                "nested": nested,
                "identical": identical,
            })

    return {
        "overlap_pairs": len(relation_pairs),
        "nested_pairs": sum(
            pair["nested"]
            for pair in relation_pairs
        ),
        "rows_in_overlap": len(overlapping_rows),
        "rows_in_nested_relation": len(nested_rows),
    }


relation_records = []

for (
    domain,
    article_id
), sub in provisional_gold.groupby(
    ["domain", "num_article"]
):
    stats = classify_span_relations(
        sub.reset_index(drop=True)
    )

    relation_records.append({
        "domain": domain,
        "num_article": article_id,
        **stats,
    })

relation_df = pd.DataFrame(relation_records)

print("\n" + "=" * 78)
print("GOLD OVERLAP AND NESTEDNESS")
print("=" * 78)

for domain, sub in relation_df.groupby("domain"):
    print(f"\nDOMAIN: {domain}")
    print("Documents:", len(sub))
    print(
        "Documents with overlapping gold spans:",
        int((sub["overlap_pairs"] > 0).sum())
    )
    print(
        "Documents with nested gold spans:",
        int((sub["nested_pairs"] > 0).sum())
    )
    print(
        "Total overlapping span pairs:",
        int(sub["overlap_pairs"].sum())
    )
    print(
        "Total nested span pairs:",
        int(sub["nested_pairs"].sum())
    )


# ------------------------------------------------------------
# 5. Save provisional resources
# ------------------------------------------------------------

PROVISIONAL_GOLD_PATH = (
    RESULTS_DIR /
    "provisional_gold_spans_before_ambiguity_resolution.csv"
)

RELATION_AUDIT_PATH = (
    RESULTS_DIR /
    "gold_overlap_nestedness_by_document.csv"
)

provisional_gold.to_csv(
    PROVISIONAL_GOLD_PATH,
    index=False
)

relation_df.to_csv(
    RELATION_AUDIT_PATH,
    index=False
)

print("\nSaved provisional span file to:")
print(PROVISIONAL_GOLD_PATH)

print("\nSaved overlap audit to:")
print(RELATION_AUDIT_PATH)

print("\nCell 4 completed successfully.")

In [ ]:
# ============================================================
# CELL 5 — Boundary-aware weak span supervision
# ============================================================

import re
from collections import Counter

import pandas as pd


# ------------------------------------------------------------
# 1. Boundary-aware exact matcher
# ------------------------------------------------------------

def is_word_character(char):
    """
    Treat Unicode letters, digits, and underscore as word characters.
    """
    return bool(char) and (char.isalnum() or char == "_")


def boundary_valid_exact_spans(text, term):
    """
    Return all case-sensitive literal occurrences of `term` whose
    outer boundaries do not cut through an alphanumeric word.

    Internal punctuation, spaces, hyphens, Greek characters,
    acronyms, and symbols are preserved exactly.
    """
    if not term:
        return []

    spans = []

    for match in re.finditer(re.escape(term), text):
        start_char = match.start()
        end_char = match.end()

        previous_char = (
            text[start_char - 1]
            if start_char > 0
            else ""
        )

        next_char = (
            text[end_char]
            if end_char < len(text)
            else ""
        )

        # If the term starts with a word character, it cannot
        # begin in the middle of another word.
        valid_left = not (
            is_word_character(term[0])
            and is_word_character(previous_char)
        )

        # If the term ends with a word character, it cannot
        # end in the middle of another word.
        valid_right = not (
            is_word_character(term[-1])
            and is_word_character(next_char)
        )

        if valid_left and valid_right:
            spans.append((start_char, end_char))

    return spans


# ------------------------------------------------------------
# 2. Collapse duplicate gold rows to unique document–term pairs
# ------------------------------------------------------------

def prepare_unique_gold_pairs(gold_df, domain):
    unique_pairs = (
        gold_df[
            ["num_article", "doi", "term"]
        ]
        .copy()
        .drop_duplicates(
            subset=["num_article", "term"]
        )
        .reset_index(drop=True)
    )

    unique_pairs["num_article"] = (
        unique_pairs["num_article"].astype(str)
    )

    unique_pairs["term"] = (
        unique_pairs["term"].astype(str)
    )

    unique_pairs["domain"] = domain

    return unique_pairs


unique_gold_pa = prepare_unique_gold_pairs(
    gold_pa,
    "parkinson"
)

unique_gold_mh = prepare_unique_gold_pairs(
    gold_mh,
    "mental_health"
)

print("=" * 78)
print("UNIQUE DOCUMENT–TERM PAIRS")
print("=" * 78)

print(
    "Parkinson:",
    len(unique_gold_pa),
    "unique document–term pairs"
)

print(
    "Mental health:",
    len(unique_gold_mh),
    "unique document–term pairs"
)


# ------------------------------------------------------------
# 3. Generate all boundary-valid occurrences
# ------------------------------------------------------------

def reconstruct_all_gold_occurrences(
    unique_gold_df,
    domain
):
    span_records = []
    unmatched_records = []
    pair_summaries = []

    for article_id, sub in unique_gold_df.groupby(
        "num_article",
        sort=False
    ):
        text = read_taska_abstract(domain, article_id)

        for row in sub.itertuples(index=False):
            term = str(row.term)

            spans = boundary_valid_exact_spans(
                text,
                term
            )

            pair_summaries.append({
                "domain": domain,
                "num_article": str(article_id),
                "doi": row.doi,
                "term": term,
                "n_boundary_valid_occurrences": len(spans),
            })

            if not spans:
                unmatched_records.append({
                    "domain": domain,
                    "num_article": str(article_id),
                    "doi": row.doi,
                    "term": term,
                })

                continue

            for occurrence_index, (
                start_char,
                end_char
            ) in enumerate(spans):

                span_records.append({
                    "domain": domain,
                    "num_article": str(article_id),
                    "doi": row.doi,
                    "term": term,
                    "start_char": start_char,
                    "end_char": end_char,
                    "occurrence_index": occurrence_index,
                })

    return (
        pd.DataFrame(span_records),
        pd.DataFrame(unmatched_records),
        pd.DataFrame(pair_summaries),
    )


(
    gold_spans_pa,
    unmatched_pa,
    pair_summary_pa,
) = reconstruct_all_gold_occurrences(
    unique_gold_pa,
    "parkinson"
)

(
    gold_spans_mh,
    unmatched_mh,
    pair_summary_mh,
) = reconstruct_all_gold_occurrences(
    unique_gold_mh,
    "mental_health"
)

gold_spans = pd.concat(
    [gold_spans_pa, gold_spans_mh],
    ignore_index=True
)

pair_summary = pd.concat(
    [pair_summary_pa, pair_summary_mh],
    ignore_index=True
)

unmatched = pd.concat(
    [unmatched_pa, unmatched_mh],
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Validate every reconstructed substring
# ------------------------------------------------------------

validation_failures = []

for (
    domain,
    article_id
), sub in gold_spans.groupby(
    ["domain", "num_article"]
):
    text = read_taska_abstract(
        domain,
        article_id
    )

    for row in sub.itertuples(index=False):
        recovered = text[
            row.start_char:row.end_char
        ]

        if recovered != row.term:
            validation_failures.append({
                "domain": domain,
                "num_article": article_id,
                "term": row.term,
                "recovered": recovered,
                "start_char": row.start_char,
                "end_char": row.end_char,
            })


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BOUNDARY-AWARE ALIGNMENT SUMMARY")
print("=" * 78)

for domain, summary_df in pair_summary.groupby("domain"):
    print(f"\nDOMAIN: {domain}")
    print(
        "Unique document–term pairs:",
        len(summary_df)
    )
    print(
        "Pairs with at least one boundary-valid occurrence:",
        int(
            (
                summary_df[
                    "n_boundary_valid_occurrences"
                ] > 0
            ).sum()
        )
    )
    print(
        "Pairs with no boundary-valid occurrence:",
        int(
            (
                summary_df[
                    "n_boundary_valid_occurrences"
                ] == 0
            ).sum()
        )
    )
    print(
        "Total reconstructed training spans:",
        int(
            summary_df[
                "n_boundary_valid_occurrences"
            ].sum()
        )
    )
    print(
        "Median occurrences per document–term pair:",
        summary_df[
            "n_boundary_valid_occurrences"
        ].median()
    )
    print(
        "Maximum occurrences for one pair:",
        summary_df[
            "n_boundary_valid_occurrences"
        ].max()
    )


print("\nTotal unmatched document–term pairs:", len(unmatched))
print("Substring validation failures:", len(validation_failures))

if len(unmatched):
    print("\nFirst unmatched pairs:")
    display(unmatched.head(30))

if validation_failures:
    print("\nFirst validation failures:")
    display(
        pd.DataFrame(
            validation_failures
        ).head(20)
    )


# ------------------------------------------------------------
# 6. Inspect previously pathological short terms
# ------------------------------------------------------------

short_term_audit = pair_summary[
    pair_summary["term"].str.len() <= 3
].sort_values(
    "n_boundary_valid_occurrences",
    ascending=False
)

print("\n" + "=" * 78)
print("SHORT-TERM AUDIT")
print("=" * 78)

display(short_term_audit.head(30))


# ------------------------------------------------------------
# 7. Save definitive weak-supervision resources
# ------------------------------------------------------------

GOLD_SPANS_PATH = (
    RESULTS_DIR /
    "taskA_boundary_valid_weak_gold_spans.csv"
)

UNIQUE_GOLD_PA_PATH = (
    RESULTS_DIR /
    "taskA_parkinson_unique_document_term_pairs.csv"
)

UNIQUE_GOLD_MH_PATH = (
    RESULTS_DIR /
    "taskA_mental_health_unique_document_term_pairs.csv"
)

PAIR_SUMMARY_PATH = (
    RESULTS_DIR /
    "taskA_gold_pair_occurrence_summary.csv"
)

gold_spans.to_csv(
    GOLD_SPANS_PATH,
    index=False
)

unique_gold_pa.to_csv(
    UNIQUE_GOLD_PA_PATH,
    index=False
)

unique_gold_mh.to_csv(
    UNIQUE_GOLD_MH_PATH,
    index=False
)

pair_summary.to_csv(
    PAIR_SUMMARY_PATH,
    index=False
)

print("\nSaved weakly supervised spans to:")
print(GOLD_SPANS_PATH)

print("\nCell 5 completed successfully.")

In [ ]:
# ============================================================
# CELL 6 — Balanced 70/15/15 document-level holdout
# ============================================================

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Document-level annotation statistics
# ------------------------------------------------------------

def light_term_normalize(term):
    term = str(term).strip().casefold()
    term = re.sub(r"\s+", " ", term)
    return term


def term_word_count(term):
    return len(str(term).split())


def is_boundary_substring(short_term, long_term):
    """
    Determine whether one normalized gold term is embedded within
    another at non-alphanumeric boundaries.
    """
    short_term = light_term_normalize(short_term)
    long_term = light_term_normalize(long_term)

    if short_term == long_term:
        return False

    for match in re.finditer(
        re.escape(short_term),
        long_term
    ):
        start = match.start()
        end = match.end()

        previous_char = (
            long_term[start - 1]
            if start > 0
            else ""
        )

        next_char = (
            long_term[end]
            if end < len(long_term)
            else ""
        )

        valid_left = not (
            short_term[0].isalnum()
            and previous_char.isalnum()
        )

        valid_right = not (
            short_term[-1].isalnum()
            and next_char.isalnum()
        )

        if valid_left and valid_right:
            return True

    return False


def count_nested_term_pairs(terms):
    normalized_terms = sorted(
        set(light_term_normalize(term) for term in terms),
        key=len
    )

    nested_pairs = 0

    for i, short_term in enumerate(normalized_terms):
        for long_term in normalized_terms[i + 1:]:
            if len(short_term) >= len(long_term):
                continue

            if is_boundary_substring(
                short_term,
                long_term
            ):
                nested_pairs += 1

    return nested_pairs


def build_document_statistics(
    unique_gold_df,
    domain
):
    records = []

    for article_id, sub in unique_gold_df.groupby(
        "num_article",
        sort=False
    ):
        terms = sub["term"].astype(str).tolist()
        word_counts = [
            term_word_count(term)
            for term in terms
        ]

        single_word_count = sum(
            count == 1
            for count in word_counts
        )

        multiword_count = sum(
            count > 1
            for count in word_counts
        )

        nested_pair_count = count_nested_term_pairs(
            terms
        )

        records.append({
            "domain": domain,
            "num_article": str(article_id),
            "doi": str(sub["doi"].iloc[0]),
            "gold_term_count": len(terms),
            "single_word_count": single_word_count,
            "multiword_count": multiword_count,
            "multiword_ratio": (
                multiword_count / len(terms)
                if terms
                else 0.0
            ),
            "mean_term_words": float(
                np.mean(word_counts)
            ),
            "max_term_words": int(
                np.max(word_counts)
            ),
            "nested_pair_count": nested_pair_count,
            "has_nested_terms": int(
                nested_pair_count > 0
            ),
        })

    return pd.DataFrame(records)


doc_stats_pa = build_document_statistics(
    unique_gold_pa,
    "parkinson"
)

doc_stats_mh = build_document_statistics(
    unique_gold_mh,
    "mental_health"
)

doc_stats = pd.concat(
    [doc_stats_pa, doc_stats_mh],
    ignore_index=True
)

print("=" * 78)
print("DOCUMENT STATISTICS")
print("=" * 78)

display(
    doc_stats.groupby("domain")[
        [
            "gold_term_count",
            "multiword_ratio",
            "mean_term_words",
            "max_term_words",
            "nested_pair_count",
            "has_nested_terms",
        ]
    ].agg(["mean", "std", "min", "max"])
)


# ------------------------------------------------------------
# 2. Candidate split generator
# ------------------------------------------------------------

SPLIT_PROPORTIONS = {
    "train": 0.70,
    "development": 0.15,
    "test": 0.15,
}

BALANCE_FEATURES = [
    "gold_term_count",
    "multiword_ratio",
    "mean_term_words",
    "max_term_words",
    "nested_pair_count",
    "has_nested_terms",
]


def assign_domain_split(
    domain_df,
    seed,
    domain_offset=0
):
    """
    Randomly assign exact document counts within one domain.
    """
    rng = np.random.default_rng(
        seed + domain_offset
    )

    indices = domain_df.index.to_numpy().copy()
    rng.shuffle(indices)

    n_docs = len(indices)
    n_train = round(
        SPLIT_PROPORTIONS["train"] * n_docs
    )
    n_dev = round(
        SPLIT_PROPORTIONS["development"] * n_docs
    )

    train_indices = indices[:n_train]
    dev_indices = indices[
        n_train:n_train + n_dev
    ]
    test_indices = indices[
        n_train + n_dev:
    ]

    assignments = {}

    for index in train_indices:
        assignments[index] = "train"

    for index in dev_indices:
        assignments[index] = "development"

    for index in test_indices:
        assignments[index] = "test"

    return assignments


# ------------------------------------------------------------
# 3. Balance objective
# ------------------------------------------------------------

def split_balance_score(
    stats_df,
    assignments
):
    evaluated = stats_df.copy()

    evaluated["split"] = (
        evaluated.index.map(assignments)
    )

    score = 0.0
    details = []

    for domain, domain_df in evaluated.groupby(
        "domain"
    ):
        for feature in BALANCE_FEATURES:
            global_mean = domain_df[feature].mean()
            global_std = domain_df[feature].std(ddof=0)

            if global_std < 1e-12:
                continue

            for split_name in [
                "train",
                "development",
                "test",
            ]:
                split_mean = domain_df.loc[
                    domain_df["split"] == split_name,
                    feature
                ].mean()

                standardized_difference = (
                    split_mean - global_mean
                ) / global_std

                score += (
                    standardized_difference ** 2
                )

                details.append({
                    "domain": domain,
                    "feature": feature,
                    "split": split_name,
                    "global_mean": global_mean,
                    "split_mean": split_mean,
                    "standardized_difference":
                        standardized_difference,
                })

    return score, details


# ------------------------------------------------------------
# 4. Search deterministic candidate seeds using metadata only
# ------------------------------------------------------------

best_seed = None
best_score = float("inf")
best_assignments = None
best_details = None

for candidate_seed in range(1000):
    assignments = {}

    assignments.update(
        assign_domain_split(
            doc_stats[
                doc_stats["domain"] == "parkinson"
            ],
            seed=candidate_seed,
            domain_offset=0,
        )
    )

    assignments.update(
        assign_domain_split(
            doc_stats[
                doc_stats["domain"] == "mental_health"
            ],
            seed=candidate_seed,
            domain_offset=100_000,
        )
    )

    score, details = split_balance_score(
        doc_stats,
        assignments
    )

    if score < best_score:
        best_seed = candidate_seed
        best_score = score
        best_assignments = assignments
        best_details = details


doc_splits = doc_stats.copy()

doc_splits["split"] = (
    doc_splits.index.map(best_assignments)
)

assert doc_splits["split"].notna().all()


# ------------------------------------------------------------
# 5. Validate split exclusivity
# ------------------------------------------------------------

for domain, domain_df in doc_splits.groupby(
    "domain"
):
    split_sets = {
        split_name: set(
            domain_df.loc[
                domain_df["split"] == split_name,
                "num_article"
            ]
        )
        for split_name in [
            "train",
            "development",
            "test",
        ]
    }

    assert split_sets["train"].isdisjoint(
        split_sets["development"]
    )

    assert split_sets["train"].isdisjoint(
        split_sets["test"]
    )

    assert split_sets["development"].isdisjoint(
        split_sets["test"]
    )

    assert (
        split_sets["train"]
        | split_sets["development"]
        | split_sets["test"]
    ) == set(domain_df["num_article"])


# ------------------------------------------------------------
# 6. Report selected split
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("SELECTED HOLDOUT SPLIT")
print("=" * 78)

print("Selected metadata-balance seed:", best_seed)
print("Balance objective:", best_score)

print("\nDocument counts:")
print(
    doc_splits.groupby(
        ["domain", "split"]
    ).size()
)

print("\nGold term counts:")
print(
    doc_splits.groupby(
        ["domain", "split"]
    )["gold_term_count"].sum()
)

print("\nMean balance characteristics:")
balance_table = (
    doc_splits.groupby(
        ["domain", "split"]
    )[BALANCE_FEATURES]
    .mean()
    .round(3)
)

display(balance_table)

balance_details_df = pd.DataFrame(
    best_details
)

print(
    "\nLargest absolute standardized difference:",
    balance_details_df[
        "standardized_difference"
    ].abs().max()
)

print("\nTen largest balance differences:")
display(
    balance_details_df.assign(
        absolute_difference=balance_details_df[
            "standardized_difference"
        ].abs()
    )
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    .head(10)
)


# ------------------------------------------------------------
# 7. Save split files and metadata
# ------------------------------------------------------------

DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_70_15_15.csv"
)

SPLIT_METADATA_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_metadata.json"
)

BALANCE_DETAILS_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_balance.csv"
)

doc_splits.to_csv(
    DOC_SPLIT_PATH,
    index=False
)

balance_details_df.to_csv(
    BALANCE_DETAILS_PATH,
    index=False
)

split_metadata = {
    "split_type": "document_level_holdout",
    "proportions": SPLIT_PROPORTIONS,
    "candidate_seeds_evaluated": 1000,
    "selected_seed": int(best_seed),
    "balance_objective": float(best_score),
    "balance_features": BALANCE_FEATURES,
    "eligible_documents": {
        "parkinson": int(len(doc_stats_pa)),
        "mental_health": int(len(doc_stats_mh)),
        "total": int(len(doc_stats)),
    },
    "excluded_unannotated_parkinson_abstracts": 40,
    "unmatched_gold_pairs_retained_for_evaluation": int(
        len(unmatched)
    ),
}

with open(
    SPLIT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        split_metadata,
        file,
        indent=2
    )

print("\nSaved document split to:")
print(DOC_SPLIT_PATH)

print("\nSaved split metadata to:")
print(SPLIT_METADATA_PATH)

print("\nCell 6 completed successfully.")

In [ ]:
# ============================================================
# CELL 7 — DOI and exact-text leakage audit
# ============================================================

import hashlib
import re

import pandas as pd


# ------------------------------------------------------------
# 1. Reload the saved split as the authoritative version
# ------------------------------------------------------------

DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_70_15_15.csv"
)

doc_splits = pd.read_csv(
    DOC_SPLIT_PATH,
    dtype={"num_article": str}
)

required_columns = {
    "domain",
    "num_article",
    "doi",
    "split",
}

missing_columns = (
    required_columns - set(doc_splits.columns)
)

assert not missing_columns, (
    f"Missing split columns: {missing_columns}"
)

assert len(doc_splits) == 540, (
    f"Expected 540 annotated documents, "
    f"found {len(doc_splits)}"
)


# ------------------------------------------------------------
# 2. Normalize DOI and text for leakage detection
# ------------------------------------------------------------

def normalize_doi(doi):
    doi = str(doi).strip().casefold()

    doi = re.sub(
        r"^https?://(?:dx\.)?doi\.org/",
        "",
        doi
    )

    doi = re.sub(
        r"^doi:\s*",
        "",
        doi
    )

    return doi.strip()


def normalize_text_for_hash(text):
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    text = re.sub(r"\s+", " ", text)
    return text.strip().casefold()


def text_sha256(text):
    normalized = normalize_text_for_hash(text)

    return hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()


document_records = []

for row in doc_splits.itertuples(index=False):
    text = read_taska_abstract(
        row.domain,
        row.num_article
    )

    document_records.append({
        "domain": row.domain,
        "num_article": str(row.num_article),
        "doi": str(row.doi),
        "doi_normalized": normalize_doi(row.doi),
        "split": row.split,
        "character_count": len(text),
        "text_hash": text_sha256(text),
    })

document_audit = pd.DataFrame(
    document_records
)


# ------------------------------------------------------------
# 3. Identify repeated DOI groups
# ------------------------------------------------------------

doi_groups = (
    document_audit[
        document_audit["doi_normalized"] != ""
    ]
    .groupby("doi_normalized")
    .agg(
        n_records=("doi_normalized", "size"),
        n_domains=("domain", "nunique"),
        n_splits=("split", "nunique"),
        domains=(
            "domain",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
        splits=(
            "split",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
        article_records=(
            "num_article",
            lambda values: " | ".join(
                map(str, values)
            )
        ),
    )
    .reset_index()
)

duplicate_dois = doi_groups[
    doi_groups["n_records"] > 1
].copy()

leaking_dois = duplicate_dois[
    duplicate_dois["n_splits"] > 1
].copy()


# ------------------------------------------------------------
# 4. Identify repeated exact-text groups
# ------------------------------------------------------------

text_groups = (
    document_audit
    .groupby("text_hash")
    .agg(
        n_records=("text_hash", "size"),
        n_domains=("domain", "nunique"),
        n_splits=("split", "nunique"),
        domains=(
            "domain",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
        splits=(
            "split",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
        article_records=(
            "num_article",
            lambda values: " | ".join(
                map(str, values)
            )
        ),
        dois=(
            "doi_normalized",
            lambda values: " | ".join(
                sorted(set(values))
            )
        ),
    )
    .reset_index()
)

duplicate_texts = text_groups[
    text_groups["n_records"] > 1
].copy()

leaking_texts = duplicate_texts[
    duplicate_texts["n_splits"] > 1
].copy()


# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print("=" * 78)
print("DOCUMENT LEAKAGE AUDIT")
print("=" * 78)

print("Documents audited:", len(document_audit))
print(
    "Unique normalized DOIs:",
    document_audit["doi_normalized"].nunique()
)
print(
    "Unique normalized text hashes:",
    document_audit["text_hash"].nunique()
)

print("\nRepeated DOI groups:", len(duplicate_dois))
print(
    "Repeated DOI groups crossing splits:",
    len(leaking_dois)
)

print("\nRepeated exact-text groups:", len(duplicate_texts))
print(
    "Repeated exact-text groups crossing splits:",
    len(leaking_texts)
)


if len(duplicate_dois):
    print("\nAll repeated DOI groups:")
    display(duplicate_dois)

if len(leaking_dois):
    print("\nWARNING — DOI groups crossing splits:")
    display(leaking_dois)

if len(duplicate_texts):
    print("\nAll repeated exact-text groups:")
    display(duplicate_texts)

if len(leaking_texts):
    print("\nWARNING — exact texts crossing splits:")
    display(leaking_texts)


# ------------------------------------------------------------
# 6. Check DOI/text consistency
# ------------------------------------------------------------

doi_to_hash_counts = (
    document_audit[
        document_audit["doi_normalized"] != ""
    ]
    .groupby("doi_normalized")[
        "text_hash"
    ]
    .nunique()
)

doi_with_different_text = doi_to_hash_counts[
    doi_to_hash_counts > 1
]

hash_to_doi_counts = (
    document_audit.groupby("text_hash")[
        "doi_normalized"
    ]
    .nunique()
)

text_with_different_doi = hash_to_doi_counts[
    hash_to_doi_counts > 1
]

print(
    "\nRepeated DOI associated with different text:",
    len(doi_with_different_text)
)

print(
    "Identical text associated with different DOI:",
    len(text_with_different_doi)
)


# ------------------------------------------------------------
# 7. Save audit
# ------------------------------------------------------------

DOCUMENT_LEAKAGE_AUDIT_PATH = (
    SPLIT_DIR /
    "taskA_document_leakage_audit.csv"
)

document_audit.to_csv(
    DOCUMENT_LEAKAGE_AUDIT_PATH,
    index=False
)

print("\nSaved leakage audit to:")
print(DOCUMENT_LEAKAGE_AUDIT_PATH)

if len(leaking_dois) == 0 and len(leaking_texts) == 0:
    print(
        "\nPASS: No DOI or exact-text group crosses "
        "the train/development/test boundary."
    )
else:
    print(
        "\nACTION REQUIRED: The split must be repaired "
        "before neural training."
    )

print("\nCell 7 completed successfully.")

In [ ]:
# ============================================================
# CELL 8 — DOI-group-aware balanced holdout split
# ============================================================

import json

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Prepare DOI groups
# ------------------------------------------------------------

grouped_stats = doc_stats.copy()

grouped_stats["doi_normalized"] = (
    grouped_stats["doi"].apply(normalize_doi)
)

# All current records have a DOI. This fallback protects against
# any future record without one.
grouped_stats["document_group"] = np.where(
    grouped_stats["doi_normalized"] != "",
    "doi::" + grouped_stats["doi_normalized"],
    (
        "doc::"
        + grouped_stats["domain"]
        + "::"
        + grouped_stats["num_article"]
    )
)

print("=" * 78)
print("DOI GROUPS")
print("=" * 78)

print("Documents:", len(grouped_stats))
print(
    "Publication/document groups:",
    grouped_stats["document_group"].nunique()
)

group_sizes = (
    grouped_stats.groupby(
        "document_group"
    ).size()
)

print(
    "Groups containing more than one record:",
    int((group_sizes > 1).sum())
)

print(
    "Largest group size:",
    int(group_sizes.max())
)


# ------------------------------------------------------------
# 2. Group-level assignment
# ------------------------------------------------------------

group_ids = np.array(
    sorted(
        grouped_stats[
            "document_group"
        ].unique()
    )
)

n_groups = len(group_ids)

n_train_groups = round(
    SPLIT_PROPORTIONS["train"] * n_groups
)

n_development_groups = round(
    SPLIT_PROPORTIONS["development"] * n_groups
)

target_document_counts = {
    ("parkinson", "train"): 129,
    ("parkinson", "development"): 28,
    ("parkinson", "test"): 27,
    ("mental_health", "train"): 249,
    ("mental_health", "development"): 53,
    ("mental_health", "test"): 54,
}


def assign_groups(candidate_seed):
    rng = np.random.default_rng(
        candidate_seed
    )

    shuffled_groups = group_ids.copy()
    rng.shuffle(shuffled_groups)

    train_groups = set(
        shuffled_groups[:n_train_groups]
    )

    development_groups = set(
        shuffled_groups[
            n_train_groups:
            n_train_groups
            + n_development_groups
        ]
    )

    test_groups = set(
        shuffled_groups[
            n_train_groups
            + n_development_groups:
        ]
    )

    assignments = {}

    for group_id in train_groups:
        assignments[group_id] = "train"

    for group_id in development_groups:
        assignments[group_id] = "development"

    for group_id in test_groups:
        assignments[group_id] = "test"

    return assignments


# ------------------------------------------------------------
# 3. Combined count-and-covariate objective
# ------------------------------------------------------------

def grouped_split_score(
    stats_df,
    group_assignments
):
    evaluated = stats_df.copy()

    evaluated["split"] = (
        evaluated["document_group"]
        .map(group_assignments)
    )

    if evaluated["split"].isna().any():
        return float("inf"), None

    score = 0.0
    detail_records = []

    # Strong but not absolute penalty for deviations from the
    # intended number of documents per domain and split.
    for (
        domain,
        split_name
    ), target_count in target_document_counts.items():

        observed_count = len(
            evaluated[
                (evaluated["domain"] == domain)
                & (
                    evaluated["split"]
                    == split_name
                )
            ]
        )

        relative_count_difference = (
            observed_count - target_count
        ) / target_count

        score += (
            50.0
            * relative_count_difference ** 2
        )

    # Balance annotation characteristics within each domain.
    for domain, domain_df in evaluated.groupby(
        "domain"
    ):
        for feature in BALANCE_FEATURES:
            global_mean = domain_df[
                feature
            ].mean()

            global_std = domain_df[
                feature
            ].std(ddof=0)

            if global_std < 1e-12:
                continue

            for split_name in [
                "train",
                "development",
                "test",
            ]:
                split_values = domain_df.loc[
                    domain_df["split"]
                    == split_name,
                    feature
                ]

                if len(split_values) == 0:
                    return float("inf"), None

                split_mean = (
                    split_values.mean()
                )

                standardized_difference = (
                    split_mean - global_mean
                ) / global_std

                score += (
                    standardized_difference ** 2
                )

                detail_records.append({
                    "domain": domain,
                    "feature": feature,
                    "split": split_name,
                    "global_mean": global_mean,
                    "split_mean": split_mean,
                    "standardized_difference":
                        standardized_difference,
                })

    return score, detail_records


# ------------------------------------------------------------
# 4. Search candidate group-aware splits
# ------------------------------------------------------------

best_group_seed = None
best_group_score = float("inf")
best_group_assignments = None
best_group_details = None

N_CANDIDATE_SEEDS = 10_000

for candidate_seed in range(
    N_CANDIDATE_SEEDS
):
    assignments = assign_groups(
        candidate_seed
    )

    score, details = grouped_split_score(
        grouped_stats,
        assignments
    )

    if score < best_group_score:
        best_group_seed = candidate_seed
        best_group_score = score
        best_group_assignments = assignments
        best_group_details = details


grouped_splits = grouped_stats.copy()

grouped_splits["split"] = (
    grouped_splits["document_group"]
    .map(best_group_assignments)
)

assert grouped_splits["split"].notna().all()


# ------------------------------------------------------------
# 5. Leakage validation
# ------------------------------------------------------------

group_split_counts = (
    grouped_splits.groupby(
        "document_group"
    )["split"].nunique()
)

assert group_split_counts.max() == 1, (
    "At least one DOI group crosses split boundaries."
)

doi_split_counts = (
    grouped_splits[
        grouped_splits[
            "doi_normalized"
        ] != ""
    ]
    .groupby("doi_normalized")[
        "split"
    ]
    .nunique()
)

assert doi_split_counts.max() == 1, (
    "At least one normalized DOI crosses split boundaries."
)


# ------------------------------------------------------------
# 6. Report repaired split
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("GROUP-AWARE HOLDOUT SPLIT")
print("=" * 78)

print(
    "Selected group-aware seed:",
    best_group_seed
)

print(
    "Combined balance objective:",
    best_group_score
)

print("\nDocument counts:")
print(
    grouped_splits.groupby(
        ["domain", "split"]
    ).size()
)

print("\nPublication-group counts:")
print(
    grouped_splits[
        [
            "document_group",
            "domain",
            "split",
        ]
    ]
    .drop_duplicates(
        subset=["document_group"]
    )
    .groupby("split")
    .size()
)

print("\nGold term counts:")
print(
    grouped_splits.groupby(
        ["domain", "split"]
    )["gold_term_count"].sum()
)

print("\nMean balance characteristics:")
grouped_balance_table = (
    grouped_splits.groupby(
        ["domain", "split"]
    )[BALANCE_FEATURES]
    .mean()
    .round(3)
)

display(grouped_balance_table)

grouped_balance_details = pd.DataFrame(
    best_group_details
)

largest_grouped_difference = (
    grouped_balance_details[
        "standardized_difference"
    ].abs().max()
)

print(
    "\nLargest absolute standardized difference:",
    largest_grouped_difference
)

print("\nTen largest balance differences:")
display(
    grouped_balance_details.assign(
        absolute_difference=(
            grouped_balance_details[
                "standardized_difference"
            ].abs()
        )
    )
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    .head(10)
)

print(
    "\nDOI groups crossing splits:",
    int((doi_split_counts > 1).sum())
)


# ------------------------------------------------------------
# 7. Save authoritative group-aware split
# ------------------------------------------------------------

GROUPED_DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_70_15_15.csv"
)

GROUPED_SPLIT_METADATA_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_metadata.json"
)

GROUPED_BALANCE_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_balance.csv"
)

grouped_splits.to_csv(
    GROUPED_DOC_SPLIT_PATH,
    index=False
)

grouped_balance_details.to_csv(
    GROUPED_BALANCE_PATH,
    index=False
)

grouped_metadata = {
    "split_type":
        "doi_grouped_document_level_holdout",
    "proportions": SPLIT_PROPORTIONS,
    "candidate_seeds_evaluated":
        N_CANDIDATE_SEEDS,
    "selected_seed":
        int(best_group_seed),
    "combined_balance_objective":
        float(best_group_score),
    "largest_absolute_standardized_difference":
        float(largest_grouped_difference),
    "grouping_variable":
        "normalized DOI",
    "n_documents":
        int(len(grouped_splits)),
    "n_groups":
        int(
            grouped_splits[
                "document_group"
            ].nunique()
        ),
    "repeated_doi_groups":
        int((group_sizes > 1).sum()),
    "doi_groups_crossing_splits": 0,
    "balance_features":
        BALANCE_FEATURES,
    "unmatched_gold_pairs_retained_for_evaluation":
        int(len(unmatched)),
}

with open(
    GROUPED_SPLIT_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        grouped_metadata,
        file,
        indent=2
    )

# This variable becomes authoritative in subsequent cells.
DOC_SPLIT_PATH = GROUPED_DOC_SPLIT_PATH
doc_splits = grouped_splits.copy()

print("\nAuthoritative split saved to:")
print(DOC_SPLIT_PATH)

print("\nCell 8 completed successfully.")

In [ ]:
# ============================================================
# CELL 9 — Document table and PubMedBERT token-length audit
# ============================================================

import math
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer


# ------------------------------------------------------------
# 1. Reload authoritative resources
# ------------------------------------------------------------

DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_70_15_15.csv"
)

GOLD_SPANS_PATH = (
    RESULTS_DIR /
    "taskA_boundary_valid_weak_gold_spans.csv"
)

UNIQUE_GOLD_PA_PATH = (
    RESULTS_DIR /
    "taskA_parkinson_unique_document_term_pairs.csv"
)

UNIQUE_GOLD_MH_PATH = (
    RESULTS_DIR /
    "taskA_mental_health_unique_document_term_pairs.csv"
)

doc_splits = pd.read_csv(
    DOC_SPLIT_PATH,
    dtype={"num_article": str}
)

gold_spans = pd.read_csv(
    GOLD_SPANS_PATH,
    dtype={"num_article": str}
)

unique_gold_pa = pd.read_csv(
    UNIQUE_GOLD_PA_PATH,
    dtype={"num_article": str}
)

unique_gold_mh = pd.read_csv(
    UNIQUE_GOLD_MH_PATH,
    dtype={"num_article": str}
)

unique_gold_pairs = pd.concat(
    [unique_gold_pa, unique_gold_mh],
    ignore_index=True
)

assert len(doc_splits) == 540
assert doc_splits["split"].notna().all()


# ------------------------------------------------------------
# 2. Build complete document table
# ------------------------------------------------------------

document_records = []

for row in doc_splits.itertuples(index=False):
    text = read_taska_abstract(
        row.domain,
        row.num_article
    )

    document_records.append({
        "domain": row.domain,
        "num_article": str(row.num_article),
        "doi": str(row.doi),
        "split": row.split,
        "text": text,
        "character_count": len(text),
    })

documents = pd.DataFrame(
    document_records
)

assert len(documents) == 540
assert documents["text"].notna().all()
assert documents["text"].str.len().gt(0).all()

print("=" * 78)
print("DOCUMENT TABLE")
print("=" * 78)

print(
    documents.groupby(
        ["domain", "split"]
    ).size()
)


# ------------------------------------------------------------
# 3. Attach split labels to gold resources
# ------------------------------------------------------------

split_lookup = doc_splits[
    ["domain", "num_article", "split"]
].drop_duplicates()

gold_spans = gold_spans.merge(
    split_lookup,
    on=["domain", "num_article"],
    how="left",
    validate="many_to_one",
)

unique_gold_pairs = unique_gold_pairs.merge(
    split_lookup,
    on=["domain", "num_article"],
    how="left",
    validate="many_to_one",
)

assert gold_spans["split"].notna().all()
assert unique_gold_pairs["split"].notna().all()

print("\nOfficial unique document–term pairs:")
print(
    unique_gold_pairs.groupby(
        ["domain", "split"]
    ).size()
)

print("\nBoundary-valid weak-supervision spans:")
print(
    gold_spans.groupby(
        ["domain", "split"]
    ).size()
)


# ------------------------------------------------------------
# 4. Load and preserve the PubMedBERT tokenizer
# ------------------------------------------------------------

MODEL_DIR = (
    BASE_DIR /
    "models" /
    "pubmedbert"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

assert tokenizer.is_fast, (
    "A fast tokenizer is required for offset mappings."
)

tokenizer.save_pretrained(
    MODEL_DIR
)

print("\nTokenizer:", tokenizer.name_or_path)
print("Fast tokenizer:", tokenizer.is_fast)
print("Vocabulary size:", tokenizer.vocab_size)
print(
    "Special tokens added per sequence:",
    tokenizer.num_special_tokens_to_add(
        pair=False
    )
)


# ------------------------------------------------------------
# 5. Sliding-window configuration
# ------------------------------------------------------------

MAX_LENGTH = 512
WINDOW_STRIDE = 128

N_SPECIAL_TOKENS = (
    tokenizer.num_special_tokens_to_add(
        pair=False
    )
)

MAX_CONTENT_TOKENS = (
    MAX_LENGTH - N_SPECIAL_TOKENS
)

WINDOW_STEP = (
    MAX_CONTENT_TOKENS - WINDOW_STRIDE
)

assert WINDOW_STEP > 0


def required_window_count(token_count):
    if token_count <= MAX_CONTENT_TOKENS:
        return 1

    return 1 + math.ceil(
        (
            token_count
            - MAX_CONTENT_TOKENS
        )
        / WINDOW_STEP
    )


# ------------------------------------------------------------
# 6. Compute token lengths without truncation
# ------------------------------------------------------------

token_counts = []

for text in tqdm(
    documents["text"],
    desc="Counting PubMedBERT tokens"
):
    encoded = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    token_counts.append(
        len(encoded["input_ids"])
    )

documents["token_count"] = token_counts

documents["requires_windowing"] = (
    documents["token_count"]
    > MAX_CONTENT_TOKENS
)

documents["required_windows"] = (
    documents["token_count"].apply(
        required_window_count
    )
)


# ------------------------------------------------------------
# 7. Token-length report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("TOKEN-LENGTH AUDIT")
print("=" * 78)

print("Maximum model length:", MAX_LENGTH)
print(
    "Maximum content tokens per window:",
    MAX_CONTENT_TOKENS
)
print("Window overlap:", WINDOW_STRIDE)
print("Window step:", WINDOW_STEP)

for (
    domain,
    split_name
), sub in documents.groupby(
    ["domain", "split"]
):
    print(
        f"\n{domain} | {split_name}"
    )
    print("Documents:", len(sub))
    print(
        "Median tokens:",
        round(sub["token_count"].median(), 1)
    )
    print(
        "Mean tokens:",
        round(sub["token_count"].mean(), 1)
    )
    print(
        "Maximum tokens:",
        int(sub["token_count"].max())
    )
    print(
        "Documents over one window:",
        int(sub["requires_windowing"].sum()),
        f"({sub['requires_windowing'].mean():.2%})"
    )
    print(
        "Total resulting windows:",
        int(sub["required_windows"].sum())
    )


print("\nOverall token-length percentiles:")
print(
    documents["token_count"]
    .quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
    .round(1)
)

print(
    "\nTotal documents requiring windowing:",
    int(documents["requires_windowing"].sum()),
    f"({documents['requires_windowing'].mean():.2%})"
)

print(
    "Total expected windows:",
    int(documents["required_windows"].sum())
)


# ------------------------------------------------------------
# 8. Longest-document preview
# ------------------------------------------------------------

print("\nTwenty longest documents:")
display(
    documents[
        [
            "domain",
            "num_article",
            "split",
            "token_count",
            "required_windows",
            "character_count",
        ]
    ]
    .sort_values(
        "token_count",
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# 9. Save document table
# ------------------------------------------------------------

DOCUMENT_TABLE_PATH = (
    SPLIT_DIR /
    "taskA_documents_with_token_lengths.csv"
)

# Store the complete text because this is an internal,
# reproducible experiment resource.
documents.to_csv(
    DOCUMENT_TABLE_PATH,
    index=False
)

print("\nSaved document table to:")
print(DOCUMENT_TABLE_PATH)

print("\nCell 9 completed successfully.")

In [ ]:
# ============================================================
# CELL 10 — Windowed PubMedBERT start/end examples
# ============================================================

import shutil
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Index weakly supervised spans by document
# ------------------------------------------------------------

span_lookup = {}

for (
    domain,
    article_id
), sub in gold_spans.groupby(
    ["domain", "num_article"],
    sort=False
):
    span_lookup[
        (domain, str(article_id))
    ] = sub[
        [
            "term",
            "start_char",
            "end_char",
        ]
    ].to_dict("records")


# ------------------------------------------------------------
# 2. Tracking structures
# ------------------------------------------------------------

window_records = []

span_coverage = defaultdict(int)
span_alignment_failures = []
partial_window_span_count = 0

expected_span_keys = set()

for row in gold_spans.itertuples(index=False):
    expected_span_keys.add(
        (
            row.domain,
            str(row.num_article),
            int(row.start_char),
            int(row.end_char),
            str(row.term),
        )
    )


# ------------------------------------------------------------
# 3. Build overlapping document windows explicitly
# ------------------------------------------------------------

# PubMedBERT adds [CLS] and [SEP].
N_SPECIAL_TOKENS = tokenizer.num_special_tokens_to_add(
    pair=False
)

CONTENT_TOKENS_PER_WINDOW = (
    MAX_LENGTH - N_SPECIAL_TOKENS
)

WINDOW_OVERLAP = 128

WINDOW_STEP = (
    CONTENT_TOKENS_PER_WINDOW
    - WINDOW_OVERLAP
)

assert CONTENT_TOKENS_PER_WINDOW == 510
assert WINDOW_STEP == 382

for document in tqdm(
    documents.itertuples(index=False),
    total=len(documents),
    desc="Creating windowed examples",
):
    domain = document.domain
    article_id = str(document.num_article)
    text = str(document.text)

    document_spans = span_lookup.get(
        (domain, article_id),
        [],
    )

    # Tokenize the entire document once, without truncation
    # and without special tokens.
    full_encoding = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_offsets_mapping=True,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    full_input_ids = full_encoding["input_ids"]
    full_offsets = full_encoding["offset_mapping"]

    assert len(full_input_ids) == len(full_offsets)

    if len(full_input_ids) == 0:
        raise ValueError(
            f"Document contains no tokens: "
            f"{domain}/{article_id}"
        )

    # Explicitly generate starts until the final token is covered.
    window_starts = list(
        range(
            0,
            len(full_input_ids),
            WINDOW_STEP,
        )
    )

    # Remove starts that would create an unnecessary window
    # entirely contained in the preceding window.
    window_starts = [
        start
        for start in window_starts
        if (
            start == 0
            or start < len(full_input_ids)
        )
    ]

    # Once a window reaches the document end, later starts are
    # unnecessary.
    required_starts = []

    for start in window_starts:
        required_starts.append(start)

        if (
            start + CONTENT_TOKENS_PER_WINDOW
            >= len(full_input_ids)
        ):
            break

    window_starts = required_starts
    n_windows = len(window_starts)

    for window_index, token_start in enumerate(
        window_starts
    ):
        token_end = min(
            token_start + CONTENT_TOKENS_PER_WINDOW,
            len(full_input_ids),
        )

        content_ids = full_input_ids[
            token_start:token_end
        ]

        content_offsets = full_offsets[
            token_start:token_end
        ]

        # PubMedBERT uses the standard single-sequence BERT layout:
        # [CLS] content tokens [SEP].
        if tokenizer.cls_token_id is None:
            raise ValueError(
                "The tokenizer has no CLS token ID."
            )

        if tokenizer.sep_token_id is None:
            raise ValueError(
                "The tokenizer has no SEP token ID."
            )

        input_ids = [
            int(tokenizer.cls_token_id),
            *[int(token_id) for token_id in content_ids],
            int(tokenizer.sep_token_id),
        ]

        special_tokens_mask = [
            1,
            *([0] * len(content_ids)),
            1,
        ]

        token_type_ids = [0] * len(input_ids)
        attention_mask = [1] * len(input_ids)

        offsets = [
            (0, 0),
            *[
                (int(start), int(end))
                for start, end in content_offsets
            ],
            (0, 0),
        ]

        sequence_length = len(input_ids)

        assert sequence_length <= MAX_LENGTH
        assert len(attention_mask) == sequence_length
        assert len(token_type_ids) == sequence_length
        assert len(offsets) == sequence_length
        assert len(special_tokens_mask) == sequence_length

        start_labels = [0.0] * sequence_length
        end_labels = [0.0] * sequence_length

        # Special tokens and partial gold spans do not
        # contribute to boundary loss.
        label_mask = [
            0.0 if is_special else 1.0
            for is_special in special_tokens_mask
        ]

        content_token_indices = [
            index
            for index, (offset, is_special) in enumerate(
                zip(offsets, special_tokens_mask)
            )
            if (
                not is_special
                and tuple(offset) != (0, 0)
            )
        ]

        assert content_token_indices, (
            f"No content tokens for "
            f"{domain}/{article_id}/"
            f"window {window_index}"
        )

        content_start = min(
            offsets[index][0]
            for index in content_token_indices
        )

        content_end = max(
            offsets[index][1]
            for index in content_token_indices
        )

        n_gold_spans_in_window = 0

        for span in document_spans:
            span_start = int(span["start_char"])
            span_end = int(span["end_char"])
            term = str(span["term"])

            overlaps_window = (
                span_start < content_end
                and span_end > content_start
            )

            if not overlaps_window:
                continue

            fully_contained = (
                span_start >= content_start
                and span_end <= content_end
            )

            overlapping_token_indices = [
                index
                for index in content_token_indices
                if (
                    offsets[index][0] < span_end
                    and offsets[index][1] > span_start
                )
            ]

            if not overlapping_token_indices:
                continue

            if not fully_contained:
                # Do not treat a truncated piece of a gold
                # span as a negative example.
                for index in overlapping_token_indices:
                    label_mask[index] = 0.0

                partial_window_span_count += 1
                continue

            first_token = overlapping_token_indices[0]
            last_token = overlapping_token_indices[-1]

            aligned_start = (
                offsets[first_token][0]
                == span_start
            )

            aligned_end = (
                offsets[last_token][1]
                == span_end
            )

            span_key = (
                domain,
                article_id,
                span_start,
                span_end,
                term,
            )

            if not (aligned_start and aligned_end):
                span_alignment_failures.append({
                    "domain": domain,
                    "num_article": article_id,
                    "window_index": window_index,
                    "term": term,
                    "span_start": span_start,
                    "span_end": span_end,
                    "first_token_offset":
                        offsets[first_token],
                    "last_token_offset":
                        offsets[last_token],
                    "recovered_text":
                        text[span_start:span_end],
                })

                for index in overlapping_token_indices:
                    label_mask[index] = 0.0

                continue

            start_labels[first_token] = 1.0
            end_labels[last_token] = 1.0

            span_coverage[span_key] += 1
            n_gold_spans_in_window += 1

        window_records.append({
            "domain": domain,
            "num_article": article_id,
            "doi": str(document.doi),
            "split": document.split,
            "window_index": window_index,
            "n_windows_for_document": n_windows,
            "content_start_char": int(content_start),
            "content_end_char": int(content_end),
            "n_gold_spans_in_window":
                n_gold_spans_in_window,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "offset_mapping": [
                [int(start), int(end)]
                for start, end in offsets
            ],
            "start_labels": start_labels,
            "end_labels": end_labels,
            "label_mask": label_mask,
        })


windows_df = pd.DataFrame(window_records)


# ------------------------------------------------------------
# 4. Coverage audit
# ------------------------------------------------------------

covered_span_keys = set(
    span_coverage.keys()
)

uncovered_span_keys = (
    expected_span_keys
    - covered_span_keys
)

coverage_counts = np.array(
    list(span_coverage.values()),
    dtype=int
)

print("=" * 78)
print("WINDOWED DATASET")
print("=" * 78)

print("Documents:", len(documents))
print("Windows created:", len(windows_df))

print("\nWindows by split:")
print(
    windows_df.groupby(
        ["domain", "split"]
    ).size()
)

print("\nExpected weak-supervision spans:")
print(len(expected_span_keys))

print(
    "Spans covered by at least one window:",
    len(covered_span_keys)
)

print(
    "Spans not covered:",
    len(uncovered_span_keys)
)

print(
    "Token-boundary alignment failures:",
    len(span_alignment_failures)
)

print(
    "Partial span/window intersections masked:",
    partial_window_span_count
)

if len(coverage_counts):
    print(
        "Median windows covering each span:",
        float(np.median(coverage_counts))
    )

    print(
        "Maximum windows covering one span:",
        int(np.max(coverage_counts))
    )

    print(
        "Spans represented in multiple windows:",
        int((coverage_counts > 1).sum())
    )


# ------------------------------------------------------------
# 5. Label statistics
# ------------------------------------------------------------

windows_df["positive_start_labels"] = (
    windows_df["start_labels"].apply(
        lambda labels: int(sum(labels))
    )
)

windows_df["positive_end_labels"] = (
    windows_df["end_labels"].apply(
        lambda labels: int(sum(labels))
    )
)

windows_df["active_loss_tokens"] = (
    windows_df["label_mask"].apply(
        lambda mask: int(sum(mask))
    )
)

print("\nPositive boundary labels by split:")
print(
    windows_df.groupby(
        ["domain", "split"]
    )[
        [
            "positive_start_labels",
            "positive_end_labels",
            "active_loss_tokens",
        ]
    ].sum()
)

print(
    "\nWindows with no positive gold boundary:",
    int(
        (
            windows_df[
                "positive_start_labels"
            ] == 0
        ).sum()
    )
)


# ------------------------------------------------------------
# 6. Show problems if any
# ------------------------------------------------------------

if span_alignment_failures:
    print("\nFirst tokenizer-alignment failures:")
    display(
        pd.DataFrame(
            span_alignment_failures
        ).head(30)
    )

if uncovered_span_keys:
    print("\nFirst uncovered spans:")

    uncovered_df = pd.DataFrame(
        list(uncovered_span_keys),
        columns=[
            "domain",
            "num_article",
            "start_char",
            "end_char",
            "term",
        ]
    )

    display(
        uncovered_df.head(30)
    )


# ------------------------------------------------------------
# 7. Critical assertions
# ------------------------------------------------------------

# Recalculate the required number of windows using the same
# explicit construction rule used above.
def calculate_required_windows(text):
    encoding = tokenizer(
        str(text),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    n_tokens = len(encoding["input_ids"])

    if n_tokens <= CONTENT_TOKENS_PER_WINDOW:
        return 1

    return (
        1
        + (
            n_tokens
            - CONTENT_TOKENS_PER_WINDOW
            + WINDOW_STEP
            - 1
        )
        // WINDOW_STEP
    )


documents["corrected_required_windows"] = (
    documents["text"].apply(
        calculate_required_windows
    )
)

window_count_changes = documents.loc[
    documents["required_windows"]
    != documents["corrected_required_windows"],
    [
        "domain",
        "num_article",
        "split",
        "token_count",
        "required_windows",
        "corrected_required_windows",
    ],
].copy()

print("\nCorrections to Cell 9 window estimates:")

if len(window_count_changes):
    display(window_count_changes)
else:
    print("None")

# Retain the corrected value as the authoritative count.
documents["required_windows"] = (
    documents["corrected_required_windows"]
)

assert len(windows_df) == int(
    documents["required_windows"].sum()
), (
    "Explicit window count differs from the "
    "corrected token-length calculation."
)

assert len(span_alignment_failures) == 0, (
    "Some gold spans do not align with "
    "PubMedBERT token boundaries."
)

assert len(uncovered_span_keys) == 0, (
    "Some boundary-valid gold spans are not "
    "represented in any window."
)


# ------------------------------------------------------------
# 8. Convert to Hugging Face DatasetDict
# ------------------------------------------------------------

dataset_dict = DatasetDict()

for split_name in [
    "train",
    "development",
    "test",
]:
    split_records = (
        windows_df[
            windows_df["split"]
            == split_name
        ]
        .drop(
            columns=[
                "positive_start_labels",
                "positive_end_labels",
                "active_loss_tokens",
            ]
        )
        .to_dict("records")
    )

    dataset_dict[split_name] = (
        Dataset.from_list(
            split_records
        )
    )

print("\nDatasetDict:")
print(dataset_dict)


# ------------------------------------------------------------
# 9. Save dataset
# ------------------------------------------------------------

WINDOW_DATASET_PATH = (
    CACHE_DIR /
    "pubmedbert_start_end_windows_v1"
)

# Exact experiment cache: safe to replace when rerunning
# this construction cell.
if WINDOW_DATASET_PATH.exists():
    shutil.rmtree(
        WINDOW_DATASET_PATH
    )

dataset_dict.save_to_disk(
    str(WINDOW_DATASET_PATH)
)

WINDOW_AUDIT_PATH = (
    RESULTS_DIR /
    "pubmedbert_window_construction_audit.csv"
)

windows_df[
    [
        "domain",
        "num_article",
        "doi",
        "split",
        "window_index",
        "n_windows_for_document",
        "content_start_char",
        "content_end_char",
        "n_gold_spans_in_window",
        "positive_start_labels",
        "positive_end_labels",
        "active_loss_tokens",
    ]
].to_csv(
    WINDOW_AUDIT_PATH,
    index=False
)

print("\nSaved windowed dataset to:")
print(WINDOW_DATASET_PATH)

print("\nSaved construction audit to:")
print(WINDOW_AUDIT_PATH)

print("\nCell 10 completed successfully.")

In [ ]:
# @title
# CELL 10A — Diagnose uncovered spans and text/window extent

document_text_lookup = {
    (
        row.domain,
        str(row.num_article),
    ): str(row.text)
    for row in documents.itertuples(index=False)
}

document_split_lookup = {
    (
        row.domain,
        str(row.num_article),
    ): row.split
    for row in documents.itertuples(index=False)
}

window_extent = (
    windows_df.groupby(
        ["domain", "num_article"],
        as_index=False,
    )
    .agg(
        actual_windows=("window_index", "size"),
        final_window_end=("content_end_char", "max"),
    )
)

diagnostic_rows = []

for (
    domain,
    article_id,
    span_start,
    span_end,
    term,
) in sorted(uncovered_span_keys):

    key = (domain, str(article_id))
    text = document_text_lookup.get(key)

    if text is None:
        diagnostic_rows.append({
            "domain": domain,
            "num_article": article_id,
            "split": None,
            "term": term,
            "start_char": span_start,
            "end_char": span_end,
            "text_length": None,
            "span_inside_text": False,
            "recovered_text": None,
            "issue": "document_not_found",
        })
        continue

    recovered_text = (
        text[span_start:span_end]
        if span_start < len(text)
        else ""
    )

    diagnostic_rows.append({
        "domain": domain,
        "num_article": article_id,
        "split": document_split_lookup[key],
        "term": term,
        "start_char": span_start,
        "end_char": span_end,
        "text_length": len(text),
        "span_inside_text": span_end <= len(text),
        "recovered_text": recovered_text,
        "exact_text_match": recovered_text == term,
        "issue": (
            "span_beyond_text"
            if span_end > len(text)
            else (
                "text_offset_mismatch"
                if recovered_text != term
                else "window_coverage_problem"
            )
        ),
    })

uncovered_diagnostic = pd.DataFrame(
    diagnostic_rows
).merge(
    window_extent,
    on=["domain", "num_article"],
    how="left",
)

uncovered_diagnostic["span_within_final_window_extent"] = (
    uncovered_diagnostic["end_char"]
    <= uncovered_diagnostic["final_window_end"]
)

print("=" * 78)
print("UNCOVERED-SPAN DIAGNOSTIC")
print("=" * 78)

print("\nIssues:")
print(
    uncovered_diagnostic["issue"]
    .value_counts(dropna=False)
)

print("\nAffected documents:")
print(
    uncovered_diagnostic[
        [
            "domain",
            "num_article",
            "split",
            "text_length",
            "final_window_end",
            "actual_windows",
            "issue",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["domain", "num_article"]
    )
    .to_string(index=False)
)

print("\nFirst 40 uncovered spans:")
print(
    uncovered_diagnostic[
        [
            "domain",
            "num_article",
            "term",
            "start_char",
            "end_char",
            "text_length",
            "final_window_end",
            "recovered_text",
            "issue",
        ]
    ]
    .head(40)
    .to_string(index=False)
)

print("\nGold spans beyond stored document text:")
print(
    int(
        (
            uncovered_diagnostic["issue"]
            == "span_beyond_text"
        ).sum()
    )
)

print("\nGold spans inside text but beyond generated windows:")
print(
    int(
        (
            uncovered_diagnostic["span_inside_text"]
            & ~uncovered_diagnostic[
                "span_within_final_window_extent"
            ]
        ).sum()
    )
)

In [ ]:
# ============================================================
# CELL 10R — Restore saved data after changing to GPU runtime
# ============================================================

from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoTokenizer


# ------------------------------------------------------------
# 1. Require an attached GPU
# ------------------------------------------------------------

assert torch.cuda.is_available(), (
    "No GPU is attached. In Colab, select:\n"
    "Runtime > Change runtime type > "
    "T4 GPU or A100 GPU."
)

DEVICE = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(
            0
        ).total_memory / 1024**3,
        1
    ),
    "GB"
)


# ------------------------------------------------------------
# 2. Restore tokenizer
# ------------------------------------------------------------

MODEL_NAME = (
    "microsoft/"
    "BiomedNLP-PubMedBERT-base-uncased-abstract"
)

MODEL_DIR = (
    BASE_DIR /
    "models" /
    "pubmedbert"
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    use_fast=True
)

assert tokenizer.is_fast


# ------------------------------------------------------------
# 3. Restore windowed DatasetDict
# ------------------------------------------------------------

WINDOW_DATASET_PATH = (
    CACHE_DIR /
    "pubmedbert_start_end_windows_v1"
)

assert WINDOW_DATASET_PATH.exists(), (
    f"Saved dataset not found:\n"
    f"{WINDOW_DATASET_PATH}"
)

dataset_dict = load_from_disk(
    str(WINDOW_DATASET_PATH)
)


# ------------------------------------------------------------
# 4. Validate restored data
# ------------------------------------------------------------

expected_sizes = {
    "train": 403,
    "development": 85,
    "test": 83,
}

for split_name, expected_size in (
    expected_sizes.items()
):
    actual_size = len(
        dataset_dict[split_name]
    )

    assert actual_size == expected_size, (
        f"{split_name}: expected "
        f"{expected_size}, found "
        f"{actual_size}"
    )

print("\nRestored DatasetDict:")
print(dataset_dict)

print("\nTokenizer:", tokenizer.name_or_path)
print("Device:", DEVICE)

print("\nCell 10R completed successfully.")

In [ ]:
# ============================================================
# CELL 11 — Model, collator, and masked boundary loss
# ============================================================

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    Trainer,
)


# ------------------------------------------------------------
# 1. Calculate training-only class weights
# ------------------------------------------------------------

def calculate_boundary_class_weights(
    dataset
):
    active_tokens = 0
    positive_starts = 0
    positive_ends = 0

    for example in dataset:
        mask = np.asarray(
            example["label_mask"],
            dtype=np.float64
        )

        start_labels = np.asarray(
            example["start_labels"],
            dtype=np.float64
        )

        end_labels = np.asarray(
            example["end_labels"],
            dtype=np.float64
        )

        active_tokens += int(mask.sum())

        positive_starts += int(
            (start_labels * mask).sum()
        )

        positive_ends += int(
            (end_labels * mask).sum()
        )

    negative_starts = (
        active_tokens - positive_starts
    )

    negative_ends = (
        active_tokens - positive_ends
    )

    start_pos_weight = (
        negative_starts / positive_starts
    )

    end_pos_weight = (
        negative_ends / positive_ends
    )

    return {
        "active_tokens": active_tokens,
        "positive_starts": positive_starts,
        "negative_starts": negative_starts,
        "positive_ends": positive_ends,
        "negative_ends": negative_ends,
        "start_pos_weight": start_pos_weight,
        "end_pos_weight": end_pos_weight,
    }


class_weight_stats = (
    calculate_boundary_class_weights(
        dataset_dict["train"]
    )
)

print("=" * 78)
print("TRAINING-ONLY CLASS WEIGHTS")
print("=" * 78)

for key, value in class_weight_stats.items():
    print(f"{key}: {value}")


START_POS_WEIGHT = float(
    class_weight_stats[
        "start_pos_weight"
    ]
)

END_POS_WEIGHT = float(
    class_weight_stats[
        "end_pos_weight"
    ]
)


# ------------------------------------------------------------
# 2. Custom dynamic-padding collator
# ------------------------------------------------------------

@dataclass
class SpanBoundaryDataCollator:
    tokenizer: object
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        model_features = []

        for feature in features:
            model_feature = {
                "input_ids":
                    feature["input_ids"],
                "attention_mask":
                    feature["attention_mask"],
            }

            if "token_type_ids" in feature:
                model_feature[
                    "token_type_ids"
                ] = feature[
                    "token_type_ids"
                ]

            model_features.append(
                model_feature
            )

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            pad_to_multiple_of=(
                self.pad_to_multiple_of
            ),
            return_tensors="pt",
        )

        padded_length = batch[
            "input_ids"
        ].shape[1]

        def pad_numeric_sequence(
            values,
            pad_value
        ):
            padded = []

            for sequence in values:
                sequence = list(sequence)

                padding_needed = (
                    padded_length
                    - len(sequence)
                )

                padded.append(
                    sequence
                    + [pad_value]
                    * padding_needed
                )

            return torch.tensor(
                padded,
                dtype=torch.float32
            )

        batch["start_labels"] = (
            pad_numeric_sequence(
                [
                    feature["start_labels"]
                    for feature in features
                ],
                pad_value=0.0,
            )
        )

        batch["end_labels"] = (
            pad_numeric_sequence(
                [
                    feature["end_labels"]
                    for feature in features
                ],
                pad_value=0.0,
            )
        )

        # Padding tokens and any previously masked tokens
        # must contribute nothing to the loss.
        batch["label_mask"] = (
            pad_numeric_sequence(
                [
                    feature["label_mask"]
                    for feature in features
                ],
                pad_value=0.0,
            )
        )

        return batch


data_collator = SpanBoundaryDataCollator(
    tokenizer=tokenizer
)


# ------------------------------------------------------------
# 3. PubMedBERT with two token-level boundary logits
#
# Logit 0 = start boundary
# Logit 1 = end boundary
# ------------------------------------------------------------

model_config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

model_config.id2label = {
    0: "START_BOUNDARY",
    1: "END_BOUNDARY",
}

model_config.label2id = {
    "START_BOUNDARY": 0,
    "END_BOUNDARY": 1,
}

model = (
    AutoModelForTokenClassification
    .from_pretrained(
        MODEL_NAME,
        config=model_config,
        ignore_mismatched_sizes=True,
    )
)

model.to(DEVICE)


# ------------------------------------------------------------
# 4. Custom Trainer with masked weighted BCE
# ------------------------------------------------------------

class MaskedBoundaryTrainer(Trainer):

    def __init__(
        self,
        *args,
        start_pos_weight,
        end_pos_weight,
        **kwargs
    ):
        super().__init__(
            *args,
            **kwargs
        )

        self.start_pos_weight = float(
            start_pos_weight
        )

        self.end_pos_weight = float(
            end_pos_weight
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs
    ):
        start_labels = inputs.pop(
            "start_labels"
        )

        end_labels = inputs.pop(
            "end_labels"
        )

        label_mask = inputs.pop(
            "label_mask"
        )

        outputs = model(**inputs)

        logits = outputs.logits

        start_logits = logits[:, :, 0]
        end_logits = logits[:, :, 1]

        start_pos_weight = torch.tensor(
            self.start_pos_weight,
            dtype=start_logits.dtype,
            device=start_logits.device,
        )

        end_pos_weight = torch.tensor(
            self.end_pos_weight,
            dtype=end_logits.dtype,
            device=end_logits.device,
        )

        start_loss_per_token = (
            F.binary_cross_entropy_with_logits(
                start_logits,
                start_labels,
                reduction="none",
                pos_weight=start_pos_weight,
            )
        )

        end_loss_per_token = (
            F.binary_cross_entropy_with_logits(
                end_logits,
                end_labels,
                reduction="none",
                pos_weight=end_pos_weight,
            )
        )

        denominator = (
            label_mask.sum().clamp_min(1.0)
        )

        start_loss = (
            start_loss_per_token
            * label_mask
        ).sum() / denominator

        end_loss = (
            end_loss_per_token
            * label_mask
        ).sum() / denominator

        loss = (
            start_loss + end_loss
        ) / 2.0

        if return_outputs:
            return loss, outputs

        return loss


# ------------------------------------------------------------
# 5. Forward-pass sanity check
# ------------------------------------------------------------

sanity_features = [
    dataset_dict["train"][0],
    dataset_dict["train"][1],
]

sanity_batch = data_collator(
    sanity_features
)

model_inputs = {
    "input_ids":
        sanity_batch[
            "input_ids"
        ].to(DEVICE),

    "attention_mask":
        sanity_batch[
            "attention_mask"
        ].to(DEVICE),

    "token_type_ids":
        sanity_batch[
            "token_type_ids"
        ].to(DEVICE),
}

with torch.no_grad():
    sanity_outputs = model(
        **model_inputs
    )

print("\n" + "=" * 78)
print("FORWARD-PASS SANITY CHECK")
print("=" * 78)

print(
    "Input shape:",
    tuple(
        sanity_batch[
            "input_ids"
        ].shape
    )
)

print(
    "Start-label shape:",
    tuple(
        sanity_batch[
            "start_labels"
        ].shape
    )
)

print(
    "End-label shape:",
    tuple(
        sanity_batch[
            "end_labels"
        ].shape
    )
)

print(
    "Label-mask shape:",
    tuple(
        sanity_batch[
            "label_mask"
        ].shape
    )
)

print(
    "Model logits shape:",
    tuple(
        sanity_outputs.logits.shape
    )
)

assert (
    sanity_outputs.logits.shape[:2]
    == sanity_batch[
        "input_ids"
    ].shape
)

assert (
    sanity_outputs.logits.shape[-1]
    == 2
)

print(
    "\nTrainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

print(
    "Total parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print("\nCell 11 completed successfully.")

In [ ]:
# ============================================================
# CELL 12 — Reproducible training and evaluation configuration
# ============================================================

import os
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import (
    precision_recall_fscore_support,
)
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    TrainingArguments,
    set_seed,
)


# ------------------------------------------------------------
# 1. Reproducibility
# ------------------------------------------------------------

NEURAL_SEED = 42

os.environ["PYTHONHASHSEED"] = str(
    NEURAL_SEED
)

random.seed(NEURAL_SEED)
np.random.seed(NEURAL_SEED)
torch.manual_seed(NEURAL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        NEURAL_SEED
    )

set_seed(NEURAL_SEED)

# This improves reproducibility while retaining practical speed.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ------------------------------------------------------------
# 2. GPU-sensitive batch configuration
# ------------------------------------------------------------

if torch.cuda.is_available():
    gpu_properties = (
        torch.cuda.get_device_properties(0)
    )

    gpu_memory_gb = (
        gpu_properties.total_memory
        / 1024**3
    )

    compute_capability = (
        torch.cuda.get_device_capability(0)
    )
else:
    gpu_memory_gb = 0
    compute_capability = (0, 0)


if gpu_memory_gb >= 30:
    # A100-class configuration
    TRAIN_BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 2
else:
    # T4-class configuration
    TRAIN_BATCH_SIZE = 4
    EVAL_BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 4


EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS
)

USE_BF16 = (
    torch.cuda.is_available()
    and compute_capability[0] >= 8
)

USE_FP16 = (
    torch.cuda.is_available()
    and not USE_BF16
)

print("=" * 78)
print("TRAINING HARDWARE CONFIGURATION")
print("=" * 78)

print(
    "GPU:",
    (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    )
)

print(
    "GPU memory:",
    round(gpu_memory_gb, 1),
    "GB"
)

print(
    "Compute capability:",
    compute_capability
)

print(
    "Train batch size:",
    TRAIN_BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRADIENT_ACCUMULATION_STEPS
)

print(
    "Effective train batch size:",
    EFFECTIVE_BATCH_SIZE
)

print(
    "Evaluation batch size:",
    EVAL_BATCH_SIZE
)

print("FP16:", USE_FP16)
print("BF16:", USE_BF16)


# ------------------------------------------------------------
# 3. Recreate model after fixing the seed
# ------------------------------------------------------------

def initialize_boundary_model():
    config = AutoConfig.from_pretrained(
        MODEL_NAME,
        num_labels=2,
    )

    config.id2label = {
        0: "START_BOUNDARY",
        1: "END_BOUNDARY",
    }

    config.label2id = {
        "START_BOUNDARY": 0,
        "END_BOUNDARY": 1,
    }

    initialized_model = (
        AutoModelForTokenClassification
        .from_pretrained(
            MODEL_NAME,
            config=config,
            ignore_mismatched_sizes=True,
        )
    )

    # Gradient checkpointing lowers T4 memory use.
    initialized_model.gradient_checkpointing_enable()

    initialized_model.config.use_cache = False

    return initialized_model


# Discard the Cell 11 sanity-test model.
del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

set_seed(NEURAL_SEED)

model = initialize_boundary_model()


# ------------------------------------------------------------
# 4. Token-boundary monitoring metrics
#
# These are diagnostic metrics at probability 0.5.
# Final term extraction will use span decoding and thresholds
# tuned only on the development partition.
# ------------------------------------------------------------

def binary_boundary_metrics(
    gold,
    predicted
):
    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            gold,
            predicted,
            average="binary",
            zero_division=0,
        )
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def compute_boundary_metrics(
    evaluation_prediction
):
    logits = evaluation_prediction.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    label_ids = (
        evaluation_prediction.label_ids
    )

    if not isinstance(
        label_ids,
        (tuple, list)
    ):
        raise ValueError(
            "Expected three label arrays: "
            "start_labels, end_labels, label_mask."
        )

    start_gold = np.asarray(
        label_ids[0]
    )

    end_gold = np.asarray(
        label_ids[1]
    )

    label_mask = np.asarray(
        label_ids[2]
    )

    start_probabilities = (
        1.0
        / (
            1.0
            + np.exp(
                -logits[:, :, 0]
            )
        )
    )

    end_probabilities = (
        1.0
        / (
            1.0
            + np.exp(
                -logits[:, :, 1]
            )
        )
    )

    active = label_mask > 0.5

    start_gold_active = (
        start_gold[active]
        .astype(int)
    )

    end_gold_active = (
        end_gold[active]
        .astype(int)
    )

    start_predictions = (
        start_probabilities[active]
        >= 0.5
    ).astype(int)

    end_predictions = (
        end_probabilities[active]
        >= 0.5
    ).astype(int)

    start_metrics = binary_boundary_metrics(
        start_gold_active,
        start_predictions,
    )

    end_metrics = binary_boundary_metrics(
        end_gold_active,
        end_predictions,
    )

    return {
        "start_precision":
            start_metrics["precision"],
        "start_recall":
            start_metrics["recall"],
        "start_f1":
            start_metrics["f1"],

        "end_precision":
            end_metrics["precision"],
        "end_recall":
            end_metrics["recall"],
        "end_f1":
            end_metrics["f1"],

        "mean_boundary_f1":
            (
                start_metrics["f1"]
                + end_metrics["f1"]
            ) / 2.0,
    }


# ------------------------------------------------------------
# 5. Training arguments — Transformers 5.16.1
# ------------------------------------------------------------

import math


RUN_NAME = (
    f"pubmedbert_boundary_seed"
    f"{NEURAL_SEED}"
)

NEURAL_RUN_DIR = (
    MODEL_OUTPUT_DIR /
    RUN_NAME
)


# ------------------------------------------------------------
# Calculate fixed warmup steps
# ------------------------------------------------------------

TRAIN_WINDOWS = len(
    dataset_dict["train"]
)

MICROBATCHES_PER_EPOCH = math.ceil(
    TRAIN_WINDOWS
    / TRAIN_BATCH_SIZE
)

UPDATE_STEPS_PER_EPOCH = math.ceil(
    MICROBATCHES_PER_EPOCH
    / GRADIENT_ACCUMULATION_STEPS
)

TOTAL_TRAINING_STEPS = (
    UPDATE_STEPS_PER_EPOCH * 5
)

WARMUP_STEPS = round(
    TOTAL_TRAINING_STEPS * 0.10
)

assert MICROBATCHES_PER_EPOCH == 51
assert UPDATE_STEPS_PER_EPOCH == 26
assert TOTAL_TRAINING_STEPS == 130
assert WARMUP_STEPS == 13

print()
print("Training-step calculation:")
print(
    "Microbatches per epoch:",
    MICROBATCHES_PER_EPOCH,
)
print(
    "Optimizer updates per epoch:",
    UPDATE_STEPS_PER_EPOCH,
)
print(
    "Total optimizer updates:",
    TOTAL_TRAINING_STEPS,
)
print(
    "Warmup steps:",
    WARMUP_STEPS,
)


# ------------------------------------------------------------
# Protect an existing run
# ------------------------------------------------------------

if (
    NEURAL_RUN_DIR.exists()
    and any(NEURAL_RUN_DIR.iterdir())
):
    raise FileExistsError(
        "The training output directory already exists "
        "and is not empty:\n"
        f"{NEURAL_RUN_DIR}\n\n"
        "Use a different run name or intentionally "
        "archive/remove the existing run before training."
    )


# ------------------------------------------------------------
# Transformers 5.16.1 arguments
# ------------------------------------------------------------

training_args = TrainingArguments(
    output_dir=str(
        NEURAL_RUN_DIR
    ),

    run_name=RUN_NAME,

    # Optimization
    num_train_epochs=5,
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,

    per_device_train_batch_size=(
        TRAIN_BATCH_SIZE
    ),

    per_device_eval_batch_size=(
        EVAL_BATCH_SIZE
    ),

    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),

    max_grad_norm=1.0,
    optim="adamw_torch",

    # Precision
    fp16=USE_FP16,
    bf16=USE_BF16,

    # Evaluation and checkpoints
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,

    # Reproducibility
    seed=NEURAL_SEED,
    data_seed=NEURAL_SEED,

    # Custom label columns required by compute_loss
    label_names=[
        "start_labels",
        "end_labels",
        "label_mask",
    ],

    remove_unused_columns=False,

    # Miscellaneous
    report_to="none",
    dataloader_num_workers=2,
    dataloader_pin_memory=(
        torch.cuda.is_available()
    ),
)

# ------------------------------------------------------------
# 6. Initialize Trainer
# ------------------------------------------------------------

trainer = MaskedBoundaryTrainer(
    model=model,
    args=training_args,

    train_dataset=(
        dataset_dict["train"]
    ),

    eval_dataset=(
        dataset_dict[
            "development"
        ]
    ),

    data_collator=data_collator,

    compute_metrics=(
        compute_boundary_metrics
    ),

    processing_class=tokenizer,

    start_pos_weight=(
        START_POS_WEIGHT
    ),

    end_pos_weight=(
        END_POS_WEIGHT
    ),
)


# ------------------------------------------------------------
# 7. Configuration report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("TRAINING RUN")
print("=" * 78)

print("Run name:", RUN_NAME)
print("Seed:", NEURAL_SEED)
print(
    "Training windows:",
    len(dataset_dict["train"])
)
print(
    "Development windows:",
    len(
        dataset_dict[
            "development"
        ]
    )
)
print("Epochs:", training_args.num_train_epochs)
print(
    "Learning rate:",
    training_args.learning_rate
)
print(
    "Checkpoint criterion:",
    training_args.metric_for_best_model
)
print(
    "Output directory:",
    NEURAL_RUN_DIR
)

print("\nCell 12 completed successfully.")

In [ ]:
# @title
# CELL 12A — Inspect installed TrainingArguments API (diagnostics)

import inspect
import transformers

requested_arguments = [
    "output_dir",
    "run_name",
    "num_train_epochs",
    "learning_rate",
    "weight_decay",
    "warmup_ratio",
    "warmup_steps",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "max_grad_norm",
    "optim",
    "fp16",
    "bf16",
    "eval_strategy",
    "save_strategy",
    "logging_strategy",
    "logging_steps",
    "load_best_model_at_end",
    "metric_for_best_model",
    "greater_is_better",
    "save_total_limit",
    "save_safetensors",
    "seed",
    "data_seed",
    "label_names",
    "remove_unused_columns",
    "report_to",
    "dataloader_num_workers",
    "dataloader_pin_memory",
]

signature = inspect.signature(
    TrainingArguments.__init__
)

supported_arguments = set(
    signature.parameters
)

print("Transformers version:")
print(transformers.__version__)

print("\nSupported requested arguments:")
for name in requested_arguments:
    if name in supported_arguments:
        print("  YES:", name)

print("\nUnsupported requested arguments:")
for name in requested_arguments:
    if name not in supported_arguments:
        print("   NO:", name)

print("\nComplete TrainingArguments signature:")
print(signature)

In [ ]:
# @title
# ============================================================
# CELL 13 — Train seed-42 neural boundary extractor
# ============================================================

import json
import time
from pathlib import Path

import pandas as pd
import torch
from transformers.trainer_utils import (
    get_last_checkpoint,
)


# ------------------------------------------------------------
# 1. Check for an earlier interrupted checkpoint
# ------------------------------------------------------------

last_checkpoint = None

if NEURAL_RUN_DIR.exists():
    last_checkpoint = get_last_checkpoint(
        str(NEURAL_RUN_DIR)
    )

print("=" * 78)
print("TRAINING CHECKPOINT STATUS")
print("=" * 78)

print("Run directory:", NEURAL_RUN_DIR)
print(
    "Existing checkpoint:",
    (
        last_checkpoint
        if last_checkpoint
        else "None"
    )
)

# For the first complete run, leave this as None.
# If Colab disconnects during training, change it to
# last_checkpoint when rerunning this cell.
RESUME_FROM_CHECKPOINT = None


# ------------------------------------------------------------
# 2. Train
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("STARTING TRAINING")
print("=" * 78)

training_start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=(
        RESUME_FROM_CHECKPOINT
    )
)

training_elapsed_seconds = (
    time.time() - training_start_time
)

print("\nTraining completed.")
print(
    "Elapsed minutes:",
    round(
        training_elapsed_seconds / 60,
        2
    )
)


# ------------------------------------------------------------
# 3. Evaluate the selected best model on development only
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BEST MODEL — DEVELOPMENT EVALUATION")
print("=" * 78)

development_metrics = trainer.evaluate(
    eval_dataset=(
        dataset_dict[
            "development"
        ]
    ),
    metric_key_prefix="development",
)

for key, value in sorted(
    development_metrics.items()
):
    print(f"{key}: {value}")


# ------------------------------------------------------------
# 4. Save selected model and tokenizer
# ------------------------------------------------------------

BEST_MODEL_DIR = (
    NEURAL_RUN_DIR /
    "best_model"
)

trainer.save_model(
    str(BEST_MODEL_DIR)
)

tokenizer.save_pretrained(
    str(BEST_MODEL_DIR)
)

print("\nBest model saved to:")
print(BEST_MODEL_DIR)


# ------------------------------------------------------------
# 5. Save training history
# ------------------------------------------------------------

history_df = pd.DataFrame(
    trainer.state.log_history
)

TRAINING_HISTORY_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_training_history.csv"
)

history_df.to_csv(
    TRAINING_HISTORY_PATH,
    index=False
)

print("\nTraining history saved to:")
print(TRAINING_HISTORY_PATH)


# ------------------------------------------------------------
# 6. Save run metadata
# ------------------------------------------------------------

RUN_METADATA_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_metadata.json"
)

run_metadata = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "seed": NEURAL_SEED,

    "split_path": str(
        DOC_SPLIT_PATH
    ),

    "window_dataset_path": str(
        WINDOW_DATASET_PATH
    ),

    "training_windows": int(
        len(
            dataset_dict[
                "train"
            ]
        )
    ),

    "development_windows": int(
        len(
            dataset_dict[
                "development"
            ]
        )
    ),

    "test_windows_untouched": int(
        len(
            dataset_dict[
                "test"
            ]
        )
    ),

    "epochs": float(
        training_args.num_train_epochs
    ),

    "learning_rate": float(
        training_args.learning_rate
    ),

    "weight_decay": float(
        training_args.weight_decay
    ),

    "warmup_ratio_planned": 0.10,

    "warmup_steps": int(
        training_args.warmup_steps
    ),

    "total_training_steps": int(
        TOTAL_TRAINING_STEPS
    ),
    "train_batch_size_per_device":
        TRAIN_BATCH_SIZE,

    "gradient_accumulation_steps":
        GRADIENT_ACCUMULATION_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "start_pos_weight":
        START_POS_WEIGHT,

    "end_pos_weight":
        END_POS_WEIGHT,

    "precision": (
        "bf16"
        if USE_BF16
        else (
            "fp16"
            if USE_FP16
            else "fp32"
        )
    ),

    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),

    "training_elapsed_seconds":
        training_elapsed_seconds,

    "best_checkpoint":
        trainer.state.best_model_checkpoint,

    "best_metric":
        trainer.state.best_metric,

    "checkpoint_selection_metric":
        training_args.metric_for_best_model,

    "development_metrics":
        {
            key: (
                float(value)
                if isinstance(
                    value,
                    (
                        int,
                        float,
                        np.integer,
                        np.floating,
                    )
                )
                else value
            )
            for key, value
            in development_metrics.items()
        },
}

with open(
    RUN_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        run_metadata,
        file,
        indent=2
    )

print("\nRun metadata saved to:")
print(RUN_METADATA_PATH)


# ------------------------------------------------------------
# 7. Compact training-history view
# ------------------------------------------------------------

history_columns = [
    column
    for column in [
        "epoch",
        "loss",
        "eval_loss",
        "eval_start_precision",
        "eval_start_recall",
        "eval_start_f1",
        "eval_end_precision",
        "eval_end_recall",
        "eval_end_f1",
        "eval_mean_boundary_f1",
        "learning_rate",
    ]
    if column in history_df.columns
]

print("\nTraining history:")
display(
    history_df[
        history_columns
    ].dropna(
        how="all",
        subset=[
            column
            for column in history_columns
            if column != "epoch"
        ]
    )
)

print("\nBest checkpoint:")
print(
    trainer.state.best_model_checkpoint
)

print("Best development loss:")
print(
    trainer.state.best_metric
)

print("\nCell 13 completed successfully.")

In [ ]:
# ============================================================
# CELL 13A — Complete metadata saving without retraining (Run this cell if Cell 13 trains but yields error)
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch


# Restore the authoritative split-path variable.
DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_70_15_15.csv"
)

assert DOC_SPLIT_PATH.exists(), (
    f"Authoritative split not found:\n"
    f"{DOC_SPLIT_PATH}"
)

assert "train_result" in globals(), (
    "train_result is not available. "
    "Training may not have completed."
)

assert "development_metrics" in globals(), (
    "development_metrics is not available. "
    "Development evaluation may not have completed."
)

assert "training_elapsed_seconds" in globals()

assert "history_df" in globals()


# ------------------------------------------------------------
# Save run metadata
# ------------------------------------------------------------

RUN_METADATA_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_metadata.json"
)

run_metadata = {
    "run_name": RUN_NAME,
    "model_name": MODEL_NAME,
    "seed": NEURAL_SEED,

    "split_path": str(
        DOC_SPLIT_PATH
    ),

    "window_dataset_path": str(
        WINDOW_DATASET_PATH
    ),

    "training_windows": int(
        len(dataset_dict["train"])
    ),

    "development_windows": int(
        len(
            dataset_dict[
                "development"
            ]
        )
    ),

    "test_windows_untouched": int(
        len(dataset_dict["test"])
    ),

    "epochs": float(
        training_args.num_train_epochs
    ),

    "learning_rate": float(
        training_args.learning_rate
    ),

    "weight_decay": float(
        training_args.weight_decay
    ),

    "warmup_ratio_planned": 0.10,

    "warmup_steps": int(
        training_args.warmup_steps
    ),

    "total_training_steps": int(
        TOTAL_TRAINING_STEPS
    ),
    "train_batch_size_per_device":
        TRAIN_BATCH_SIZE,

    "gradient_accumulation_steps":
        GRADIENT_ACCUMULATION_STEPS,

    "effective_batch_size":
        EFFECTIVE_BATCH_SIZE,

    "start_pos_weight":
        START_POS_WEIGHT,

    "end_pos_weight":
        END_POS_WEIGHT,

    "precision": (
        "bf16"
        if USE_BF16
        else (
            "fp16"
            if USE_FP16
            else "fp32"
        )
    ),

    "gpu": torch.cuda.get_device_name(0),

    "training_elapsed_seconds":
        float(training_elapsed_seconds),

    "best_checkpoint":
        trainer.state.best_model_checkpoint,

    "best_metric": float(
        trainer.state.best_metric
    ),

    "checkpoint_selection_metric":
        training_args.metric_for_best_model,

    "development_metrics": {
        key: (
            float(value)
            if isinstance(
                value,
                (
                    int,
                    float,
                    np.integer,
                    np.floating,
                )
            )
            else value
        )
        for key, value
        in development_metrics.items()
    },
}

with open(
    RUN_METADATA_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        run_metadata,
        file,
        indent=2
    )

print("Run metadata saved to:")
print(RUN_METADATA_PATH)


# ------------------------------------------------------------
# Display completed results
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("BEST MODEL")
print("=" * 78)

print(
    "Best checkpoint:",
    trainer.state.best_model_checkpoint
)

print(
    "Best development loss:",
    trainer.state.best_metric
)

print("\nDevelopment metrics:")

for key, value in sorted(
    development_metrics.items()
):
    print(f"{key}: {value}")


history_columns = [
    column
    for column in [
        "epoch",
        "loss",
        "eval_loss",
        "eval_start_precision",
        "eval_start_recall",
        "eval_start_f1",
        "eval_end_precision",
        "eval_end_recall",
        "eval_end_f1",
        "eval_mean_boundary_f1",
        "learning_rate",
    ]
    if column in history_df.columns
]

print("\nTraining history:")

display(
    history_df[
        history_columns
    ].dropna(
        how="all",
        subset=[
            column
            for column in history_columns
            if column != "epoch"
        ]
    )
)

print("\nCell 13A completed successfully.")

In [ ]:
# ============================================================
# CELL 14 — Development logits and threshold diagnostics
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit


# ------------------------------------------------------------
# 1. Predict development windows with the selected best model
# ------------------------------------------------------------

print("=" * 78)
print("GENERATING DEVELOPMENT LOGITS")
print("=" * 78)

development_output = trainer.predict(
    dataset_dict["development"],
    metric_key_prefix="development_prediction",
)

development_logits = (
    development_output.predictions
)

if isinstance(
    development_logits,
    tuple
):
    development_logits = (
        development_logits[0]
    )

assert development_logits.shape[0] == len(
    dataset_dict["development"]
)

assert development_logits.shape[2] == 2

# Stable sigmoid; no overflow warnings.
development_probabilities = expit(
    development_logits.astype(
        np.float64
    )
)

print(
    "Development logits shape:",
    development_logits.shape
)

print(
    "Probability range:",
    float(
        development_probabilities.min()
    ),
    "to",
    float(
        development_probabilities.max()
    )
)


# ------------------------------------------------------------
# 2. Recover gold labels and masks
# ------------------------------------------------------------

label_ids = (
    development_output.label_ids
)

assert isinstance(
    label_ids,
    (tuple, list)
)

assert len(label_ids) == 3

development_start_gold = np.asarray(
    label_ids[0]
)

development_end_gold = np.asarray(
    label_ids[1]
)

development_label_mask = np.asarray(
    label_ids[2]
)

assert (
    development_start_gold.shape
    == development_logits.shape[:2]
)

assert (
    development_end_gold.shape
    == development_logits.shape[:2]
)

assert (
    development_label_mask.shape
    == development_logits.shape[:2]
)


# ------------------------------------------------------------
# 3. Boundary metrics over a compact threshold grid
#
# These diagnostics do not choose the final extraction threshold.
# Final selection will use decoded term-level development F1.
# ------------------------------------------------------------

def calculate_binary_prf(
    gold,
    predicted
):
    gold = np.asarray(
        gold,
        dtype=bool
    )

    predicted = np.asarray(
        predicted,
        dtype=bool
    )

    tp = int(
        np.logical_and(
            gold,
            predicted
        ).sum()
    )

    fp = int(
        np.logical_and(
            ~gold,
            predicted
        ).sum()
    )

    fn = int(
        np.logical_and(
            gold,
            ~predicted
        ).sum()
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    f1 = (
        2
        * precision
        * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


active = (
    development_label_mask > 0.5
)

start_gold_active = (
    development_start_gold[
        active
    ] > 0.5
)

end_gold_active = (
    development_end_gold[
        active
    ] > 0.5
)

start_probabilities_active = (
    development_probabilities[
        :, :, 0
    ][active]
)

end_probabilities_active = (
    development_probabilities[
        :, :, 1
    ][active]
)


BOUNDARY_THRESHOLDS = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
]

threshold_rows = []

for threshold in BOUNDARY_THRESHOLDS:
    start_metrics = (
        calculate_binary_prf(
            start_gold_active,
            (
                start_probabilities_active
                >= threshold
            ),
        )
    )

    end_metrics = (
        calculate_binary_prf(
            end_gold_active,
            (
                end_probabilities_active
                >= threshold
            ),
        )
    )

    threshold_rows.append({
        "threshold": threshold,

        "start_precision":
            start_metrics["precision"],
        "start_recall":
            start_metrics["recall"],
        "start_f1":
            start_metrics["f1"],

        "end_precision":
            end_metrics["precision"],
        "end_recall":
            end_metrics["recall"],
        "end_f1":
            end_metrics["f1"],

        "mean_boundary_f1": (
            start_metrics["f1"]
            + end_metrics["f1"]
        ) / 2.0,

        "predicted_start_boundaries":
            start_metrics["tp"]
            + start_metrics["fp"],

        "predicted_end_boundaries":
            end_metrics["tp"]
            + end_metrics["fp"],
    })


boundary_threshold_diagnostics = (
    pd.DataFrame(
        threshold_rows
    )
)

print("\nBoundary-threshold diagnostics:")

display(
    boundary_threshold_diagnostics.round(
        4
    )
)

best_boundary_row = (
    boundary_threshold_diagnostics
    .sort_values(
        "mean_boundary_f1",
        ascending=False
    )
    .iloc[0]
)

print(
    "\nBest diagnostic boundary threshold:",
    best_boundary_row["threshold"]
)

print(
    "Best diagnostic mean boundary F1:",
    best_boundary_row[
        "mean_boundary_f1"
    ]
)


# ------------------------------------------------------------
# 4. Save logits and diagnostics
# ------------------------------------------------------------

DEVELOPMENT_LOGITS_PATH = (
    PREDICTION_DIR /
    f"{RUN_NAME}_development_logits.npz"
)

np.savez_compressed(
    DEVELOPMENT_LOGITS_PATH,
    logits=development_logits.astype(
        np.float32
    ),
    probabilities=(
        development_probabilities.astype(
            np.float32
        )
    ),
    start_gold=(
        development_start_gold.astype(
            np.float32
        )
    ),
    end_gold=(
        development_end_gold.astype(
            np.float32
        )
    ),
    label_mask=(
        development_label_mask.astype(
            np.float32
        )
    ),
)

BOUNDARY_THRESHOLD_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_development_boundary_thresholds.csv"
)

boundary_threshold_diagnostics.to_csv(
    BOUNDARY_THRESHOLD_PATH,
    index=False
)

print("\nSaved development logits to:")
print(DEVELOPMENT_LOGITS_PATH)

print("\nSaved boundary diagnostics to:")
print(BOUNDARY_THRESHOLD_PATH)

print("\nCell 14 completed successfully.")

In [ ]:
# @title
# ============================================================
# CELL 15R — Restore official unique document–term pairs (Run this only of Cell 15 yields error.)
# ============================================================

import pandas as pd


UNIQUE_GOLD_PA_PATH = (
    RESULTS_DIR /
    "taskA_parkinson_unique_document_term_pairs.csv"
)

UNIQUE_GOLD_MH_PATH = (
    RESULTS_DIR /
    "taskA_mental_health_unique_document_term_pairs.csv"
)

DOC_SPLIT_PATH = (
    SPLIT_DIR /
    "taskA_document_holdout_grouped_70_15_15.csv"
)


unique_gold_pa = pd.read_csv(
    UNIQUE_GOLD_PA_PATH,
    dtype={"num_article": str}
)

unique_gold_mh = pd.read_csv(
    UNIQUE_GOLD_MH_PATH,
    dtype={"num_article": str}
)

unique_gold_pairs = pd.concat(
    [
        unique_gold_pa,
        unique_gold_mh,
    ],
    ignore_index=True
)


doc_split_lookup = pd.read_csv(
    DOC_SPLIT_PATH,
    dtype={"num_article": str}
)[
    [
        "domain",
        "num_article",
        "split",
    ]
].drop_duplicates()


# Prevent a duplicate split column if this restoration cell
# is rerun.
if "split" in unique_gold_pairs.columns:
    unique_gold_pairs = (
        unique_gold_pairs.drop(
            columns=["split"]
        )
    )


unique_gold_pairs = (
    unique_gold_pairs.merge(
        doc_split_lookup,
        on=[
            "domain",
            "num_article",
        ],
        how="left",
        validate="many_to_one",
    )
)

assert len(unique_gold_pairs) == 14771

assert (
    unique_gold_pairs[
        "split"
    ].notna().all()
)


print(
    unique_gold_pairs.groupby(
        ["domain", "split"]
    ).size()
)

print(
    "\nRestored unique gold pairs:",
    len(unique_gold_pairs)
)

print(
    "\nCell 15R completed successfully."
)

In [ ]:
# ============================================================
# CELL 15 — Training-derived span-length constraint
# ============================================================

import math

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Calculate PubMedBERT token length of each official term
#
# Test terms are excluded from parameter determination.
# ------------------------------------------------------------

term_length_records = []

for row in unique_gold_pairs.itertuples(
    index=False
):
    if row.split == "test":
        continue

    token_ids = tokenizer(
        str(row.term),
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )["input_ids"]

    term_length_records.append({
        "domain": row.domain,
        "num_article": str(
            row.num_article
        ),
        "split": row.split,
        "term": str(row.term),
        "word_count": len(
            str(row.term).split()
        ),
        "wordpiece_count": len(
            token_ids
        ),
    })


term_lengths = pd.DataFrame(
    term_length_records
)

training_term_lengths = (
    term_lengths[
        term_lengths["split"]
        == "train"
    ].copy()
)

development_term_lengths = (
    term_lengths[
        term_lengths["split"]
        == "development"
    ].copy()
)


# ------------------------------------------------------------
# 2. Distribution report
# ------------------------------------------------------------

print("=" * 78)
print("TERM-LENGTH DISTRIBUTION")
print("=" * 78)

for (
    domain,
    split_name
), sub in term_lengths.groupby(
    ["domain", "split"]
):
    print(
        f"\n{domain} | {split_name}"
    )

    print("Terms:", len(sub))

    print(
        "WordPiece percentiles:"
    )

    print(
        sub["wordpiece_count"]
        .quantile(
            [
                0.50,
                0.75,
                0.90,
                0.95,
                0.99,
                0.995,
                1.00,
            ]
        )
        .round(1)
    )


# ------------------------------------------------------------
# 3. Training-derived maximum
#
# Round the largest training term up to the next multiple of 8.
# A hard safety ceiling of 64 prevents pathological candidate
# pairing if the gold contains an extreme malformed sequence.
# ------------------------------------------------------------

TRAINING_MAX_WORDPIECES = int(
    training_term_lengths[
        "wordpiece_count"
    ].max()
)

TRAINING_P995_WORDPIECES = float(
    training_term_lengths[
        "wordpiece_count"
    ].quantile(0.995)
)

rounded_training_max = int(
    math.ceil(
        TRAINING_MAX_WORDPIECES / 8
    ) * 8
)

MAX_SPAN_TOKENS = min(
    64,
    max(
        16,
        rounded_training_max
    )
)

print("\n" + "=" * 78)
print("SPAN-LENGTH CONSTRAINT")
print("=" * 78)

print(
    "Training maximum WordPieces:",
    TRAINING_MAX_WORDPIECES
)

print(
    "Training 99.5th percentile:",
    TRAINING_P995_WORDPIECES
)

print(
    "Selected MAX_SPAN_TOKENS:",
    MAX_SPAN_TOKENS
)


# ------------------------------------------------------------
# 4. Development coverage under the training-derived maximum
# ------------------------------------------------------------

development_over_limit = (
    development_term_lengths[
        development_term_lengths[
            "wordpiece_count"
        ] > MAX_SPAN_TOKENS
    ]
)

print(
    "\nDevelopment terms over limit:",
    len(development_over_limit)
)

print(
    "Development term-length coverage:",
    (
        1
        - len(
            development_over_limit
        )
        / len(
            development_term_lengths
        )
    )
)

if len(development_over_limit):
    print(
        "\nDevelopment terms over the "
        "training-derived limit:"
    )

    display(
        development_over_limit.sort_values(
            "wordpiece_count",
            ascending=False
        ).head(30)
    )


# ------------------------------------------------------------
# 5. Longest training terms
# ------------------------------------------------------------

print("\nTwenty longest training terms:")

display(
    training_term_lengths[
        [
            "domain",
            "num_article",
            "term",
            "word_count",
            "wordpiece_count",
        ]
    ]
    .sort_values(
        [
            "wordpiece_count",
            "word_count",
        ],
        ascending=False
    )
    .head(20)
)


# ------------------------------------------------------------
# 6. Save audit
# ------------------------------------------------------------

TERM_LENGTH_AUDIT_PATH = (
    RESULTS_DIR /
    "pubmedbert_training_development_term_lengths.csv"
)

term_lengths.to_csv(
    TERM_LENGTH_AUDIT_PATH,
    index=False
)

print("\nSaved term-length audit to:")
print(TERM_LENGTH_AUDIT_PATH)

print("\nCell 15 completed successfully.")

In [ ]:
# ============================================================
# CELL 15A — Robust training-derived span-length constraint
# ============================================================

import math


TRAINING_P999_WORDPIECES = float(
    training_term_lengths[
        "wordpiece_count"
    ].quantile(0.999)
)

MAX_SPAN_TOKENS = max(
    16,
    int(
        math.ceil(
            TRAINING_P999_WORDPIECES / 8
        ) * 8
    )
)

# Keep a general safety ceiling.
MAX_SPAN_TOKENS = min(
    MAX_SPAN_TOKENS,
    64
)


training_over_limit = (
    training_term_lengths[
        training_term_lengths[
            "wordpiece_count"
        ] > MAX_SPAN_TOKENS
    ]
)

development_over_limit = (
    development_term_lengths[
        development_term_lengths[
            "wordpiece_count"
        ] > MAX_SPAN_TOKENS
    ]
)


print("=" * 78)
print("ROBUST SPAN-LENGTH CONSTRAINT")
print("=" * 78)

print(
    "Training 99.9th percentile:",
    TRAINING_P999_WORDPIECES
)

print(
    "Selected MAX_SPAN_TOKENS:",
    MAX_SPAN_TOKENS
)

print(
    "\nTraining terms over limit:",
    len(training_over_limit),
    "of",
    len(training_term_lengths),
)

print(
    "Training coverage:",
    1
    - (
        len(training_over_limit)
        / len(training_term_lengths)
    )
)

print(
    "\nDevelopment terms over limit:",
    len(development_over_limit),
    "of",
    len(development_term_lengths),
)

print(
    "Development coverage:",
    1
    - (
        len(development_over_limit)
        / len(development_term_lengths)
    )
)


if len(training_over_limit):
    print(
        "\nTraining entries excluded by "
        "the robust span limit:"
    )

    display(
        training_over_limit[
            [
                "domain",
                "num_article",
                "term",
                "word_count",
                "wordpiece_count",
            ]
        ]
        .sort_values(
            "wordpiece_count",
            ascending=False
        )
    )


if len(development_over_limit):
    print(
        "\nDevelopment entries over "
        "the training-derived limit:"
    )

    display(
        development_over_limit[
            [
                "domain",
                "num_article",
                "term",
                "word_count",
                "wordpiece_count",
            ]
        ]
        .sort_values(
            "wordpiece_count",
            ascending=False
        )
    )


print(
    "\nCell 15A completed successfully."
)

In [ ]:
# ============================================================
# CELL 16 — Decode development candidate spans
# ============================================================

import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Supporting functions restored after runtime restart
# ------------------------------------------------------------

def read_taska_abstract(
    domain,
    article_id
):
    prefix = (
        "pa_"
        if domain == "parkinson"
        else "mh_"
    )

    abstract_dir = (
        TASKA_TRAIN_ABS_DIR /
        domain
    )

    path = (
        abstract_dir /
        f"{prefix}{article_id}.txt"
    )

    return path.read_text(
        encoding="utf-8",
        errors="replace",
    ).replace(
        "\r\n",
        "\n"
    )


def term_key_light(term):
    term = unicodedata.normalize(
        "NFKC",
        str(term).strip(),
    )

    term = re.sub(
        r"\s+",
        " ",
        term,
    )

    return term.casefold()


# ------------------------------------------------------------
# 2. Candidate-pool configuration
# ------------------------------------------------------------

MIN_BOUNDARY_PROBABILITY = 0.05

print("=" * 78)
print("DEVELOPMENT CANDIDATE DECODING")
print("=" * 78)

print(
    "Minimum boundary probability:",
    MIN_BOUNDARY_PROBABILITY
)

print(
    "Maximum span tokens:",
    MAX_SPAN_TOKENS
)


# ------------------------------------------------------------
# 3. Decode compatible start/end pairs
# ------------------------------------------------------------

candidate_records = []

for example_index in tqdm(
    range(
        len(
            dataset_dict[
                "development"
            ]
        )
    ),
    desc="Decoding development windows"
):
    example = (
        dataset_dict[
            "development"
        ][example_index]
    )

    domain = example["domain"]
    article_id = str(
        example["num_article"]
    )

    text = read_taska_abstract(
        domain,
        article_id
    )

    input_ids = example["input_ids"]

    offsets = [
        tuple(offset)
        for offset in example[
            "offset_mapping"
        ]
    ]

    sequence_length = len(
        input_ids
    )

    start_probabilities = (
        development_probabilities[
            example_index,
            :sequence_length,
            0,
        ]
    )

    end_probabilities = (
        development_probabilities[
            example_index,
            :sequence_length,
            1,
        ]
    )

    valid_token_indices = [
        token_index
        for token_index, (
            start_char,
            end_char
        ) in enumerate(offsets)
        if not (
            start_char == 0
            and end_char == 0
        )
    ]

    start_indices = [
        token_index
        for token_index
        in valid_token_indices
        if (
            start_probabilities[
                token_index
            ]
            >= MIN_BOUNDARY_PROBABILITY
        )
    ]

    end_indices = [
        token_index
        for token_index
        in valid_token_indices
        if (
            end_probabilities[
                token_index
            ]
            >= MIN_BOUNDARY_PROBABILITY
        )
    ]

    # Lists are already in token order.
    for start_token in start_indices:
        maximum_end_token = min(
            start_token
            + MAX_SPAN_TOKENS
            - 1,
            sequence_length - 1,
        )

        for end_token in end_indices:
            if end_token < start_token:
                continue

            if end_token > maximum_end_token:
                break

            start_char = int(
                offsets[
                    start_token
                ][0]
            )

            end_char = int(
                offsets[
                    end_token
                ][1]
            )

            if end_char <= start_char:
                continue

            mention = text[
                start_char:end_char
            ]

            # Terms should not cross DETECH document-section
            # or paragraph boundaries.
            if "\n" in mention:
                continue

            if not mention.strip():
                continue

            # Token offsets should already exclude outer spaces.
            # Keep this check rather than silently changing offsets.
            if mention != mention.strip():
                continue

            p_start = float(
                start_probabilities[
                    start_token
                ]
            )

            p_end = float(
                end_probabilities[
                    end_token
                ]
            )

            p_term = float(
                np.sqrt(
                    p_start * p_end
                )
            )

            candidate_records.append({
                "domain": domain,
                "num_article": article_id,
                "doi": str(
                    example["doi"]
                ),
                "split": "development",
                "window_index": int(
                    example[
                        "window_index"
                    ]
                ),
                "start_token": int(
                    start_token
                ),
                "end_token": int(
                    end_token
                ),
                "span_token_count": int(
                    end_token
                    - start_token
                    + 1
                ),
                "start_char": start_char,
                "end_char": end_char,
                "mention": mention,
                "term_key":
                    term_key_light(mention),
                "p_start": p_start,
                "p_end": p_end,
                "p_term": p_term,
            })


raw_development_candidates = (
    pd.DataFrame(
        candidate_records
    )
)

print(
    "\nRaw window-level candidates:",
    len(
        raw_development_candidates
    )
)


# ------------------------------------------------------------
# 4. Deduplicate candidates from overlapping windows
#
# Preserve the strongest boundary realization.
# ------------------------------------------------------------

development_candidates = (
    raw_development_candidates
    .sort_values(
        [
            "p_term",
            "p_start",
            "p_end",
        ],
        ascending=False,
    )
    .drop_duplicates(
        subset=[
            "domain",
            "num_article",
            "start_char",
            "end_char",
            "mention",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print(
    "Unique occurrence-level candidates:",
    len(development_candidates)
)

print(
    "Candidates removed as duplicate "
    "window predictions:",
    len(raw_development_candidates)
    - len(development_candidates)
)


# ------------------------------------------------------------
# 5. Candidate-pool oracle coverage against official gold
#
# Evaluation uses unique (document, surface term) pairs.
# ------------------------------------------------------------

development_gold = (
    unique_gold_pairs[
        unique_gold_pairs["split"]
        == "development"
    ].copy()
)

oracle_rows = []

for domain in [
    "parkinson",
    "mental_health",
]:
    gold_domain = (
        development_gold[
            development_gold["domain"]
            == domain
        ]
    )

    candidate_domain = (
        development_candidates[
            development_candidates[
                "domain"
            ] == domain
        ]
    )

    gold_set = set(
        zip(
            gold_domain[
                "num_article"
            ].astype(str),
            gold_domain[
                "term"
            ].astype(str),
        )
    )

    candidate_set = set(
        zip(
            candidate_domain[
                "num_article"
            ].astype(str),
            candidate_domain[
                "mention"
            ].astype(str),
        )
    )

    tp = len(
        gold_set & candidate_set
    )

    fn = len(
        gold_set - candidate_set
    )

    oracle_recall = (
        tp / len(gold_set)
        if gold_set
        else 0.0
    )

    oracle_rows.append({
        "domain": domain,
        "gold_pairs": len(
            gold_set
        ),
        "candidate_pairs": len(
            candidate_set
        ),
        "oracle_tp": tp,
        "oracle_fn": fn,
        "oracle_recall":
            oracle_recall,
    })


oracle_coverage = pd.DataFrame(
    oracle_rows
)

print("\nCandidate-pool oracle coverage:")

display(
    oracle_coverage.round(4)
)


# ------------------------------------------------------------
# 6. Confidence and length diagnostics
# ------------------------------------------------------------

print("\nCandidate confidence percentiles:")

print(
    development_candidates[
        "p_term"
    ].quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)

print("\nCandidate span-length percentiles:")

print(
    development_candidates[
        "span_token_count"
    ].quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)


# ------------------------------------------------------------
# 7. Save development candidate pool
# ------------------------------------------------------------

DEVELOPMENT_CANDIDATE_PATH = (
    PREDICTION_DIR /
    f"{RUN_NAME}_development_candidate_pool.csv"
)

DEVELOPMENT_ORACLE_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_development_candidate_oracle.csv"
)

development_candidates.to_csv(
    DEVELOPMENT_CANDIDATE_PATH,
    index=False
)

oracle_coverage.to_csv(
    DEVELOPMENT_ORACLE_PATH,
    index=False
)

print("\nSaved candidate pool to:")
print(DEVELOPMENT_CANDIDATE_PATH)

print("\nSaved oracle coverage to:")
print(DEVELOPMENT_ORACLE_PATH)

print("\nCell 16 completed successfully.")

In [ ]:
# ============================================================
# CELL 17 — Neural-only threshold and overlap-policy tuning
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Official DETECH evaluation
# ------------------------------------------------------------

def calculate_prf_from_sets(
    gold_set,
    predicted_set
):
    tp = len(
        gold_set & predicted_set
    )

    fp = len(
        predicted_set - gold_set
    )

    fn = len(
        gold_set - predicted_set
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    f1 = (
        2
        * precision
        * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def evaluate_taska_domain(
    gold_df,
    prediction_df,
    domain
):
    gold_domain = gold_df[
        gold_df["domain"] == domain
    ].copy()

    prediction_domain = prediction_df[
        prediction_df["domain"] == domain
    ].copy()

    # Mention-level Micro evaluation:
    # unique document + exact surface string.
    gold_micro = set(
        zip(
            gold_domain[
                "num_article"
            ].astype(str),
            gold_domain[
                "term"
            ].astype(str),
        )
    )

    predicted_micro = set(
        zip(
            prediction_domain[
                "num_article"
            ].astype(str),
            prediction_domain[
                "mention"
            ].astype(str),
        )
    )

    # Type evaluation:
    # case- and formatting-insensitive unique types.
    gold_types = set(
        gold_domain[
            "term"
        ].apply(
            term_key_light
        )
    )

    predicted_types = set(
        prediction_domain[
            "mention"
        ].apply(
            term_key_light
        )
    )

    micro = calculate_prf_from_sets(
        gold_micro,
        predicted_micro,
    )

    term_type = calculate_prf_from_sets(
        gold_types,
        predicted_types,
    )

    return {
        "micro_precision":
            micro["precision"],
        "micro_recall":
            micro["recall"],
        "micro_f1":
            micro["f1"],
        "micro_tp":
            micro["tp"],
        "micro_fp":
            micro["fp"],
        "micro_fn":
            micro["fn"],

        "type_precision":
            term_type["precision"],
        "type_recall":
            term_type["recall"],
        "type_f1":
            term_type["f1"],
        "type_tp":
            term_type["tp"],
        "type_fp":
            term_type["fp"],
        "type_fn":
            term_type["fn"],
    }


# ------------------------------------------------------------
# 2. Highest-confidence non-overlap resolution
# ------------------------------------------------------------

def greedy_highest_confidence_nonoverlap(
    candidate_df
):
    kept_groups = []

    for (
        domain,
        article_id
    ), sub in candidate_df.groupby(
        ["domain", "num_article"],
        sort=False
    ):
        ordered = sub.sort_values(
            [
                "p_term",
                "p_start",
                "p_end",
                "span_token_count",
                "start_char",
            ],
            ascending=[
                False,
                False,
                False,
                True,
                True,
            ],
        )

        kept_rows = []
        kept_intervals = []

        for row in ordered.itertuples(
            index=False
        ):
            candidate_start = int(
                row.start_char
            )

            candidate_end = int(
                row.end_char
            )

            overlaps_existing = any(
                max(
                    candidate_start,
                    kept_start,
                )
                < min(
                    candidate_end,
                    kept_end,
                )
                for (
                    kept_start,
                    kept_end
                ) in kept_intervals
            )

            if overlaps_existing:
                continue

            kept_rows.append(
                row._asdict()
            )

            kept_intervals.append(
                (
                    candidate_start,
                    candidate_end,
                )
            )

        if kept_rows:
            kept_groups.append(
                pd.DataFrame(
                    kept_rows
                )
            )

    if not kept_groups:
        return candidate_df.iloc[
            0:0
        ].copy()

    return pd.concat(
        kept_groups,
        ignore_index=True
    )


# ------------------------------------------------------------
# 3. Development threshold grid
# ------------------------------------------------------------

NEURAL_CONFIDENCE_THRESHOLDS = (
    np.round(
        np.arange(
            0.30,
            1.00,
            0.01,
        ),
        2,
    )
    .tolist()
)

tuning_rows = []

for threshold in tqdm(
    NEURAL_CONFIDENCE_THRESHOLDS,
    desc="Tuning neural confidence"
):
    threshold_candidates = (
        development_candidates[
            development_candidates[
                "p_term"
            ] >= threshold
        ].copy()
    )

    policy_predictions = {
        "NESTED":
            threshold_candidates,

        "NONOVERLAP":
            greedy_highest_confidence_nonoverlap(
                threshold_candidates
            ),
    }

    for policy_name, predictions in (
        policy_predictions.items()
    ):
        domain_metrics = {}

        for domain in [
            "parkinson",
            "mental_health",
        ]:
            domain_metrics[domain] = (
                evaluate_taska_domain(
                    development_gold,
                    predictions,
                    domain,
                )
            )

        average_micro_f1 = np.mean(
            [
                domain_metrics[
                    domain
                ]["micro_f1"]
                for domain in [
                    "parkinson",
                    "mental_health",
                ]
            ]
        )

        average_type_f1 = np.mean(
            [
                domain_metrics[
                    domain
                ]["type_f1"]
                for domain in [
                    "parkinson",
                    "mental_health",
                ]
            ]
        )

        selection_score = (
            0.5 * average_micro_f1
            + 0.5 * average_type_f1
        )

        row = {
            "policy": policy_name,
            "p_term_threshold":
                threshold,
            "n_occurrence_predictions":
                len(predictions),
            "n_document_term_pairs":
                len(
                    set(
                        zip(
                            predictions[
                                "domain"
                            ],
                            predictions[
                                "num_article"
                            ].astype(str),
                            predictions[
                                "mention"
                            ],
                        )
                    )
                ),
            "avg_micro_f1":
                average_micro_f1,
            "avg_type_f1":
                average_type_f1,
            "selection_score":
                selection_score,
        }

        for domain in [
            "parkinson",
            "mental_health",
        ]:
            prefix = (
                "PA"
                if domain == "parkinson"
                else "MH"
            )

            for metric_name, value in (
                domain_metrics[
                    domain
                ].items()
            ):
                row[
                    f"{prefix}_{metric_name}"
                ] = value

        tuning_rows.append(row)


neural_tuning = pd.DataFrame(
    tuning_rows
)


# ------------------------------------------------------------
# 4. Select best threshold separately for each policy
# ------------------------------------------------------------

selected_neural_rows = []

for policy_name in [
    "NESTED",
    "NONOVERLAP",
]:
    policy_results = (
        neural_tuning[
            neural_tuning["policy"]
            == policy_name
        ]
        .sort_values(
            [
                "selection_score",
                "avg_micro_f1",
                "avg_type_f1",
                "p_term_threshold",
            ],
            ascending=[
                False,
                False,
                False,
                False,
            ],
        )
    )

    selected_neural_rows.append(
        policy_results.iloc[0]
    )

    print(
        "\n" + "=" * 78
    )

    print(
        f"TOP 10 — {policy_name}"
    )

    print("=" * 78)

    display(
        policy_results[
            [
                "policy",
                "p_term_threshold",
                "n_occurrence_predictions",
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
                "PA_micro_precision",
                "PA_micro_recall",
                "PA_micro_f1",
                "PA_type_f1",
                "MH_micro_precision",
                "MH_micro_recall",
                "MH_micro_f1",
                "MH_type_f1",
            ]
        ]
        .head(10)
        .round(4)
    )


selected_neural = pd.DataFrame(
    selected_neural_rows
).reset_index(drop=True)

print(
    "\n" + "=" * 78
)

print(
    "SELECTED NEURAL BASELINES"
)

print("=" * 78)

display(
    selected_neural[
        [
            "policy",
            "p_term_threshold",
            "n_occurrence_predictions",
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
            "PA_micro_precision",
            "PA_micro_recall",
            "PA_micro_f1",
            "PA_type_precision",
            "PA_type_recall",
            "PA_type_f1",
            "MH_micro_precision",
            "MH_micro_recall",
            "MH_micro_f1",
            "MH_type_precision",
            "MH_type_recall",
            "MH_type_f1",
        ]
    ].round(4)
)


# ------------------------------------------------------------
# 5. Materialize selected development predictions
# ------------------------------------------------------------

selected_prediction_frames = {}

for row in selected_neural.itertuples(
    index=False
):
    threshold = float(
        row.p_term_threshold
    )

    predictions = (
        development_candidates[
            development_candidates[
                "p_term"
            ] >= threshold
        ].copy()
    )

    if row.policy == "NONOVERLAP":
        predictions = (
            greedy_highest_confidence_nonoverlap(
                predictions
            )
        )

    selected_prediction_frames[
        row.policy
    ] = predictions

    output_path = (
        PREDICTION_DIR /
        (
            f"{RUN_NAME}_development_"
            f"{row.policy.lower()}_selected.csv"
        )
    )

    predictions.to_csv(
        output_path,
        index=False
    )

    print(
        f"\nSaved {row.policy} "
        f"predictions to:"
    )

    print(output_path)


# ------------------------------------------------------------
# 6. Save tuning results and selected parameters
# ------------------------------------------------------------

NEURAL_TUNING_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_development_neural_tuning.csv"
)

NEURAL_SELECTION_PATH = (
    RESULTS_DIR /
    f"{RUN_NAME}_development_neural_selection.json"
)

neural_tuning.to_csv(
    NEURAL_TUNING_PATH,
    index=False
)

selection_dictionary = {}

for row in selected_neural.itertuples(
    index=False
):
    selection_dictionary[
        row.policy
    ] = {
        "p_term_threshold":
            float(
                row.p_term_threshold
            ),
        "avg_micro_f1":
            float(
                row.avg_micro_f1
            ),
        "avg_type_f1":
            float(
                row.avg_type_f1
            ),
        "selection_score":
            float(
                row.selection_score
            ),
    }

with open(
    NEURAL_SELECTION_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        selection_dictionary,
        file,
        indent=2
    )

print("\nSaved full tuning table to:")
print(NEURAL_TUNING_PATH)

print("\nSaved selected parameters to:")
print(NEURAL_SELECTION_PATH)

print("\nCell 17 completed successfully.")

In [ ]:
# ============================================================
# CELL 18 — Bidirectional top-k boundary pairing
# ============================================================

from pathlib import Path

import pandas as pd


TOP_K_BOUNDARY_PARTNERS = 2


# ------------------------------------------------------------
# 1. Select strongest end candidates for each start
# ------------------------------------------------------------

best_ends_for_each_start = (
    raw_development_candidates
    .sort_values(
        [
            "domain",
            "num_article",
            "window_index",
            "start_token",
            "p_term",
            "p_end",
            "span_token_count",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            False,
            True,
        ],
    )
    .groupby(
        [
            "domain",
            "num_article",
            "window_index",
            "start_token",
        ],
        sort=False,
        group_keys=False,
    )
    .head(
        TOP_K_BOUNDARY_PARTNERS
    )
)


# ------------------------------------------------------------
# 2. Select strongest start candidates for each end
# ------------------------------------------------------------

best_starts_for_each_end = (
    raw_development_candidates
    .sort_values(
        [
            "domain",
            "num_article",
            "window_index",
            "end_token",
            "p_term",
            "p_start",
            "span_token_count",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            False,
            True,
        ],
    )
    .groupby(
        [
            "domain",
            "num_article",
            "window_index",
            "end_token",
        ],
        sort=False,
        group_keys=False,
    )
    .head(
        TOP_K_BOUNDARY_PARTNERS
    )
)


# ------------------------------------------------------------
# 3. Union and deduplicate across directions and windows
# ------------------------------------------------------------

paired_window_candidates = pd.concat(
    [
        best_ends_for_each_start,
        best_starts_for_each_end,
    ],
    ignore_index=True,
)

paired_window_candidates = (
    paired_window_candidates
    .sort_values(
        [
            "p_term",
            "p_start",
            "p_end",
            "span_token_count",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=[
            "domain",
            "num_article",
            "window_index",
            "start_token",
            "end_token",
            "start_char",
            "end_char",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


paired_development_candidates = (
    paired_window_candidates
    .sort_values(
        [
            "p_term",
            "p_start",
            "p_end",
            "span_token_count",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )
    .drop_duplicates(
        subset=[
            "domain",
            "num_article",
            "start_char",
            "end_char",
            "mention",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)


print("=" * 78)
print("CONSTRAINED BIDIRECTIONAL PAIRING")
print("=" * 78)

print(
    "Top partners per boundary:",
    TOP_K_BOUNDARY_PARTNERS
)

print(
    "Complete cross-product candidates:",
    len(
        raw_development_candidates
    )
)

print(
    "Window-level constrained candidates:",
    len(
        paired_window_candidates
    )
)

print(
    "Occurrence-level constrained candidates:",
    len(
        paired_development_candidates
    )
)

print(
    "Reduction from complete pairing:",
    1
    - (
        len(
            paired_development_candidates
        )
        / len(
            development_candidates
        )
    )
)


# ------------------------------------------------------------
# 4. Oracle recall of constrained pool
# ------------------------------------------------------------

paired_oracle_rows = []

for domain in [
    "parkinson",
    "mental_health",
]:
    gold_domain = (
        development_gold[
            development_gold["domain"]
            == domain
        ]
    )

    candidate_domain = (
        paired_development_candidates[
            paired_development_candidates[
                "domain"
            ] == domain
        ]
    )

    gold_set = set(
        zip(
            gold_domain[
                "num_article"
            ].astype(str),
            gold_domain[
                "term"
            ].astype(str),
        )
    )

    candidate_set = set(
        zip(
            candidate_domain[
                "num_article"
            ].astype(str),
            candidate_domain[
                "mention"
            ].astype(str),
        )
    )

    tp = len(
        gold_set & candidate_set
    )

    fn = len(
        gold_set - candidate_set
    )

    paired_oracle_rows.append({
        "domain": domain,
        "gold_pairs": len(
            gold_set
        ),
        "candidate_pairs": len(
            candidate_set
        ),
        "oracle_tp": tp,
        "oracle_fn": fn,
        "oracle_recall": (
            tp / len(gold_set)
            if gold_set
            else 0.0
        ),
    })


paired_oracle_coverage = (
    pd.DataFrame(
        paired_oracle_rows
    )
)

print("\nConstrained-pool oracle coverage:")

display(
    paired_oracle_coverage.round(4)
)


# ------------------------------------------------------------
# 5. Candidate-count diagnostics by document
# ------------------------------------------------------------

candidate_counts_by_document = (
    paired_development_candidates
    .groupby(
        [
            "domain",
            "num_article",
        ]
    )
    .size()
    .rename(
        "candidate_count"
    )
    .reset_index()
)

print(
    "\nCandidate counts per document:"
)

display(
    candidate_counts_by_document.groupby(
        "domain"
    )["candidate_count"].describe().round(2)
)


# ------------------------------------------------------------
# 6. Save constrained pool
# ------------------------------------------------------------

PAIRED_DEVELOPMENT_CANDIDATE_PATH = (
    PREDICTION_DIR /
    (
        f"{RUN_NAME}_development_"
        f"bidirectional_top"
        f"{TOP_K_BOUNDARY_PARTNERS}_"
        f"candidate_pool.csv"
    )
)

PAIRED_ORACLE_PATH = (
    RESULTS_DIR /
    (
        f"{RUN_NAME}_development_"
        f"bidirectional_top"
        f"{TOP_K_BOUNDARY_PARTNERS}_"
        f"oracle.csv"
    )
)

paired_development_candidates.to_csv(
    PAIRED_DEVELOPMENT_CANDIDATE_PATH,
    index=False,
)

paired_oracle_coverage.to_csv(
    PAIRED_ORACLE_PATH,
    index=False,
)

print("\nSaved constrained pool to:")
print(
    PAIRED_DEVELOPMENT_CANDIDATE_PATH
)

print("\nCell 18 completed successfully.")

In [ ]:
# ============================================================
# CELL 18A — Full pairing with robust 16-token constraint
# ============================================================

ROBUST_MAX_SPAN_TOKENS = 16

# Freeze the corrected constraint for subsequent experiments.
MAX_SPAN_TOKENS = ROBUST_MAX_SPAN_TOKENS


# ------------------------------------------------------------
# 1. Filter the complete-pairing pool by the robust limit
# ------------------------------------------------------------

robust_development_candidates = (
    development_candidates[
        development_candidates[
            "span_token_count"
        ] <= MAX_SPAN_TOKENS
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 78)
print("ROBUST FULL-PAIRING CANDIDATE POOL")
print("=" * 78)

print(
    "Maximum span tokens:",
    MAX_SPAN_TOKENS
)

print(
    "Original 40-token candidates:",
    len(development_candidates)
)

print(
    "Robust 16-token candidates:",
    len(
        robust_development_candidates
    )
)

print(
    "Candidate reduction:",
    1
    - (
        len(
            robust_development_candidates
        )
        / len(
            development_candidates
        )
    )
)


# ------------------------------------------------------------
# 2. Gold-length coverage under the 16-token constraint
# ------------------------------------------------------------

training_over_robust_limit = (
    training_term_lengths[
        training_term_lengths[
            "wordpiece_count"
        ] > MAX_SPAN_TOKENS
    ]
)

development_over_robust_limit = (
    development_term_lengths[
        development_term_lengths[
            "wordpiece_count"
        ] > MAX_SPAN_TOKENS
    ]
)

print(
    "\nTraining terms over 16 tokens:",
    len(
        training_over_robust_limit
    ),
    "of",
    len(training_term_lengths),
)

print(
    "Training term-length coverage:",
    1
    - (
        len(
            training_over_robust_limit
        )
        / len(
            training_term_lengths
        )
    )
)

print(
    "\nDevelopment terms over 16 tokens:",
    len(
        development_over_robust_limit
    ),
    "of",
    len(development_term_lengths),
)

print(
    "Development term-length coverage:",
    1
    - (
        len(
            development_over_robust_limit
        )
        / len(
            development_term_lengths
        )
    )
)


# ------------------------------------------------------------
# 3. Candidate-pool oracle coverage
# ------------------------------------------------------------

robust_oracle_rows = []

for domain in [
    "parkinson",
    "mental_health",
]:
    gold_domain = (
        development_gold[
            development_gold["domain"]
            == domain
        ]
    )

    candidate_domain = (
        robust_development_candidates[
            robust_development_candidates[
                "domain"
            ] == domain
        ]
    )

    gold_set = set(
        zip(
            gold_domain[
                "num_article"
            ].astype(str),
            gold_domain[
                "term"
            ].astype(str),
        )
    )

    candidate_set = set(
        zip(
            candidate_domain[
                "num_article"
            ].astype(str),
            candidate_domain[
                "mention"
            ].astype(str),
        )
    )

    tp = len(
        gold_set & candidate_set
    )

    fn = len(
        gold_set - candidate_set
    )

    robust_oracle_rows.append({
        "domain": domain,
        "gold_pairs": len(
            gold_set
        ),
        "candidate_pairs": len(
            candidate_set
        ),
        "oracle_tp": tp,
        "oracle_fn": fn,
        "oracle_recall": (
            tp / len(gold_set)
            if gold_set
            else 0.0
        ),
    })


robust_oracle_coverage = (
    pd.DataFrame(
        robust_oracle_rows
    )
)

print(
    "\nRobust candidate-pool oracle coverage:"
)

display(
    robust_oracle_coverage.round(4)
)


# ------------------------------------------------------------
# 4. Inspect development terms excluded by length
# ------------------------------------------------------------

if len(
    development_over_robust_limit
):
    print(
        "\nDevelopment gold entries "
        "over 16 WordPieces:"
    )

    display(
        development_over_robust_limit[
            [
                "domain",
                "num_article",
                "term",
                "word_count",
                "wordpiece_count",
            ]
        ]
        .sort_values(
            "wordpiece_count",
            ascending=False
        )
    )


# ------------------------------------------------------------
# 5. Save corrected candidate pool
# ------------------------------------------------------------

ROBUST_DEVELOPMENT_CANDIDATE_PATH = (
    PREDICTION_DIR /
    (
        f"{RUN_NAME}_development_"
        f"full_pairing_max"
        f"{MAX_SPAN_TOKENS}_"
        f"candidate_pool.csv"
    )
)

ROBUST_ORACLE_PATH = (
    RESULTS_DIR /
    (
        f"{RUN_NAME}_development_"
        f"full_pairing_max"
        f"{MAX_SPAN_TOKENS}_"
        f"oracle.csv"
    )
)

robust_development_candidates.to_csv(
    ROBUST_DEVELOPMENT_CANDIDATE_PATH,
    index=False,
)

robust_oracle_coverage.to_csv(
    ROBUST_ORACLE_PATH,
    index=False,
)

print("\nSaved robust candidate pool to:")
print(
    ROBUST_DEVELOPMENT_CANDIDATE_PATH
)

print("\nCell 18A completed successfully.")

In [ ]:
# ============================================================
# CELL 19 — Activate robust candidate pool for neural tuning
# ============================================================

development_candidates = (
    robust_development_candidates.copy()
)

assert (
    development_candidates[
        "span_token_count"
    ].max()
    <= 16
)

print(
    "Active development candidates:",
    len(development_candidates)
)

print(
    "Maximum span-token count:",
    development_candidates[
        "span_token_count"
    ].max()
)

print(
    "\nCell 19 completed successfully."
)

In [ ]:
# ============================================================
# CELL 20 — Training logits and robust candidate pool
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import expit
from tqdm.auto import tqdm


# ------------------------------------------------------------
# 1. Generate training-window logits
#
# Disable token diagnostics for this pass to avoid the old
# NumPy sigmoid warnings and unnecessary metric computation.
# ------------------------------------------------------------

print("=" * 78)
print("GENERATING TRAINING LOGITS")
print("=" * 78)

saved_compute_metrics = (
    trainer.compute_metrics
)

trainer.compute_metrics = None

training_output = trainer.predict(
    dataset_dict["train"],
    metric_key_prefix="training_prediction",
)

trainer.compute_metrics = (
    saved_compute_metrics
)

training_logits = (
    training_output.predictions
)

if isinstance(
    training_logits,
    tuple
):
    training_logits = (
        training_logits[0]
    )

training_probabilities = expit(
    training_logits.astype(
        np.float64
    )
)

print(
    "Training logits shape:",
    training_logits.shape
)

print(
    "Probability range:",
    float(
        training_probabilities.min()
    ),
    "to",
    float(
        training_probabilities.max()
    )
)


# ------------------------------------------------------------
# 2. General full-pairing decoder
# ------------------------------------------------------------

def decode_candidate_pool(
    window_dataset,
    probabilities,
    max_span_tokens,
    minimum_boundary_probability=0.05,
    split_name=None,
):
    candidate_records = []

    for example_index in tqdm(
        range(len(window_dataset)),
        desc=(
            f"Decoding {split_name or 'dataset'} "
            f"windows"
        ),
    ):
        example = window_dataset[
            example_index
        ]

        domain = example["domain"]
        article_id = str(
            example["num_article"]
        )

        text = read_taska_abstract(
            domain,
            article_id
        )

        offsets = [
            tuple(offset)
            for offset in example[
                "offset_mapping"
            ]
        ]

        sequence_length = len(
            example["input_ids"]
        )

        start_probabilities = (
            probabilities[
                example_index,
                :sequence_length,
                0,
            ]
        )

        end_probabilities = (
            probabilities[
                example_index,
                :sequence_length,
                1,
            ]
        )

        valid_token_indices = [
            token_index
            for token_index, (
                start_char,
                end_char
            ) in enumerate(offsets)
            if not (
                start_char == 0
                and end_char == 0
            )
        ]

        start_indices = [
            token_index
            for token_index
            in valid_token_indices
            if (
                start_probabilities[
                    token_index
                ]
                >= minimum_boundary_probability
            )
        ]

        end_indices = [
            token_index
            for token_index
            in valid_token_indices
            if (
                end_probabilities[
                    token_index
                ]
                >= minimum_boundary_probability
            )
        ]

        for start_token in start_indices:
            maximum_end_token = min(
                start_token
                + max_span_tokens
                - 1,
                sequence_length - 1,
            )

            for end_token in end_indices:
                if end_token < start_token:
                    continue

                if end_token > maximum_end_token:
                    break

                start_char = int(
                    offsets[
                        start_token
                    ][0]
                )

                end_char = int(
                    offsets[
                        end_token
                    ][1]
                )

                if end_char <= start_char:
                    continue

                mention = text[
                    start_char:end_char
                ]

                if "\n" in mention:
                    continue

                if not mention.strip():
                    continue

                if mention != mention.strip():
                    continue

                p_start = float(
                    start_probabilities[
                        start_token
                    ]
                )

                p_end = float(
                    end_probabilities[
                        end_token
                    ]
                )

                p_term = float(
                    np.sqrt(
                        p_start * p_end
                    )
                )

                candidate_records.append({
                    "domain": domain,
                    "num_article":
                        article_id,
                    "doi": str(
                        example["doi"]
                    ),
                    "split": (
                        split_name
                        or example["split"]
                    ),
                    "window_index": int(
                        example[
                            "window_index"
                        ]
                    ),
                    "start_token": int(
                        start_token
                    ),
                    "end_token": int(
                        end_token
                    ),
                    "span_token_count": int(
                        end_token
                        - start_token
                        + 1
                    ),
                    "start_char":
                        start_char,
                    "end_char":
                        end_char,
                    "mention":
                        mention,
                    "term_key":
                        term_key_light(
                            mention
                        ),
                    "p_start":
                        p_start,
                    "p_end":
                        p_end,
                    "p_term":
                        p_term,
                })

    raw_candidates = pd.DataFrame(
        candidate_records
    )

    deduplicated_candidates = (
        raw_candidates
        .sort_values(
            [
                "p_term",
                "p_start",
                "p_end",
                "span_token_count",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .drop_duplicates(
            subset=[
                "domain",
                "num_article",
                "start_char",
                "end_char",
                "mention",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

    return (
        raw_candidates,
        deduplicated_candidates,
    )


# ------------------------------------------------------------
# 3. Decode training candidates with frozen constraints
# ------------------------------------------------------------

(
    raw_training_candidates,
    training_candidates,
) = decode_candidate_pool(
    window_dataset=(
        dataset_dict["train"]
    ),
    probabilities=(
        training_probabilities
    ),
    max_span_tokens=(
        MAX_SPAN_TOKENS
    ),
    minimum_boundary_probability=(
        MIN_BOUNDARY_PROBABILITY
    ),
    split_name="train",
)


# ------------------------------------------------------------
# 4. Diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("TRAINING CANDIDATE POOL")
print("=" * 78)

print(
    "Raw window-level candidates:",
    len(raw_training_candidates)
)

print(
    "Unique occurrence-level candidates:",
    len(training_candidates)
)

print(
    "Duplicate window candidates removed:",
    len(raw_training_candidates)
    - len(training_candidates)
)

print("\nCandidates by domain:")

print(
    training_candidates.groupby(
        "domain"
    ).size()
)

print("\nConfidence percentiles:")

print(
    training_candidates[
        "p_term"
    ].quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)

print("\nSpan-length percentiles:")

print(
    training_candidates[
        "span_token_count"
    ].quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)


# ------------------------------------------------------------
# 5. Save logits and candidate pool
# ------------------------------------------------------------

TRAINING_LOGITS_PATH = (
    PREDICTION_DIR /
    f"{RUN_NAME}_training_logits.npz"
)

np.savez_compressed(
    TRAINING_LOGITS_PATH,
    logits=training_logits.astype(
        np.float32
    ),
    probabilities=(
        training_probabilities.astype(
            np.float32
        )
    ),
)

TRAINING_CANDIDATE_PATH = (
    PREDICTION_DIR /
    (
        f"{RUN_NAME}_training_"
        f"full_pairing_max"
        f"{MAX_SPAN_TOKENS}_"
        f"candidate_pool.csv"
    )
)

training_candidates.to_csv(
    TRAINING_CANDIDATE_PATH,
    index=False,
)

print("\nSaved training logits to:")
print(TRAINING_LOGITS_PATH)

print("\nSaved training candidate pool to:")
print(TRAINING_CANDIDATE_PATH)

print("\nCell 20 completed successfully.")

In [ ]:
# ============================================================
# CELL 21 — Frozen contextual candidate embeddings
# ============================================================

import json
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel


# ------------------------------------------------------------
# 1. Fixed collapse-eligibility floor
# ------------------------------------------------------------

COLLAPSE_ELIGIBILITY_FLOOR = 0.50

eligible_training_candidates = (
    training_candidates[
        training_candidates[
            "p_term"
        ] >= COLLAPSE_ELIGIBILITY_FLOOR
    ]
    .copy()
    .reset_index(drop=True)
)

eligible_development_candidates = (
    robust_development_candidates[
        robust_development_candidates[
            "p_term"
        ] >= COLLAPSE_ELIGIBILITY_FLOOR
    ]
    .copy()
    .reset_index(drop=True)
)

eligible_training_candidates[
    "candidate_id"
] = np.arange(
    len(
        eligible_training_candidates
    ),
    dtype=np.int64,
)

eligible_development_candidates[
    "candidate_id"
] = np.arange(
    len(
        eligible_development_candidates
    ),
    dtype=np.int64,
)


print("=" * 78)
print("COLLAPSE-ELIGIBLE CANDIDATES")
print("=" * 78)

print(
    "Eligibility floor:",
    COLLAPSE_ELIGIBILITY_FLOOR
)

print(
    "Training candidates:",
    len(
        eligible_training_candidates
    )
)

print(
    "Development candidates:",
    len(
        eligible_development_candidates
    )
)

print("\nTraining candidates by domain:")

print(
    eligible_training_candidates.groupby(
        "domain"
    ).size()
)

print(
    "\nDevelopment candidates by domain:"
)

print(
    eligible_development_candidates.groupby(
        "domain"
    ).size()
)


# ------------------------------------------------------------
# 2. Free extractor GPU memory temporarily
# ------------------------------------------------------------

trainer.model.to("cpu")

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ------------------------------------------------------------
# 3. Load frozen pretrained PubMedBERT encoder
# ------------------------------------------------------------

contextual_encoder = (
    AutoModel.from_pretrained(
        MODEL_DIR
    )
)

contextual_encoder.to(DEVICE)
contextual_encoder.eval()

for parameter in (
    contextual_encoder.parameters()
):
    parameter.requires_grad = False


CONTEXT_BATCH_SIZE = (
    24
    if gpu_memory_gb >= 30
    else 8
)

HIDDEN_SIZE = int(
    contextual_encoder.config.hidden_size
)

print(
    "\nContextual encoder hidden size:",
    HIDDEN_SIZE
)

print(
    "Context batch size:",
    CONTEXT_BATCH_SIZE
)


# ------------------------------------------------------------
# 4. Model-input padding without labels
# ------------------------------------------------------------

def collate_model_inputs(
    examples
):
    model_features = []

    for example in examples:
        feature = {
            "input_ids":
                example["input_ids"],
            "attention_mask":
                example[
                    "attention_mask"
                ],
        }

        if "token_type_ids" in example:
            feature[
                "token_type_ids"
            ] = example[
                "token_type_ids"
            ]

        model_features.append(
            feature
        )

    return tokenizer.pad(
        model_features,
        padding=True,
        pad_to_multiple_of=8,
        return_tensors="pt",
    )


# ------------------------------------------------------------
# 5. Efficient span-pooling function
# ------------------------------------------------------------

@torch.no_grad()
def compute_candidate_embeddings(
    window_dataset,
    candidate_df,
    description
):
    candidates = (
        candidate_df
        .copy()
        .reset_index(drop=True)
    )

    embeddings = np.zeros(
        (
            len(candidates),
            HIDDEN_SIZE,
        ),
        dtype=np.float32,
    )

    filled = np.zeros(
        len(candidates),
        dtype=bool,
    )

    # Map each document window to candidate row positions.
    grouped_positions = (
        candidates.groupby(
            [
                "domain",
                "num_article",
                "window_index",
            ],
            sort=False,
        ).indices
    )

    for batch_start in tqdm(
        range(
            0,
            len(window_dataset),
            CONTEXT_BATCH_SIZE,
        ),
        desc=description,
    ):
        batch_end = min(
            batch_start
            + CONTEXT_BATCH_SIZE,
            len(window_dataset),
        )

        batch_examples = [
            window_dataset[index]
            for index in range(
                batch_start,
                batch_end,
            )
        ]

        model_batch = (
            collate_model_inputs(
                batch_examples
            )
        )

        model_batch = {
            key: value.to(DEVICE)
            for key, value
            in model_batch.items()
        }

        autocast_context = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.bfloat16,
            )
            if USE_BF16
            else (
                torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                )
                if USE_FP16
                else nullcontext()
            )
        )

        with autocast_context:
            hidden_batch = (
                contextual_encoder(
                    **model_batch
                ).last_hidden_state
            )

        for local_index, example in enumerate(
            batch_examples
        ):
            key = (
                example["domain"],
                str(example["num_article"]),
                int(
                    example[
                        "window_index"
                    ]
                ),
            )

            candidate_positions = (
                grouped_positions.get(
                    key
                )
            )

            if candidate_positions is None:
                continue

            candidate_positions = (
                np.asarray(
                    candidate_positions,
                    dtype=np.int64,
                )
            )

            sub = candidates.iloc[
                candidate_positions
            ]

            starts = torch.tensor(
                sub[
                    "start_token"
                ].to_numpy(),
                dtype=torch.long,
                device=DEVICE,
            )

            ends_plus_one = torch.tensor(
                (
                    sub[
                        "end_token"
                    ].to_numpy()
                    + 1
                ),
                dtype=torch.long,
                device=DEVICE,
            )

            sequence_hidden = (
                hidden_batch[
                    local_index
                ]
            )

            # prefix[i] = sum of token vectors before i
            prefix = torch.cat(
                [
                    torch.zeros(
                        (
                            1,
                            HIDDEN_SIZE,
                        ),
                        dtype=(
                            sequence_hidden.dtype
                        ),
                        device=DEVICE,
                    ),
                    sequence_hidden.cumsum(
                        dim=0
                    ),
                ],
                dim=0,
            )

            lengths = (
                ends_plus_one - starts
            ).to(
                sequence_hidden.dtype
            ).unsqueeze(1)

            pooled = (
                prefix[
                    ends_plus_one
                ]
                - prefix[
                    starts
                ]
            ) / lengths

            embeddings[
                candidate_positions
            ] = (
                pooled.float()
                .cpu()
                .numpy()
            )

            filled[
                candidate_positions
            ] = True

        del hidden_batch
        del model_batch

    if not filled.all():
        missing_positions = np.where(
            ~filled
        )[0]

        raise RuntimeError(
            f"{len(missing_positions)} "
            f"candidate embeddings were not filled. "
            f"First missing positions: "
            f"{missing_positions[:10]}"
        )

    return embeddings


# ------------------------------------------------------------
# 6. Compute training and development embeddings
# ------------------------------------------------------------

training_candidate_embeddings = (
    compute_candidate_embeddings(
        window_dataset=(
            dataset_dict["train"]
        ),
        candidate_df=(
            eligible_training_candidates
        ),
        description=(
            "Embedding training candidates"
        ),
    )
)

development_candidate_embeddings = (
    compute_candidate_embeddings(
        window_dataset=(
            dataset_dict[
                "development"
            ]
        ),
        candidate_df=(
            eligible_development_candidates
        ),
        description=(
            "Embedding development candidates"
        ),
    )
)


# ------------------------------------------------------------
# 7. Validate and report
# ------------------------------------------------------------

print("\n" + "=" * 78)
print("CONTEXTUAL EMBEDDINGS")
print("=" * 78)

print(
    "Training embedding shape:",
    training_candidate_embeddings.shape
)

print(
    "Development embedding shape:",
    development_candidate_embeddings.shape
)

print(
    "Training finite values:",
    bool(
        np.isfinite(
            training_candidate_embeddings
        ).all()
    )
)

print(
    "Development finite values:",
    bool(
        np.isfinite(
            development_candidate_embeddings
        ).all()
    )
)

print(
    "Training mean vector norm:",
    float(
        np.linalg.norm(
            training_candidate_embeddings,
            axis=1,
        ).mean()
    )
)

print(
    "Development mean vector norm:",
    float(
        np.linalg.norm(
            development_candidate_embeddings,
            axis=1,
        ).mean()
    )
)

assert (
    training_candidate_embeddings.shape
    == (
        len(
            eligible_training_candidates
        ),
        HIDDEN_SIZE,
    )
)

assert (
    development_candidate_embeddings.shape
    == (
        len(
            eligible_development_candidates
        ),
        HIDDEN_SIZE,
    )
)

assert np.isfinite(
    training_candidate_embeddings
).all()

assert np.isfinite(
    development_candidate_embeddings
).all()


# ------------------------------------------------------------
# 8. Save candidates and embedding matrices
# ------------------------------------------------------------

TRAIN_ELIGIBLE_PATH = (
    PREDICTION_DIR /
    (
        f"{RUN_NAME}_training_"
        f"collapse_eligible_"
        f"p{COLLAPSE_ELIGIBILITY_FLOOR}.csv"
    )
)

DEV_ELIGIBLE_PATH = (
    PREDICTION_DIR /
    (
        f"{RUN_NAME}_development_"
        f"collapse_eligible_"
        f"p{COLLAPSE_ELIGIBILITY_FLOOR}.csv"
    )
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR /
    (
        f"{RUN_NAME}_training_"
        f"candidate_embeddings_"
        f"p{COLLAPSE_ELIGIBILITY_FLOOR}.npy"
    )
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR /
    (
        f"{RUN_NAME}_development_"
        f"candidate_embeddings_"
        f"p{COLLAPSE_ELIGIBILITY_FLOOR}.npy"
    )
)

eligible_training_candidates.to_csv(
    TRAIN_ELIGIBLE_PATH,
    index=False,
)

eligible_development_candidates.to_csv(
    DEV_ELIGIBLE_PATH,
    index=False,
)

np.save(
    TRAIN_EMBEDDING_PATH,
    training_candidate_embeddings,
)

np.save(
    DEV_EMBEDDING_PATH,
    development_candidate_embeddings,
)


EMBEDDING_METADATA_PATH = (
    CACHE_DIR /
    (
        f"{RUN_NAME}_candidate_"
        f"embedding_metadata.json"
    )
)

embedding_metadata = {
    "encoder": str(MODEL_DIR),
    "encoder_status": (
        "frozen pretrained PubMedBERT"
    ),
    "pooling": "mean",
    "eligibility_floor":
        COLLAPSE_ELIGIBILITY_FLOOR,
    "max_span_tokens":
        MAX_SPAN_TOKENS,
    "hidden_size":
        HIDDEN_SIZE,
    "training_candidates":
        len(
            eligible_training_candidates
        ),
    "development_candidates":
        len(
            eligible_development_candidates
        ),
    "training_embedding_path":
        str(
            TRAIN_EMBEDDING_PATH
        ),
    "development_embedding_path":
        str(
            DEV_EMBEDDING_PATH
        ),
}

with open(
    EMBEDDING_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        embedding_metadata,
        file,
        indent=2,
    )


print("\nSaved training embeddings to:")
print(TRAIN_EMBEDDING_PATH)

print("\nSaved development embeddings to:")
print(DEV_EMBEDDING_PATH)

print("\nCell 21 completed successfully.")

In [ ]:
# CELL 22 — Fit training-only collapse states and score development candidates

import json
import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Fixed paths and parameters
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
MODEL_STATE_DIR = EXPERIMENT_DIR / "models/collapse_states_seed42"

for directory in [
    CACHE_DIR,
    PREDICTION_DIR,
    RESULTS_DIR,
    MODEL_STATE_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
K_CLUSTERS = 100
ELIGIBILITY_FLOOR = 0.50

TRAIN_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_full_pairing_max16_candidate_pool.csv"
)

DEV_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_full_pairing_max16_candidate_pool.csv"
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_training_candidate_embeddings_p0.5.npy"
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_development_candidate_embeddings_p0.5.npy"
)

TRAIN_OUTPUT_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

DEV_OUTPUT_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_candidates_with_collapse.csv"
)

METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_collapse_training_metadata.json"
)

SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_collapse_distribution_summary.csv"
)

# ---------------------------------------------------------------------
# Load candidate tables and reproduce Cell 21 eligibility/order
# ---------------------------------------------------------------------

train_candidates = pd.read_csv(TRAIN_POOL_PATH)
dev_candidates = pd.read_csv(DEV_POOL_PATH)

required_columns = {"domain", "p_term"}

for table_name, table in [
    ("training", train_candidates),
    ("development", dev_candidates),
]:
    missing = required_columns.difference(table.columns)
    if missing:
        raise ValueError(
            f"{table_name} candidate table is missing columns: {sorted(missing)}"
        )

train_eligible = (
    train_candidates.loc[
        train_candidates["p_term"] >= ELIGIBILITY_FLOOR
    ]
    .reset_index(drop=True)
    .copy()
)

dev_eligible = (
    dev_candidates.loc[
        dev_candidates["p_term"] >= ELIGIBILITY_FLOOR
    ]
    .reset_index(drop=True)
    .copy()
)

train_embeddings = np.load(
    TRAIN_EMBEDDING_PATH,
    mmap_mode="r",
)

dev_embeddings = np.load(
    DEV_EMBEDDING_PATH,
    mmap_mode="r",
)

if len(train_eligible) != train_embeddings.shape[0]:
    raise ValueError(
        "Training candidate/embedding mismatch: "
        f"{len(train_eligible)} rows versus "
        f"{train_embeddings.shape[0]} embeddings."
    )

if len(dev_eligible) != dev_embeddings.shape[0]:
    raise ValueError(
        "Development candidate/embedding mismatch: "
        f"{len(dev_eligible)} rows versus "
        f"{dev_embeddings.shape[0]} embeddings."
    )

train_eligible.insert(
    0,
    "candidate_id",
    np.arange(len(train_eligible), dtype=np.int64),
)

dev_eligible.insert(
    0,
    "candidate_id",
    np.arange(len(dev_eligible), dtype=np.int64),
)

print("=" * 78)
print("TRAINING-ONLY COLLAPSE FIT")
print("=" * 78)
print(f"Clusters per domain: {K_CLUSTERS}")
print(f"Random seed: {SEED}")
print(f"Eligibility floor: {ELIGIBILITY_FLOOR}")
print(f"Training candidates: {len(train_eligible)}")
print(f"Development candidates: {len(dev_eligible)}")
print()

# Allocate result arrays in original eligible-candidate order
train_cluster = np.full(len(train_eligible), -1, dtype=np.int32)
dev_cluster = np.full(len(dev_eligible), -1, dtype=np.int32)

train_distance = np.full(len(train_eligible), np.nan, dtype=np.float32)
dev_distance = np.full(len(dev_eligible), np.nan, dtype=np.float32)

train_collapse = np.full(len(train_eligible), np.nan, dtype=np.float32)
dev_collapse = np.full(len(dev_eligible), np.nan, dtype=np.float32)

domain_metadata = {}
distribution_rows = []

# ---------------------------------------------------------------------
# Fit a separate training-only state for each official DETECH domain
# ---------------------------------------------------------------------

for domain in sorted(train_eligible["domain"].unique()):
    print("-" * 78)
    print(f"DOMAIN: {domain}")
    print("-" * 78)

    train_index = np.flatnonzero(
        train_eligible["domain"].to_numpy() == domain
    )

    dev_index = np.flatnonzero(
        dev_eligible["domain"].to_numpy() == domain
    )

    if len(train_index) < K_CLUSTERS:
        raise ValueError(
            f"{domain} has only {len(train_index)} training candidates, "
            f"which is insufficient for K={K_CLUSTERS}."
        )

    # Materialize one domain at a time to control RAM use.
    X_train = np.asarray(
        train_embeddings[train_index],
        dtype=np.float32,
    )

    X_dev = np.asarray(
        dev_embeddings[dev_index],
        dtype=np.float32,
    )

    print(f"Training candidates: {len(train_index)}")
    print(f"Development candidates: {len(dev_index)}")
    print(f"Embedding dimensions: {X_train.shape[1]}")
    print("Fitting training StandardScaler...")

    scaler = StandardScaler(
        copy=True,
        with_mean=True,
        with_std=True,
    )

    X_train_scaled = scaler.fit_transform(X_train).astype(
        np.float32,
        copy=False,
    )

    X_dev_scaled = scaler.transform(X_dev).astype(
        np.float32,
        copy=False,
    )

    print("Fitting training KMeans...")
    start_time = time.time()

    kmeans = KMeans(
        n_clusters=K_CLUSTERS,
        init="k-means++",
        n_init=10,
        max_iter=300,
        tol=1e-4,
        random_state=SEED,
        algorithm="lloyd",
        verbose=0,
    )

    train_labels = kmeans.fit_predict(X_train_scaled)
    elapsed_minutes = (time.time() - start_time) / 60

    # The development set only uses the already fitted state.
    dev_labels = kmeans.predict(X_dev_scaled)

    centers = kmeans.cluster_centers_.astype(
        np.float32,
        copy=False,
    )

    domain_train_distance = np.linalg.norm(
        X_train_scaled - centers[train_labels],
        axis=1,
    ).astype(np.float32)

    domain_dev_distance = np.linalg.norm(
        X_dev_scaled - centers[dev_labels],
        axis=1,
    ).astype(np.float32)

    # Temperature estimated from training distances only.
    q25, q75 = np.quantile(
        domain_train_distance,
        [0.25, 0.75],
    )

    temperature = float(q75 - q25)

    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError(
            f"Invalid training-derived IQR temperature for {domain}: "
            f"{temperature}"
        )

    domain_train_collapse = np.exp(
        -domain_train_distance / temperature
    ).astype(np.float32)

    domain_dev_collapse = np.exp(
        -domain_dev_distance / temperature
    ).astype(np.float32)

    train_cluster[train_index] = train_labels
    dev_cluster[dev_index] = dev_labels

    train_distance[train_index] = domain_train_distance
    dev_distance[dev_index] = domain_dev_distance

    train_collapse[train_index] = domain_train_collapse
    dev_collapse[dev_index] = domain_dev_collapse

    train_occupancy = np.bincount(
        train_labels,
        minlength=K_CLUSTERS,
    )

    dev_occupancy = np.bincount(
        dev_labels,
        minlength=K_CLUSTERS,
    )

    state_path = (
        MODEL_STATE_DIR
        / f"{domain}_standard_scaler_kmeans_K{K_CLUSTERS}.joblib"
    )

    joblib.dump(
        {
            "domain": domain,
            "seed": SEED,
            "n_clusters": K_CLUSTERS,
            "eligibility_floor": ELIGIBILITY_FLOOR,
            "scaler": scaler,
            "kmeans": kmeans,
            "training_distance_q25": float(q25),
            "training_distance_q75": float(q75),
            "temperature_iqr": temperature,
        },
        state_path,
        compress=3,
    )

    domain_metadata[domain] = {
        "training_candidates": int(len(train_index)),
        "development_candidates": int(len(dev_index)),
        "embedding_dimensions": int(X_train.shape[1]),
        "n_clusters": K_CLUSTERS,
        "kmeans_inertia": float(kmeans.inertia_),
        "kmeans_iterations": int(kmeans.n_iter_),
        "fit_elapsed_minutes": float(elapsed_minutes),
        "training_distance_q25": float(q25),
        "training_distance_q75": float(q75),
        "temperature_iqr": temperature,
        "occupied_training_clusters": int(np.sum(train_occupancy > 0)),
        "occupied_development_clusters": int(np.sum(dev_occupancy > 0)),
        "minimum_training_cluster_size": int(train_occupancy.min()),
        "median_training_cluster_size": float(
            np.median(train_occupancy)
        ),
        "maximum_training_cluster_size": int(train_occupancy.max()),
        "state_path": str(state_path),
    }

    for split_name, distances, signals, occupancy in [
        (
            "train",
            domain_train_distance,
            domain_train_collapse,
            train_occupancy,
        ),
        (
            "development",
            domain_dev_distance,
            domain_dev_collapse,
            dev_occupancy,
        ),
    ]:
        distribution_rows.append(
            {
                "domain": domain,
                "split": split_name,
                "n_candidates": int(len(distances)),
                "distance_min": float(np.min(distances)),
                "distance_q25": float(np.quantile(distances, 0.25)),
                "distance_median": float(np.median(distances)),
                "distance_q75": float(np.quantile(distances, 0.75)),
                "distance_q95": float(np.quantile(distances, 0.95)),
                "distance_max": float(np.max(distances)),
                "collapse_min": float(np.min(signals)),
                "collapse_q25": float(np.quantile(signals, 0.25)),
                "collapse_median": float(np.median(signals)),
                "collapse_q75": float(np.quantile(signals, 0.75)),
                "collapse_q95": float(np.quantile(signals, 0.95)),
                "collapse_max": float(np.max(signals)),
                "occupied_clusters": int(np.sum(occupancy > 0)),
                "minimum_cluster_size": int(occupancy.min()),
                "median_cluster_size": float(np.median(occupancy)),
                "maximum_cluster_size": int(occupancy.max()),
            }
        )

    print(f"KMeans elapsed minutes: {elapsed_minutes:.2f}")
    print(f"KMeans iterations: {kmeans.n_iter_}")
    print(f"Training distance IQR temperature: {temperature:.6f}")
    print(
        "Training cluster sizes — "
        f"min: {train_occupancy.min()}, "
        f"median: {np.median(train_occupancy):.1f}, "
        f"max: {train_occupancy.max()}"
    )
    print(
        "Development collapse percentiles: "
        f"q25={np.quantile(domain_dev_collapse, 0.25):.4f}, "
        f"median={np.median(domain_dev_collapse):.4f}, "
        f"q75={np.quantile(domain_dev_collapse, 0.75):.4f}, "
        f"q95={np.quantile(domain_dev_collapse, 0.95):.4f}"
    )

    # Release large domain-specific arrays before the next fit.
    del X_train
    del X_dev
    del X_train_scaled
    del X_dev_scaled

# ---------------------------------------------------------------------
# Attach scores and validate
# ---------------------------------------------------------------------

train_eligible["collapse_cluster"] = train_cluster
train_eligible["collapse_distance"] = train_distance
train_eligible["collapse"] = train_collapse

dev_eligible["collapse_cluster"] = dev_cluster
dev_eligible["collapse_distance"] = dev_distance
dev_eligible["collapse"] = dev_collapse

for split_name, table in [
    ("training", train_eligible),
    ("development", dev_eligible),
]:
    if (table["collapse_cluster"] < 0).any():
        raise ValueError(f"Unassigned clusters remain in {split_name}.")

    if not np.isfinite(
        table["collapse_distance"].to_numpy()
    ).all():
        raise ValueError(
            f"Non-finite collapse distances found in {split_name}."
        )

    if not np.isfinite(table["collapse"].to_numpy()).all():
        raise ValueError(
            f"Non-finite collapse values found in {split_name}."
        )

    if not table["collapse"].between(0.0, 1.0).all():
        raise ValueError(
            f"Collapse values outside [0, 1] found in {split_name}."
        )

train_eligible.to_csv(TRAIN_OUTPUT_PATH, index=False)
dev_eligible.to_csv(DEV_OUTPUT_PATH, index=False)

distribution_summary = pd.DataFrame(distribution_rows)
distribution_summary.to_csv(SUMMARY_PATH, index=False)

metadata = {
    "method": "domain_specific_training_only_standardized_kmeans",
    "random_seed": SEED,
    "n_clusters_per_domain": K_CLUSTERS,
    "eligibility_floor": ELIGIBILITY_FLOOR,
    "distance": "euclidean distance in training-standardized embedding space",
    "temperature_definition": (
        "training assigned-centroid distance Q75 minus Q25, "
        "separately by domain"
    ),
    "collapse_definition": "exp(-distance / training_temperature_iqr)",
    "development_information_used_for_fit": False,
    "training_output_path": str(TRAIN_OUTPUT_PATH),
    "development_output_path": str(DEV_OUTPUT_PATH),
    "distribution_summary_path": str(SUMMARY_PATH),
    "domains": domain_metadata,
}

with open(METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print()
print("=" * 78)
print("COLLAPSE FIT COMPLETE")
print("=" * 78)
print()
print("Distribution summary:")
print(
    distribution_summary[
        [
            "domain",
            "split",
            "n_candidates",
            "distance_median",
            "collapse_q25",
            "collapse_median",
            "collapse_q75",
            "occupied_clusters",
            "minimum_cluster_size",
            "maximum_cluster_size",
        ]
    ].to_string(index=False)
)

print()
print("Saved training candidates with collapse to:")
print(TRAIN_OUTPUT_PATH)

print()
print("Saved development candidates with collapse to:")
print(DEV_OUTPUT_PATH)

print()
print("Saved collapse metadata to:")
print(METADATA_PATH)

print()
print("Cell 22 completed successfully.")

In [ ]:
# CELL 22A — Candidate/embedding provenance and reproducibility audit

import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

ELIGIBILITY_FLOOR = 0.50

TRAIN_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_full_pairing_max16_candidate_pool.csv"
)

DEV_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_full_pairing_max16_candidate_pool.csv"
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_training_candidate_embeddings_p0.5.npy"
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_development_candidate_embeddings_p0.5.npy"
)

TRAIN_COLLAPSE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

DEV_COLLAPSE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_candidates_with_collapse.csv"
)

AUDIT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_candidate_embedding_provenance.json"
)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def dataframe_signature(table):
    """
    Order-sensitive signature of the complete candidate table.
    This lets us detect both row-content and row-order changes.
    """
    normalized = table.copy()

    # Stable text representation for missing values and floating points
    for column in normalized.columns:
        if pd.api.types.is_float_dtype(normalized[column]):
            normalized[column] = normalized[column].map(
                lambda value: (
                    "<NA>"
                    if pd.isna(value)
                    else format(float(value), ".17g")
                )
            )
        else:
            normalized[column] = (
                normalized[column]
                .astype("string")
                .fillna("<NA>")
            )

    hashed_rows = pd.util.hash_pandas_object(
        normalized,
        index=False,
    ).to_numpy(dtype=np.uint64)

    return hashlib.sha256(
        hashed_rows.tobytes()
    ).hexdigest()


# ---------------------------------------------------------------------
# Load authoritative current artifacts
# ---------------------------------------------------------------------

train_pool = pd.read_csv(TRAIN_POOL_PATH)
dev_pool = pd.read_csv(DEV_POOL_PATH)

train_current = (
    train_pool.loc[
        train_pool["p_term"] >= ELIGIBILITY_FLOOR
    ]
    .reset_index(drop=True)
    .copy()
)

dev_current = (
    dev_pool.loc[
        dev_pool["p_term"] >= ELIGIBILITY_FLOOR
    ]
    .reset_index(drop=True)
    .copy()
)

train_collapse = pd.read_csv(TRAIN_COLLAPSE_PATH)
dev_collapse = pd.read_csv(DEV_COLLAPSE_PATH)

train_embedding = np.load(
    TRAIN_EMBEDDING_PATH,
    mmap_mode="r",
)

dev_embedding = np.load(
    DEV_EMBEDDING_PATH,
    mmap_mode="r",
)

# candidate_id was added after the embeddings were constructed
train_collapse_comparable = train_collapse.drop(
    columns=[
        "candidate_id",
        "collapse_cluster",
        "collapse_distance",
        "collapse",
    ],
    errors="ignore",
)

dev_collapse_comparable = dev_collapse.drop(
    columns=[
        "candidate_id",
        "collapse_cluster",
        "collapse_distance",
        "collapse",
    ],
    errors="ignore",
)

train_columns_match = (
    list(train_current.columns)
    == list(train_collapse_comparable.columns)
)

dev_columns_match = (
    list(dev_current.columns)
    == list(dev_collapse_comparable.columns)
)

train_exact_match = (
    train_columns_match
    and len(train_current) == len(train_collapse_comparable)
    and dataframe_signature(train_current)
    == dataframe_signature(train_collapse_comparable)
)

dev_exact_match = (
    dev_columns_match
    and len(dev_current) == len(dev_collapse_comparable)
    and dataframe_signature(dev_current)
    == dataframe_signature(dev_collapse_comparable)
)

train_embedding_count_match = (
    len(train_current) == train_embedding.shape[0]
)

dev_embedding_count_match = (
    len(dev_current) == dev_embedding.shape[0]
)

# ---------------------------------------------------------------------
# Inspect values immediately around the eligibility boundary
# ---------------------------------------------------------------------

def boundary_audit(table):
    distance = np.abs(
        table["p_term"].astype(float) - ELIGIBILITY_FLOOR
    )

    nearest = (
        table.assign(distance_from_floor=distance)
        .nsmallest(10, "distance_from_floor")
    )

    return nearest[
        [
            column
            for column in [
                "domain",
                "num_article",
                "term",
                "start_char",
                "end_char",
                "p_term",
                "distance_from_floor",
            ]
            if column in nearest.columns
        ]
    ]


print("=" * 78)
print("CANDIDATE–EMBEDDING PROVENANCE AUDIT")
print("=" * 78)

print()
print("Current authoritative counts:")
print(f"Training eligible candidates: {len(train_current)}")
print(f"Training embedding rows: {train_embedding.shape[0]}")
print(f"Training collapse rows: {len(train_collapse)}")
print()
print(f"Development eligible candidates: {len(dev_current)}")
print(f"Development embedding rows: {dev_embedding.shape[0]}")
print(f"Development collapse rows: {len(dev_collapse)}")

print()
print("Alignment checks:")
print(
    "Training candidate count matches embeddings:",
    train_embedding_count_match,
)
print(
    "Development candidate count matches embeddings:",
    dev_embedding_count_match,
)
print(
    "Training source candidates exactly match collapse rows:",
    train_exact_match,
)
print(
    "Development source candidates exactly match collapse rows:",
    dev_exact_match,
)

print()
print("Training candidates nearest p_term = 0.50:")
print(boundary_audit(train_pool).to_string(index=False))

print()
print("Development candidates nearest p_term = 0.50:")
print(boundary_audit(dev_pool).to_string(index=False))

# ---------------------------------------------------------------------
# Persist authoritative fingerprints
# ---------------------------------------------------------------------

audit = {
    "eligibility_rule": "p_term >= 0.50",
    "training": {
        "candidate_pool_path": str(TRAIN_POOL_PATH),
        "candidate_pool_sha256": file_sha256(TRAIN_POOL_PATH),
        "eligible_candidate_count": int(len(train_current)),
        "eligible_candidate_ordered_signature": dataframe_signature(
            train_current
        ),
        "embedding_path": str(TRAIN_EMBEDDING_PATH),
        "embedding_file_sha256": file_sha256(TRAIN_EMBEDDING_PATH),
        "embedding_shape": [
            int(value) for value in train_embedding.shape
        ],
        "collapse_path": str(TRAIN_COLLAPSE_PATH),
        "collapse_file_sha256": file_sha256(TRAIN_COLLAPSE_PATH),
        "candidate_embedding_count_match": bool(
            train_embedding_count_match
        ),
        "candidate_collapse_exact_match": bool(train_exact_match),
    },
    "development": {
        "candidate_pool_path": str(DEV_POOL_PATH),
        "candidate_pool_sha256": file_sha256(DEV_POOL_PATH),
        "eligible_candidate_count": int(len(dev_current)),
        "eligible_candidate_ordered_signature": dataframe_signature(
            dev_current
        ),
        "embedding_path": str(DEV_EMBEDDING_PATH),
        "embedding_file_sha256": file_sha256(DEV_EMBEDDING_PATH),
        "embedding_shape": [
            int(value) for value in dev_embedding.shape
        ],
        "collapse_path": str(DEV_COLLAPSE_PATH),
        "collapse_file_sha256": file_sha256(DEV_COLLAPSE_PATH),
        "candidate_embedding_count_match": bool(
            dev_embedding_count_match
        ),
        "candidate_collapse_exact_match": bool(dev_exact_match),
    },
}

all_checks_pass = all(
    [
        train_embedding_count_match,
        dev_embedding_count_match,
        train_exact_match,
        dev_exact_match,
    ]
)

audit["all_alignment_checks_pass"] = bool(all_checks_pass)

with open(AUDIT_PATH, "w", encoding="utf-8") as file:
    json.dump(audit, file, indent=2)

print()
print("=" * 78)

if all_checks_pass:
    print("AUDIT PASSED")
    print(
        "The current candidate pools, embeddings, and collapse tables "
        "are aligned exactly."
    )
else:
    print("AUDIT FAILED")
    print(
        "Do not tune collapse parameters until the failed alignment "
        "checks are repaired."
    )

print("=" * 78)

print()
print("Saved provenance audit to:")
print(AUDIT_PATH)

print()
print("Cell 22A completed.")

In [ ]:
# @title
# CELL 23 — Diagnostic validity of the raw collapse signal

import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import spearmanr
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

DATA_DIR = BASE_DIR / "data/taskA/training"
EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

DEV_COLLAPSE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_candidates_with_collapse.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

PA_GOLD_PATH = (
    DATA_DIR
    / "csv/taskA_parkinson_terms_train.csv"
)

MH_GOLD_PATH = (
    DATA_DIR
    / "csv/taskA_mental_health_terms_train.csv"
)

DIAGNOSTIC_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_diagnostic.csv"
)

SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_diagnostic_summary.json"
)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_article_id(series):
    """
    Make IDs robust to CSV inference such as 7 versus 7.0.
    """
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def infer_surface_column(table):
    candidates = [
        "mention",
        "term",
        "candidate_text",
        "surface_form",
        "span_text",
        "text",
        "prediction",
        "predicted_term",
    ]


def safe_auc(labels, scores):
    if len(np.unique(labels)) < 2:
        return np.nan

    return float(roc_auc_score(labels, scores))


def safe_average_precision(labels, scores):
    if np.sum(labels) == 0:
        return np.nan

    return float(average_precision_score(labels, scores))


def percentile_record(values, prefix):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return {
            f"{prefix}_min": np.nan,
            f"{prefix}_q05": np.nan,
            f"{prefix}_q25": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_q75": np.nan,
            f"{prefix}_q95": np.nan,
            f"{prefix}_max": np.nan,
            f"{prefix}_mean": np.nan,
        }

    return {
        f"{prefix}_min": float(np.min(values)),
        f"{prefix}_q05": float(np.quantile(values, 0.05)),
        f"{prefix}_q25": float(np.quantile(values, 0.25)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_q75": float(np.quantile(values, 0.75)),
        f"{prefix}_q95": float(np.quantile(values, 0.95)),
        f"{prefix}_max": float(np.max(values)),
        f"{prefix}_mean": float(np.mean(values)),
    }


# ---------------------------------------------------------------------
# Load development candidates
# ---------------------------------------------------------------------

development = pd.read_csv(DEV_COLLAPSE_PATH)
surface_column = "mention"

if surface_column not in development.columns:
    raise ValueError(
        f"Required surface column '{surface_column}' is absent. "
        f"Available columns: {list(development.columns)}"
    )

required_candidate_columns = {
    "domain",
    "num_article",
    "p_term",
    "collapse",
}

missing = required_candidate_columns.difference(development.columns)

if missing:
    raise ValueError(
        f"Development candidate file is missing: {sorted(missing)}"
    )

development["domain"] = (
    development["domain"]
    .astype(str)
    .str.strip()
)

development["num_article"] = normalize_article_id(
    development["num_article"]
)

development[surface_column] = (
    development[surface_column]
    .fillna("")
    .astype(str)
)

# ---------------------------------------------------------------------
# Load official gold inventories
# ---------------------------------------------------------------------

parkinson_gold = pd.read_csv(PA_GOLD_PATH)
parkinson_gold["domain"] = "parkinson"

mental_health_gold = pd.read_csv(MH_GOLD_PATH)
mental_health_gold["domain"] = "mental_health"

gold = pd.concat(
    [
        parkinson_gold,
        mental_health_gold,
    ],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

# Official Task A evaluation unit used in our experiment:
# unique document–surface-form pairs.
gold_pairs = (
    gold[
        [
            "domain",
            "num_article",
            "term",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Confirm that only development documents are used.
split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)
split_table["domain"] = (
    split_table["domain"]
    .astype(str)
    .str.strip()
)

development_documents = (
    split_table.loc[
        split_table["split"] == "development",
        [
            "domain",
            "num_article",
        ],
    ]
    .drop_duplicates()
)

development_gold = gold_pairs.merge(
    development_documents,
    on=[
        "domain",
        "num_article",
    ],
    how="inner",
)

# ---------------------------------------------------------------------
# Candidate-level gold compatibility
# ---------------------------------------------------------------------

gold_lookup = set(
    zip(
        development_gold["domain"],
        development_gold["num_article"],
        development_gold["term"],
    )
)

development["gold_compatible"] = [
    int((domain, article, surface) in gold_lookup)
    for domain, article, surface in zip(
        development["domain"],
        development["num_article"],
        development[surface_column],
    )
]

development["normalized_surface"] = (
    development[surface_column]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# This diagnostic uses exact case-sensitive compatibility because that
# is the official matching convention used in the current evaluation.
# The normalized form is retained only for later error analysis.

# ---------------------------------------------------------------------
# Summaries
# ---------------------------------------------------------------------

summary_rows = []

for domain in [
    "parkinson",
    "mental_health",
    "ALL",
]:
    if domain == "ALL":
        subset = development.copy()
    else:
        subset = development.loc[
            development["domain"] == domain
        ].copy()

    labels = subset["gold_compatible"].to_numpy(dtype=int)
    neural = subset["p_term"].to_numpy(dtype=float)
    collapse = subset["collapse"].to_numpy(dtype=float)

    positive = subset.loc[
        subset["gold_compatible"] == 1
    ]

    negative = subset.loc[
        subset["gold_compatible"] == 0
    ]

    correlation, correlation_p = spearmanr(
        neural,
        collapse,
    )

    row = {
        "domain": domain,
        "n_candidates": int(len(subset)),
        "n_gold_compatible_candidates": int(labels.sum()),
        "gold_compatible_ratio": float(labels.mean()),
        "n_non_gold_candidates": int((labels == 0).sum()),
        "neural_roc_auc": safe_auc(labels, neural),
        "neural_average_precision": safe_average_precision(
            labels,
            neural,
        ),
        "collapse_roc_auc": safe_auc(labels, collapse),
        "collapse_average_precision": safe_average_precision(
            labels,
            collapse,
        ),
        "neural_collapse_spearman": float(correlation),
        "neural_collapse_spearman_p": float(correlation_p),
    }

    row.update(
        percentile_record(
            positive["p_term"],
            "gold_neural",
        )
    )
    row.update(
        percentile_record(
            negative["p_term"],
            "nongold_neural",
        )
    )
    row.update(
        percentile_record(
            positive["collapse"],
            "gold_collapse",
        )
    )
    row.update(
        percentile_record(
            negative["collapse"],
            "nongold_collapse",
        )
    )

    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# Candidate recovery at selected collapse percentiles
# ---------------------------------------------------------------------

threshold_rows = []

for domain in ["parkinson", "mental_health"]:
    subset = development.loc[
        development["domain"] == domain
    ].copy()

    for quantile in [
        0.50,
        0.60,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
    ]:
        threshold = float(
            subset["collapse"].quantile(quantile)
        )

        selected = subset.loc[
            subset["collapse"] >= threshold
        ]

        threshold_rows.append(
            {
                "domain": domain,
                "collapse_quantile": quantile,
                "collapse_threshold": threshold,
                "selected_candidates": int(len(selected)),
                "gold_compatible_candidates": int(
                    selected["gold_compatible"].sum()
                ),
                "candidate_precision": float(
                    selected["gold_compatible"].mean()
                ),
                "share_of_all_gold_compatible_candidates": float(
                    selected["gold_compatible"].sum()
                    / max(1, subset["gold_compatible"].sum())
                ),
            }
        )

threshold_table = pd.DataFrame(threshold_rows)

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

development.to_csv(
    DIAGNOSTIC_PATH,
    index=False,
)

serializable_summary = {
    "surface_column": surface_column,
    "matching_definition": (
        "exact case-sensitive unique document–surface-form compatibility"
    ),
    "diagnostic_scope": (
        "development candidates with p_term >= 0.50"
    ),
    "candidate_level_warning": (
        "This is a diagnostic of candidate enrichment, not the final "
        "document-level Task A evaluation."
    ),
    "summary": summary.replace(
        {np.nan: None}
    ).to_dict(orient="records"),
    "collapse_quantile_thresholds": threshold_table.replace(
        {np.nan: None}
    ).to_dict(orient="records"),
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(
        serializable_summary,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("RAW COLLAPSE-SIGNAL DIAGNOSTIC")
print("=" * 78)
print(f"Candidate surface column: {surface_column}")
print(
    "Matching: exact case-sensitive document–surface-form compatibility"
)

print()
print("Discrimination and independence diagnostics:")

display_columns = [
    "domain",
    "n_candidates",
    "n_gold_compatible_candidates",
    "gold_compatible_ratio",
    "neural_roc_auc",
    "neural_average_precision",
    "collapse_roc_auc",
    "collapse_average_precision",
    "neural_collapse_spearman",
]

print(
    summary[display_columns]
    .round(4)
    .to_string(index=False)
)

print()
print("Gold-compatible versus non-gold collapse distributions:")

distribution_columns = [
    "domain",
    "gold_collapse_q25",
    "gold_collapse_median",
    "gold_collapse_q75",
    "gold_collapse_q95",
    "nongold_collapse_q25",
    "nongold_collapse_median",
    "nongold_collapse_q75",
    "nongold_collapse_q95",
]

print(
    summary[distribution_columns]
    .round(4)
    .to_string(index=False)
)

print()
print("Collapse-quantile candidate enrichment:")

print(
    threshold_table.round(4).to_string(index=False)
)

print()
print("Saved annotated diagnostic candidates to:")
print(DIAGNOSTIC_PATH)

print()
print("Saved diagnostic summary to:")
print(SUMMARY_PATH)

print()
print("Cell 23 completed successfully.")

In [ ]:
# CELL 24 — Audit cleaner training-only populations for collapse fitting

import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths and fixed neural thresholds
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

TRAIN_COLLAPSE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_training_collapse_anchor_audit.csv"
)

NESTED_THRESHOLD = 0.97
NONOVERLAP_THRESHOLD = 0.81

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def intervals_overlap(start_a, end_a, start_b, end_b):
    """
    Character intervals are treated as half-open: [start, end).
    """
    return start_a < end_b and start_b < end_a


def greedy_nonoverlap(group):
    """
    Retain the highest-confidence occurrence first.

    Tie-breaking is deterministic:
      1. Higher p_term
      2. Higher minimum boundary probability
      3. Shorter span
      4. Earlier start
      5. Earlier end
      6. Lower candidate_id
    """
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_char_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        by=[
            "p_term",
            "_boundary_min",
            "_char_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted_indices = []
    accepted_intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        conflicts = any(
            intervals_overlap(
                start,
                end,
                accepted_start,
                accepted_end,
            )
            for accepted_start, accepted_end
            in accepted_intervals
        )

        if not conflicts:
            accepted_indices.append(row.Index)
            accepted_intervals.append((start, end))

    return group.loc[accepted_indices].copy()


# ---------------------------------------------------------------------
# Load the current authoritative training candidate table
# ---------------------------------------------------------------------

training = pd.read_csv(TRAIN_COLLAPSE_PATH)

required_columns = {
    "candidate_id",
    "domain",
    "num_article",
    "window_index",
    "start_char",
    "end_char",
    "mention",
    "p_start",
    "p_end",
    "p_term",
}

missing = required_columns.difference(training.columns)

if missing:
    raise ValueError(
        f"Training candidate table is missing: {sorted(missing)}"
    )

training["num_article"] = normalize_article_id(
    training["num_article"]
)

training["mention"] = (
    training["mention"]
    .fillna("")
    .astype(str)
)

training["character_length"] = (
    training["end_char"] - training["start_char"]
)

training["word_count"] = (
    training["mention"]
    .str.strip()
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

# ---------------------------------------------------------------------
# Population 1: high-confidence nested candidates
# ---------------------------------------------------------------------

nested_anchors = (
    training.loc[
        training["p_term"] >= NESTED_THRESHOLD
    ]
    .copy()
)

nested_anchors["anchor_policy"] = "NESTED_HIGH_CONFIDENCE"

# ---------------------------------------------------------------------
# Population 2: high-confidence greedy non-overlap candidates
# ---------------------------------------------------------------------

nonoverlap_source = (
    training.loc[
        training["p_term"] >= NONOVERLAP_THRESHOLD
    ]
    .copy()
)

nonoverlap_parts = []

document_columns = [
    "domain",
    "num_article",
]

for _, document_group in nonoverlap_source.groupby(
    document_columns,
    sort=False,
):
    nonoverlap_parts.append(
        greedy_nonoverlap(document_group)
    )

nonoverlap_anchors = pd.concat(
    nonoverlap_parts,
    ignore_index=False,
).copy()

nonoverlap_anchors["anchor_policy"] = (
    "NONOVERLAP_HIGH_CONFIDENCE"
)

# ---------------------------------------------------------------------
# Summarize anchor populations
# ---------------------------------------------------------------------

summary_rows = []

for policy_name, table, threshold in [
    (
        "NESTED_HIGH_CONFIDENCE",
        nested_anchors,
        NESTED_THRESHOLD,
    ),
    (
        "NONOVERLAP_HIGH_CONFIDENCE",
        nonoverlap_anchors,
        NONOVERLAP_THRESHOLD,
    ),
]:
    for domain in sorted(training["domain"].unique()):
        subset = table.loc[
            table["domain"] == domain
        ]

        total_domain_candidates = int(
            (training["domain"] == domain).sum()
        )

        unique_documents = int(
            training.loc[
                training["domain"] == domain,
                "num_article",
            ].nunique()
        )

        summary_rows.append(
            {
                "anchor_policy": policy_name,
                "domain": domain,
                "neural_threshold": threshold,
                "anchor_candidates": int(len(subset)),
                "total_eligible_candidates": total_domain_candidates,
                "retained_candidate_ratio": (
                    len(subset)
                    / max(1, total_domain_candidates)
                ),
                "documents": unique_documents,
                "documents_with_anchor": int(
                    subset["num_article"].nunique()
                ),
                "mean_anchors_per_document": float(
                    len(subset) / max(1, unique_documents)
                ),
                "unique_mentions": int(
                    subset["mention"].nunique()
                ),
                "median_p_term": float(
                    subset["p_term"].median()
                ),
                "median_word_count": float(
                    subset["word_count"].median()
                ),
                "word_count_q95": float(
                    subset["word_count"].quantile(0.95)
                ),
                "median_character_length": float(
                    subset["character_length"].median()
                ),
            }
        )

summary = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# Cluster feasibility checks
# ---------------------------------------------------------------------

summary["sufficient_for_K100"] = (
    summary["anchor_candidates"] >= 100
)

summary["anchors_per_cluster_K100"] = (
    summary["anchor_candidates"] / 100
)

if not summary["sufficient_for_K100"].all():
    raise ValueError(
        "At least one anchor population has fewer than 100 "
        "candidates in a domain."
    )

# ---------------------------------------------------------------------
# Save anchor IDs and summary
# ---------------------------------------------------------------------

anchor_membership = pd.concat(
    [
        nested_anchors[
            [
                "candidate_id",
                "domain",
                "num_article",
                "anchor_policy",
                "p_term",
            ]
        ],
        nonoverlap_anchors[
            [
                "candidate_id",
                "domain",
                "num_article",
                "anchor_policy",
                "p_term",
            ]
        ],
    ],
    ignore_index=True,
)

anchor_membership_path = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_training_collapse_anchor_membership.csv"
)

summary.to_csv(
    OUTPUT_PATH,
    index=False,
)

anchor_membership.to_csv(
    anchor_membership_path,
    index=False,
)

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("TRAINING-ONLY COLLAPSE ANCHOR AUDIT")
print("=" * 78)

print()
print(
    summary.round(4).to_string(index=False)
)

print()
print("Candidate counts before anchoring:")
print(
    training.groupby("domain")
    .size()
    .to_string()
)

print()
print("Nested-anchor confidence percentiles:")
print(
    nested_anchors.groupby("domain")["p_term"]
    .quantile([0.00, 0.25, 0.50, 0.75, 1.00])
    .round(6)
    .to_string()
)

print()
print("Non-overlap-anchor confidence percentiles:")
print(
    nonoverlap_anchors.groupby("domain")["p_term"]
    .quantile([0.00, 0.25, 0.50, 0.75, 1.00])
    .round(6)
    .to_string()
)

print()
print("Saved anchor audit to:")
print(OUTPUT_PATH)

print()
print("Saved anchor membership to:")
print(anchor_membership_path)

print()
print("Cell 24 completed successfully.")

In [ ]:
# CELL 25 — Fit and compare two training-only anchor-based collapse states

import json
import time
import joblib
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

STATE_DIR = (
    EXPERIMENT_DIR
    / "models/collapse_anchor_comparison_seed42"
)

STATE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

DEV_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_candidates_with_collapse.csv"
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_training_candidate_embeddings_p0.5.npy"
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_development_candidate_embeddings_p0.5.npy"
)

ANCHOR_MEMBERSHIP_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_training_collapse_anchor_membership.csv"
)

DEV_DIAGNOSTIC_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_diagnostic.csv"
)

DEV_OUTPUT_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_anchor_collapse_comparison.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_anchor_collapse_comparison.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_anchor_collapse_comparison_metadata.json"
)

SEED = 42
K_CLUSTERS = 100

POLICIES = [
    "NESTED_HIGH_CONFIDENCE",
    "NONOVERLAP_HIGH_CONFIDENCE",
]

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def safe_auc(labels, scores):
    if len(np.unique(labels)) < 2:
        return np.nan

    return float(roc_auc_score(labels, scores))


def safe_ap(labels, scores):
    if labels.sum() == 0:
        return np.nan

    return float(average_precision_score(labels, scores))


# ---------------------------------------------------------------------
# Load and verify frozen current artifacts
# ---------------------------------------------------------------------

training = pd.read_csv(TRAIN_CANDIDATE_PATH)
development = pd.read_csv(DEV_CANDIDATE_PATH)
membership = pd.read_csv(ANCHOR_MEMBERSHIP_PATH)
diagnostic = pd.read_csv(DEV_DIAGNOSTIC_PATH)

train_embeddings = np.load(
    TRAIN_EMBEDDING_PATH,
    mmap_mode="r",
)

dev_embeddings = np.load(
    DEV_EMBEDDING_PATH,
    mmap_mode="r",
)

if len(training) != train_embeddings.shape[0]:
    raise ValueError(
        "Training candidates and embeddings are not aligned: "
        f"{len(training)} versus {train_embeddings.shape[0]}."
    )

if len(development) != dev_embeddings.shape[0]:
    raise ValueError(
        "Development candidates and embeddings are not aligned: "
        f"{len(development)} versus {dev_embeddings.shape[0]}."
    )

for name, table in [
    ("training", training),
    ("development", development),
]:
    expected_ids = np.arange(len(table), dtype=np.int64)
    actual_ids = table["candidate_id"].to_numpy(dtype=np.int64)

    if not np.array_equal(expected_ids, actual_ids):
        raise ValueError(
            f"{name} candidate_id is not identical to row order."
        )

diagnostic_labels = (
    diagnostic[
        [
            "candidate_id",
            "gold_compatible",
        ]
    ]
    .drop_duplicates(subset=["candidate_id"])
)

development = development.merge(
    diagnostic_labels,
    on="candidate_id",
    how="left",
    validate="one_to_one",
)

if development["gold_compatible"].isna().any():
    raise ValueError(
        "Some current development candidates lack diagnostic labels. "
        "Rerun Cell 23 against the current candidate artifacts."
    )

development["gold_compatible"] = (
    development["gold_compatible"].astype(int)
)

print("=" * 78)
print("ANCHOR-BASED COLLAPSE COMPARISON")
print("=" * 78)
print(f"Training candidates: {len(training)}")
print(f"Development candidates: {len(development)}")
print(f"Embedding dimensions: {train_embeddings.shape[1]}")
print()

# Preserve neural discrimination as the reference
summary_rows = []

for domain in sorted(development["domain"].unique()):
    domain_dev = development["domain"].eq(domain)
    labels = development.loc[
        domain_dev,
        "gold_compatible",
    ].to_numpy()

    neural = development.loc[
        domain_dev,
        "p_term",
    ].to_numpy()

    summary_rows.append(
        {
            "policy": "NEURAL_CONFIDENCE",
            "domain": domain,
            "training_anchors": np.nan,
            "development_candidates": int(domain_dev.sum()),
            "temperature": np.nan,
            "roc_auc": safe_auc(labels, neural),
            "average_precision": safe_ap(labels, neural),
            "gold_median": float(
                np.median(neural[labels == 1])
            ),
            "nongold_median": float(
                np.median(neural[labels == 0])
            ),
            "gold_minus_nongold_median": float(
                np.median(neural[labels == 1])
                - np.median(neural[labels == 0])
            ),
        }
    )

# ---------------------------------------------------------------------
# Fit each candidate anchor policy separately by domain
# ---------------------------------------------------------------------

state_metadata = {}

for policy in POLICIES:
    print("-" * 78)
    print(f"ANCHOR POLICY: {policy}")
    print("-" * 78)

    policy_membership = membership.loc[
        membership["anchor_policy"] == policy
    ].copy()

    output_column = (
        "collapse_nested_anchor"
        if policy == "NESTED_HIGH_CONFIDENCE"
        else "collapse_nonoverlap_anchor"
    )

    distance_column = (
        "distance_nested_anchor"
        if policy == "NESTED_HIGH_CONFIDENCE"
        else "distance_nonoverlap_anchor"
    )

    development[output_column] = np.nan
    development[distance_column] = np.nan

    state_metadata[policy] = {}

    for domain in sorted(training["domain"].unique()):
        anchor_ids = (
            policy_membership.loc[
                policy_membership["domain"] == domain,
                "candidate_id",
            ]
            .astype(int)
            .drop_duplicates()
            .to_numpy()
        )

        # Ensure membership IDs point to the correct domain
        if not training.iloc[anchor_ids]["domain"].eq(domain).all():
            raise ValueError(
                f"Anchor candidate IDs are misaligned for {policy}, "
                f"{domain}."
            )

        dev_ids = np.flatnonzero(
            development["domain"].to_numpy() == domain
        )

        X_anchor = np.asarray(
            train_embeddings[anchor_ids],
            dtype=np.float32,
        )

        X_dev = np.asarray(
            dev_embeddings[dev_ids],
            dtype=np.float32,
        )

        print()
        print(f"Domain: {domain}")
        print(f"Training anchors: {len(anchor_ids)}")
        print(f"Development candidates: {len(dev_ids)}")

        scaler = StandardScaler()
        X_anchor_scaled = scaler.fit_transform(
            X_anchor
        ).astype(np.float32, copy=False)

        X_dev_scaled = scaler.transform(
            X_dev
        ).astype(np.float32, copy=False)

        kmeans = KMeans(
            n_clusters=K_CLUSTERS,
            init="k-means++",
            n_init=10,
            max_iter=300,
            tol=1e-4,
            random_state=SEED,
            algorithm="lloyd",
        )

        start_time = time.time()
        anchor_labels = kmeans.fit_predict(
            X_anchor_scaled
        )
        fit_minutes = (time.time() - start_time) / 60

        dev_labels = kmeans.predict(
            X_dev_scaled
        )

        centers = kmeans.cluster_centers_.astype(
            np.float32,
            copy=False,
        )

        anchor_distances = np.linalg.norm(
            X_anchor_scaled - centers[anchor_labels],
            axis=1,
        )

        dev_distances = np.linalg.norm(
            X_dev_scaled - centers[dev_labels],
            axis=1,
        )

        q25, q75 = np.quantile(
            anchor_distances,
            [0.25, 0.75],
        )

        temperature = float(q75 - q25)

        if not np.isfinite(temperature) or temperature <= 0:
            raise ValueError(
                f"Invalid temperature for {policy}, {domain}: "
                f"{temperature}"
            )

        dev_collapse = np.exp(
            -dev_distances / temperature
        )

        development.loc[
            development["domain"] == domain,
            distance_column,
        ] = dev_distances

        development.loc[
            development["domain"] == domain,
            output_column,
        ] = dev_collapse

        labels = development.loc[
            development["domain"] == domain,
            "gold_compatible",
        ].to_numpy(dtype=int)

        gold_scores = dev_collapse[labels == 1]
        nongold_scores = dev_collapse[labels == 0]

        auc = safe_auc(labels, dev_collapse)
        ap = safe_ap(labels, dev_collapse)

        summary_rows.append(
            {
                "policy": policy,
                "domain": domain,
                "training_anchors": int(len(anchor_ids)),
                "development_candidates": int(len(dev_ids)),
                "temperature": temperature,
                "roc_auc": auc,
                "average_precision": ap,
                "gold_median": float(
                    np.median(gold_scores)
                ),
                "nongold_median": float(
                    np.median(nongold_scores)
                ),
                "gold_minus_nongold_median": float(
                    np.median(gold_scores)
                    - np.median(nongold_scores)
                ),
            }
        )

        occupancy = np.bincount(
            anchor_labels,
            minlength=K_CLUSTERS,
        )

        state_path = (
            STATE_DIR
            / f"{policy.lower()}_{domain}_K100.joblib"
        )

        joblib.dump(
            {
                "policy": policy,
                "domain": domain,
                "seed": SEED,
                "n_clusters": K_CLUSTERS,
                "scaler": scaler,
                "kmeans": kmeans,
                "temperature": temperature,
                "training_distance_q25": float(q25),
                "training_distance_q75": float(q75),
            },
            state_path,
            compress=3,
        )

        state_metadata[policy][domain] = {
            "training_anchors": int(len(anchor_ids)),
            "fit_minutes": float(fit_minutes),
            "kmeans_iterations": int(kmeans.n_iter_),
            "temperature": temperature,
            "minimum_cluster_size": int(occupancy.min()),
            "median_cluster_size": float(
                np.median(occupancy)
            ),
            "maximum_cluster_size": int(occupancy.max()),
            "state_path": str(state_path),
        }

        print(f"Fit minutes: {fit_minutes:.2f}")
        print(f"Iterations: {kmeans.n_iter_}")
        print(f"Temperature: {temperature:.6f}")
        print(f"Collapse ROC-AUC: {auc:.4f}")
        print(f"Collapse average precision: {ap:.4f}")
        print(
            "Median collapse — "
            f"gold: {np.median(gold_scores):.4f}; "
            f"non-gold: {np.median(nongold_scores):.4f}"
        )

        del X_anchor
        del X_dev
        del X_anchor_scaled
        del X_dev_scaled

# ---------------------------------------------------------------------
# Combined-domain summaries
# ---------------------------------------------------------------------

labels_all = development[
    "gold_compatible"
].to_numpy(dtype=int)

for policy, score_column in [
    (
        "NESTED_HIGH_CONFIDENCE",
        "collapse_nested_anchor",
    ),
    (
        "NONOVERLAP_HIGH_CONFIDENCE",
        "collapse_nonoverlap_anchor",
    ),
]:
    scores = development[
        score_column
    ].to_numpy(dtype=float)

    summary_rows.append(
        {
            "policy": policy,
            "domain": "ALL",
            "training_anchors": int(
                membership.loc[
                    membership["anchor_policy"] == policy,
                    "candidate_id",
                ].nunique()
            ),
            "development_candidates": int(len(development)),
            "temperature": np.nan,
            "roc_auc": safe_auc(labels_all, scores),
            "average_precision": safe_ap(
                labels_all,
                scores,
            ),
            "gold_median": float(
                np.median(scores[labels_all == 1])
            ),
            "nongold_median": float(
                np.median(scores[labels_all == 0])
            ),
            "gold_minus_nongold_median": float(
                np.median(scores[labels_all == 1])
                - np.median(scores[labels_all == 0])
            ),
        }
    )

summary = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# Save frozen comparison artifacts
# ---------------------------------------------------------------------

development.to_csv(
    DEV_OUTPUT_PATH,
    index=False,
)

summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

metadata = {
    "purpose": (
        "Development-only comparison of training-only collapse "
        "anchor populations"
    ),
    "test_data_accessed": False,
    "n_clusters": K_CLUSTERS,
    "seed": SEED,
    "training_candidate_count": int(len(training)),
    "development_candidate_count": int(len(development)),
    "artifact_hashes": {
        "training_candidates": sha256(
            TRAIN_CANDIDATE_PATH
        ),
        "development_candidates": sha256(
            DEV_CANDIDATE_PATH
        ),
        "training_embeddings": sha256(
            TRAIN_EMBEDDING_PATH
        ),
        "development_embeddings": sha256(
            DEV_EMBEDDING_PATH
        ),
        "anchor_membership": sha256(
            ANCHOR_MEMBERSHIP_PATH
        ),
        "development_diagnostic": sha256(
            DEV_DIAGNOSTIC_PATH
        ),
    },
    "states": state_metadata,
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2)

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("ANCHOR COMPARISON SUMMARY")
print("=" * 78)

print(
    summary[
        [
            "policy",
            "domain",
            "training_anchors",
            "development_candidates",
            "roc_auc",
            "average_precision",
            "gold_median",
            "nongold_median",
            "gold_minus_nongold_median",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("Saved scored development candidates to:")
print(DEV_OUTPUT_PATH)

print()
print("Saved comparison summary to:")
print(SUMMARY_OUTPUT_PATH)

print()
print("Saved frozen artifact metadata to:")
print(METADATA_OUTPUT_PATH)

print()
print("Cell 25 completed successfully.")

In [ ]:
# CELL 26 — Development tuning of the two matched collapse branches

import json
import itertools
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Configuration and paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

DEV_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_anchor_collapse_comparison.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

TUNING_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_matched_collapse_tuning.csv"
)

SELECTION_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_matched_collapse_selection.json"
)

NESTED_P_KEEP = 0.97
NONOVERLAP_P_KEEP = 0.81

ALPHA_GRID = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    1.00,  # neural-only rescue control
]

P_MIN_GRID = [
    0.50,
    0.60,
    0.65,
    0.70,
    0.75,
]

TAU_GRID = np.round(
    np.arange(0.20, 0.951, 0.05),
    2,
).tolist()

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(
        str(value).casefold().split()
    )


def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_character_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_character_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted_indices = []
    accepted_intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        conflict = any(
            intervals_overlap(
                start,
                end,
                accepted_start,
                accepted_end,
            )
            for accepted_start, accepted_end
            in accepted_intervals
        )

        if not conflict:
            accepted_indices.append(row.Index)
            accepted_intervals.append((start, end))

    return group.loc[accepted_indices].copy()


def prf(predicted, gold):
    true_positive = len(predicted & gold)
    false_positive = len(predicted - gold)
    false_negative = len(gold - predicted)

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive
        else 0.0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": true_positive,
        "fp": false_positive,
        "fn": false_negative,
    }


def evaluate_predictions(predictions, gold_by_domain):
    result = {}

    for domain in ["parkinson", "mental_health"]:
        domain_predictions = predictions.loc[
            predictions["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                domain_predictions["num_article"],
                domain_predictions["mention"],
            )
        )

        predicted_types = {
            normalize_type(mention)
            for mention in domain_predictions["mention"]
            if str(mention).strip()
        }

        pair_metrics = prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_metrics = prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        result[domain] = {
            "micro": pair_metrics,
            "type": type_metrics,
        }

    result["avg_micro_f1"] = np.mean(
        [
            result["parkinson"]["micro"]["f1"],
            result["mental_health"]["micro"]["f1"],
        ]
    )

    result["avg_type_f1"] = np.mean(
        [
            result["parkinson"]["type"]["f1"],
            result["mental_health"]["type"]["f1"],
        ]
    )

    result["selection_score"] = np.mean(
        [
            result["avg_micro_f1"],
            result["avg_type_f1"],
        ]
    )

    return result


def flatten_result(
    branch,
    alpha,
    p_min,
    tau,
    p_keep,
    predictions,
    metrics,
):
    row = {
        "branch": branch,
        "alpha": alpha,
        "p_min": p_min,
        "tau": tau,
        "p_keep": p_keep,
        "n_occurrence_predictions": int(len(predictions)),
        "avg_micro_f1": metrics["avg_micro_f1"],
        "avg_type_f1": metrics["avg_type_f1"],
        "selection_score": metrics["selection_score"],
    }

    for domain, prefix in [
        ("parkinson", "PA"),
        ("mental_health", "MH"),
    ]:
        for level in ["micro", "type"]:
            for metric in [
                "precision",
                "recall",
                "f1",
                "tp",
                "fp",
                "fn",
            ]:
                row[
                    f"{prefix}_{level}_{metric}"
                ] = metrics[domain][level][metric]

    return row


# ---------------------------------------------------------------------
# Load development candidates
# ---------------------------------------------------------------------

development = pd.read_csv(DEV_PATH)

required_columns = {
    "candidate_id",
    "domain",
    "num_article",
    "mention",
    "start_char",
    "end_char",
    "p_start",
    "p_end",
    "p_term",
    "collapse_nested_anchor",
    "collapse_nonoverlap_anchor",
}

missing = required_columns.difference(development.columns)

if missing:
    raise ValueError(
        f"Development table is missing: {sorted(missing)}"
    )

development["num_article"] = normalize_article_id(
    development["num_article"]
)

development["mention"] = (
    development["mention"]
    .fillna("")
    .astype(str)
)

# ---------------------------------------------------------------------
# Load official development gold
# ---------------------------------------------------------------------

PA_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

MH_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(PA_GOLD_PATH)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(MH_GOLD_PATH)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

development_documents = (
    split_table.loc[
        split_table["split"] == "development",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

development_gold = (
    gold.merge(
        development_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [
        ["domain", "num_article", "term"]
    ]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = development_gold.loc[
        development_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(term)
            for term in subset["term"]
            if str(term).strip()
        },
    }

print("=" * 78)
print("MATCHED COLLAPSE TUNING")
print("=" * 78)
print(f"Development candidates: {len(development)}")
print(
    "Gold pairs — Parkinson:",
    len(gold_by_domain["parkinson"]["pairs"]),
)
print(
    "Gold pairs — Mental health:",
    len(gold_by_domain["mental_health"]["pairs"]),
)

# ---------------------------------------------------------------------
# Prepare the broad non-overlap candidate population once
# ---------------------------------------------------------------------

nonoverlap_parts = []

for _, document_group in development.groupby(
    ["domain", "num_article"],
    sort=False,
):
    nonoverlap_parts.append(
        greedy_nonoverlap(document_group)
    )

development_nonoverlap = pd.concat(
    nonoverlap_parts,
    ignore_index=False,
).copy()

print(
    "Broad greedy non-overlap candidates:",
    len(development_nonoverlap),
)

# ---------------------------------------------------------------------
# Baseline evaluations
# ---------------------------------------------------------------------

rows = []

nested_baseline = development.loc[
    development["p_term"] >= NESTED_P_KEEP
].copy()

nested_baseline_metrics = evaluate_predictions(
    nested_baseline,
    gold_by_domain,
)

rows.append(
    flatten_result(
        branch="NESTED_NEURAL_BASELINE",
        alpha=np.nan,
        p_min=np.nan,
        tau=np.nan,
        p_keep=NESTED_P_KEEP,
        predictions=nested_baseline,
        metrics=nested_baseline_metrics,
    )
)

nonoverlap_baseline = development_nonoverlap.loc[
    development_nonoverlap["p_term"]
    >= NONOVERLAP_P_KEEP
].copy()

nonoverlap_baseline_metrics = evaluate_predictions(
    nonoverlap_baseline,
    gold_by_domain,
)

rows.append(
    flatten_result(
        branch="NONOVERLAP_NEURAL_BASELINE",
        alpha=np.nan,
        p_min=np.nan,
        tau=np.nan,
        p_keep=NONOVERLAP_P_KEEP,
        predictions=nonoverlap_baseline,
        metrics=nonoverlap_baseline_metrics,
    )
)

# ---------------------------------------------------------------------
# Tune matched collapse branches
# ---------------------------------------------------------------------

parameter_grid = list(
    itertools.product(
        ALPHA_GRID,
        P_MIN_GRID,
        TAU_GRID,
    )
)

branch_definitions = [
    {
        "name": "NESTED_PLUS_COLLAPSE",
        "source": development,
        "collapse_column": "collapse_nested_anchor",
        "p_keep": NESTED_P_KEEP,
    },
    {
        "name": "NONOVERLAP_PLUS_COLLAPSE",
        "source": development_nonoverlap,
        "collapse_column": "collapse_nonoverlap_anchor",
        "p_keep": NONOVERLAP_P_KEEP,
    },
]

for branch in branch_definitions:
    source = branch["source"]
    collapse_column = branch["collapse_column"]
    p_keep = branch["p_keep"]

    neural = source["p_term"].to_numpy(dtype=float)
    collapse = source[
        collapse_column
    ].to_numpy(dtype=float)

    for alpha, p_min, tau in tqdm(
        parameter_grid,
        desc=f"Tuning {branch['name']}",
    ):
        combined = (
            alpha * neural
            + (1.0 - alpha) * collapse
        )

        keep = (
            (neural >= p_keep)
            |
            (
                (neural >= p_min)
                & (combined >= tau)
            )
        )

        predictions = source.loc[keep].copy()

        metrics = evaluate_predictions(
            predictions,
            gold_by_domain,
        )

        rows.append(
            flatten_result(
                branch=branch["name"],
                alpha=alpha,
                p_min=p_min,
                tau=tau,
                p_keep=p_keep,
                predictions=predictions,
                metrics=metrics,
            )
        )

results = pd.DataFrame(rows)

# ---------------------------------------------------------------------
# Select configurations with deterministic conservative tie-breaking
# ---------------------------------------------------------------------

selected = {}

for branch_name in [
    "NESTED_PLUS_COLLAPSE",
    "NONOVERLAP_PLUS_COLLAPSE",
]:
    subset = results.loc[
        results["branch"] == branch_name
    ].copy()

    subset = subset.sort_values(
        by=[
            "selection_score",
            "avg_micro_f1",
            "avg_type_f1",
            "n_occurrence_predictions",
            "alpha",
            "p_min",
            "tau",
        ],
        ascending=[
            False,
            False,
            False,
            True,   # prefer fewer predictions
            True,   # prefer more collapse influence if tied
            False,  # prefer stricter eligibility
            False,  # prefer stricter decision threshold
        ],
        kind="mergesort",
    )

    best = subset.iloc[0]
    selected[branch_name] = {
        key: (
            float(best[key])
            if key not in [
                "branch",
                "n_occurrence_predictions",
            ]
            else (
                int(best[key])
                if key == "n_occurrence_predictions"
                else best[key]
            )
        )
        for key in [
            "branch",
            "alpha",
            "p_min",
            "tau",
            "p_keep",
            "n_occurrence_predictions",
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
            "PA_micro_precision",
            "PA_micro_recall",
            "PA_micro_f1",
            "PA_type_f1",
            "MH_micro_precision",
            "MH_micro_recall",
            "MH_micro_f1",
            "MH_type_f1",
        ]
    }

results.to_csv(
    TUNING_OUTPUT_PATH,
    index=False,
)

selection_metadata = {
    "test_data_accessed": False,
    "selection_metric": (
        "mean of domain-averaged document-surface micro F1 "
        "and domain-averaged normalized type F1"
    ),
    "nested_neural_threshold": NESTED_P_KEEP,
    "nonoverlap_neural_threshold": NONOVERLAP_P_KEEP,
    "alpha_grid": ALPHA_GRID,
    "p_min_grid": P_MIN_GRID,
    "tau_grid": TAU_GRID,
    "alpha_1_interpretation": (
        "neural-only lower-threshold rescue control"
    ),
    "selected": selected,
}

with open(
    SELECTION_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        selection_metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("NEURAL BASELINES")
print("=" * 78)

baseline_columns = [
    "branch",
    "n_occurrence_predictions",
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_precision",
    "PA_micro_recall",
    "PA_micro_f1",
    "MH_micro_precision",
    "MH_micro_recall",
    "MH_micro_f1",
]

print(
    results.loc[
        results["branch"].str.endswith(
            "NEURAL_BASELINE"
        ),
        baseline_columns,
    ]
    .round(4)
    .to_string(index=False)
)

for branch_name in [
    "NESTED_PLUS_COLLAPSE",
    "NONOVERLAP_PLUS_COLLAPSE",
]:
    print()
    print("=" * 78)
    print(f"TOP 10 — {branch_name}")
    print("=" * 78)

    top = (
        results.loc[
            results["branch"] == branch_name
        ]
        .sort_values(
            "selection_score",
            ascending=False,
        )
        .head(10)
    )

    print(
        top[
            [
                "alpha",
                "p_min",
                "tau",
                "n_occurrence_predictions",
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
                "PA_micro_f1",
                "PA_type_f1",
                "MH_micro_f1",
                "MH_type_f1",
            ]
        ]
        .round(4)
        .to_string(index=False)
    )

print()
print("=" * 78)
print("SELECTED MATCHED COLLAPSE CONFIGURATIONS")
print("=" * 78)

print(
    pd.DataFrame(
        selected.values()
    ).round(4).to_string(index=False)
)

print()
print("Saved complete tuning table to:")
print(TUNING_OUTPUT_PATH)

print()
print("Saved selected configurations to:")
print(SELECTION_OUTPUT_PATH)

print()
print("Cell 26 completed successfully.")

In [ ]:
# CELL 27 — Added-candidate audit and collapse permutation control

import json
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

DEV_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_anchor_collapse_comparison.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

ADDED_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_added_candidates.csv"
)

PERMUTATION_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_permutation.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_permutation_summary.json"
)

P_KEEP = 0.81
ALPHA = 0.80
P_MIN = 0.75
TAU = 0.65

N_PERMUTATIONS = 1000
PERMUTATION_SEED = 20260917

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(str(value).casefold().split())


def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_character_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_character_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted = []
    intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        if not any(
            intervals_overlap(start, end, old_start, old_end)
            for old_start, old_end in intervals
        ):
            accepted.append(row.Index)
            intervals.append((start, end))

    return group.loc[accepted].copy()


def prf(predicted, gold):
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def evaluate(table, gold_by_domain):
    domain_results = {}

    for domain in ["parkinson", "mental_health"]:
        subset = table.loc[
            table["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                subset["num_article"],
                subset["mention"],
            )
        )

        predicted_types = {
            normalize_type(value)
            for value in subset["mention"]
            if str(value).strip()
        }

        pair_scores = prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_scores = prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        domain_results[domain] = {
            "micro_precision": pair_scores[0],
            "micro_recall": pair_scores[1],
            "micro_f1": pair_scores[2],
            "type_precision": type_scores[0],
            "type_recall": type_scores[1],
            "type_f1": type_scores[2],
        }

    avg_micro_f1 = np.mean(
        [
            domain_results["parkinson"]["micro_f1"],
            domain_results["mental_health"]["micro_f1"],
        ]
    )

    avg_type_f1 = np.mean(
        [
            domain_results["parkinson"]["type_f1"],
            domain_results["mental_health"]["type_f1"],
        ]
    )

    return {
        "domains": domain_results,
        "avg_micro_f1": float(avg_micro_f1),
        "avg_type_f1": float(avg_type_f1),
        "selection_score": float(
            (avg_micro_f1 + avg_type_f1) / 2
        ),
    }


# ---------------------------------------------------------------------
# Load candidates and official development gold
# ---------------------------------------------------------------------

development = pd.read_csv(DEV_PATH)

development["num_article"] = normalize_article_id(
    development["num_article"]
)

development["mention"] = (
    development["mention"]
    .fillna("")
    .astype(str)
)

PA_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

MH_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(PA_GOLD_PATH)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(MH_GOLD_PATH)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

development_documents = (
    split_table.loc[
        split_table["split"] == "development",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

development_gold = (
    gold.merge(
        development_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = development_gold.loc[
        development_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(value)
            for value in subset["term"]
            if str(value).strip()
        },
    }

# ---------------------------------------------------------------------
# Construct broad non-overlap population
# ---------------------------------------------------------------------

nonoverlap_parts = []

for _, document_group in development.groupby(
    ["domain", "num_article"],
    sort=False,
):
    nonoverlap_parts.append(
        greedy_nonoverlap(document_group)
    )

nonoverlap = pd.concat(
    nonoverlap_parts,
    ignore_index=False,
).copy()

neural = nonoverlap["p_term"].to_numpy(dtype=float)
collapse = nonoverlap[
    "collapse_nonoverlap_anchor"
].to_numpy(dtype=float)

baseline_mask = neural >= P_KEEP

observed_combined = (
    ALPHA * neural
    + (1.0 - ALPHA) * collapse
)

observed_mask = (
    baseline_mask
    |
    (
        (neural >= P_MIN)
        & (observed_combined >= TAU)
    )
)

baseline_predictions = nonoverlap.loc[
    baseline_mask
].copy()

observed_predictions = nonoverlap.loc[
    observed_mask
].copy()

baseline_metrics = evaluate(
    baseline_predictions,
    gold_by_domain,
)

observed_metrics = evaluate(
    observed_predictions,
    gold_by_domain,
)

observed_improvement = (
    observed_metrics["selection_score"]
    - baseline_metrics["selection_score"]
)

# ---------------------------------------------------------------------
# Audit candidates added by collapse
# ---------------------------------------------------------------------

baseline_ids = set(
    baseline_predictions["candidate_id"]
)

added = observed_predictions.loc[
    ~observed_predictions["candidate_id"].isin(
        baseline_ids
    )
].copy()

added["combined_score"] = (
    ALPHA * added["p_term"]
    + (1.0 - ALPHA)
    * added["collapse_nonoverlap_anchor"]
)

gold_pair_lookup = (
    gold_by_domain["parkinson"]["pairs"]
    | gold_by_domain["mental_health"]["pairs"]
)

added["exact_gold_pair"] = [
    (article, mention) in gold_pair_lookup
    for article, mention in zip(
        added["num_article"],
        added["mention"],
    )
]

gold_type_lookup = (
    gold_by_domain["parkinson"]["types"]
    | gold_by_domain["mental_health"]["types"]
)

added["gold_type"] = [
    normalize_type(mention) in gold_type_lookup
    for mention in added["mention"]
]

added.to_csv(
    ADDED_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Permutation test
# ---------------------------------------------------------------------

rng = np.random.default_rng(
    PERMUTATION_SEED
)

domain_values = nonoverlap["domain"].to_numpy()
unique_domains = np.unique(domain_values)

permutation_rows = []

for permutation_id in tqdm(
    range(N_PERMUTATIONS),
    desc="Permuting collapse signal",
):
    permuted_collapse = collapse.copy()

    # Shuffle only within domain to preserve domain distributions.
    for domain in unique_domains:
        positions = np.flatnonzero(
            domain_values == domain
        )

        permuted_collapse[positions] = rng.permutation(
            permuted_collapse[positions]
        )

    permuted_combined = (
        ALPHA * neural
        + (1.0 - ALPHA) * permuted_collapse
    )

    permuted_mask = (
        baseline_mask
        |
        (
            (neural >= P_MIN)
            & (permuted_combined >= TAU)
        )
    )

    permuted_predictions = nonoverlap.loc[
        permuted_mask
    ]

    permuted_metrics = evaluate(
        permuted_predictions,
        gold_by_domain,
    )

    improvement = (
        permuted_metrics["selection_score"]
        - baseline_metrics["selection_score"]
    )

    permutation_rows.append(
        {
            "permutation": permutation_id,
            "n_predictions": int(
                permuted_mask.sum()
            ),
            "avg_micro_f1": (
                permuted_metrics["avg_micro_f1"]
            ),
            "avg_type_f1": (
                permuted_metrics["avg_type_f1"]
            ),
            "selection_score": (
                permuted_metrics["selection_score"]
            ),
            "improvement_over_baseline": improvement,
        }
    )

permutations = pd.DataFrame(
    permutation_rows
)

permutation_p_value = float(
    (
        1
        + (
            permutations["improvement_over_baseline"]
            >= observed_improvement
        ).sum()
    )
    / (N_PERMUTATIONS + 1)
)

permutations.to_csv(
    PERMUTATION_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Save summary
# ---------------------------------------------------------------------

summary = {
    "parameters": {
        "p_keep": P_KEEP,
        "alpha": ALPHA,
        "p_min": P_MIN,
        "tau": TAU,
    },
    "baseline_occurrence_predictions": int(
        len(baseline_predictions)
    ),
    "collapse_occurrence_predictions": int(
        len(observed_predictions)
    ),
    "added_occurrence_predictions": int(len(added)),
    "added_exact_gold_occurrences": int(
        added["exact_gold_pair"].sum()
    ),
    "added_gold_type_occurrences": int(
        added["gold_type"].sum()
    ),
    "baseline_metrics": baseline_metrics,
    "observed_metrics": observed_metrics,
    "observed_selection_improvement": float(
        observed_improvement
    ),
    "permutations": N_PERMUTATIONS,
    "permutation_mean_improvement": float(
        permutations[
            "improvement_over_baseline"
        ].mean()
    ),
    "permutation_improvement_q025": float(
        permutations[
            "improvement_over_baseline"
        ].quantile(0.025)
    ),
    "permutation_improvement_q975": float(
        permutations[
            "improvement_over_baseline"
        ].quantile(0.975)
    ),
    "permutation_p_value_one_sided": permutation_p_value,
}

with open(
    SUMMARY_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(summary, file, indent=2)

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("SELECTED COLLAPSE CONFIGURATION AUDIT")
print("=" * 78)

print(f"Baseline predictions: {len(baseline_predictions)}")
print(f"Collapse predictions: {len(observed_predictions)}")
print(f"Added predictions: {len(added)}")
print(
    "Added exact-gold predictions:",
    int(added["exact_gold_pair"].sum()),
)
print(
    "Added gold-type predictions:",
    int(added["gold_type"].sum()),
)

print()
print(
    "Baseline selection score:",
    round(baseline_metrics["selection_score"], 6),
)
print(
    "Observed selection score:",
    round(observed_metrics["selection_score"], 6),
)
print(
    "Observed improvement:",
    round(observed_improvement, 6),
)

print()
print("Permutation null:")
print(
    "Mean improvement:",
    round(
        permutations[
            "improvement_over_baseline"
        ].mean(),
        6,
    ),
)
print(
    "95% null interval:",
    (
        round(
            permutations[
                "improvement_over_baseline"
            ].quantile(0.025),
            6,
        ),
        round(
            permutations[
                "improvement_over_baseline"
            ].quantile(0.975),
            6,
        ),
    ),
)
print(
    "One-sided permutation p-value:",
    round(permutation_p_value, 6),
)

print()
print("Added candidates:")
display_columns = [
    "domain",
    "num_article",
    "mention",
    "p_term",
    "collapse_nonoverlap_anchor",
    "combined_score",
    "exact_gold_pair",
    "gold_type",
]

print(
    added[display_columns]
    .sort_values(
        ["exact_gold_pair", "p_term"],
        ascending=[False, False],
    )
    .round(6)
    .to_string(index=False)
)

print()
print("Saved added-candidate audit to:")
print(ADDED_OUTPUT_PATH)

print()
print("Saved permutation results to:")
print(PERMUTATION_OUTPUT_PATH)

print()
print("Saved permutation summary to:")
print(SUMMARY_OUTPUT_PATH)

print()
print("Cell 27 completed successfully.")

In [ ]:
# CELL 28 — Lock protocol and verify candidate-confidence construction

import json
import hashlib
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
MODEL_DIR = EXPERIMENT_DIR / "models"

DEV_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_full_pairing_max16_candidate_pool.csv"
)

SELECTION_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_matched_collapse_selection.json"
)

PERMUTATION_SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_permutation_summary.json"
)

ANCHOR_METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_anchor_collapse_comparison_metadata.json"
)

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

CHECKPOINT_PATH = (
    MODEL_DIR
    / "pubmedbert_boundary_seed42/checkpoint-104"
)

LOCKED_PROTOCOL_PATH = (
    RESULTS_DIR
    / "taskA_seed42_locked_test_protocol.json"
)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def maximum_absolute_error(observed, reconstructed):
    return float(
        np.max(
            np.abs(
                observed.astype(float)
                - reconstructed.astype(float)
            )
        )
    )


# ---------------------------------------------------------------------
# Verify required artifacts
# ---------------------------------------------------------------------

required_paths = [
    DEV_CANDIDATE_PATH,
    SELECTION_PATH,
    PERMUTATION_SUMMARY_PATH,
    ANCHOR_METADATA_PATH,
    DATASET_PATH,
    CHECKPOINT_PATH,
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required artifacts are missing:\n"
        + "\n".join(missing_paths)
    )

# ---------------------------------------------------------------------
# Infer the exact p_term construction used previously
# ---------------------------------------------------------------------

development = pd.read_csv(
    DEV_CANDIDATE_PATH
)

required_columns = {
    "p_start",
    "p_end",
    "p_term",
}

missing = required_columns.difference(
    development.columns
)

if missing:
    raise ValueError(
        f"Candidate pool is missing: {sorted(missing)}"
    )

p_start = development[
    "p_start"
].to_numpy(dtype=float)

p_end = development[
    "p_end"
].to_numpy(dtype=float)

p_term = development[
    "p_term"
].to_numpy(dtype=float)

candidate_formulas = {
    "arithmetic_mean": (
        p_start + p_end
    ) / 2.0,

    "geometric_mean": np.sqrt(
        np.clip(p_start * p_end, 0.0, None)
    ),

    "minimum": np.minimum(
        p_start,
        p_end,
    ),

    "product": (
        p_start * p_end
    ),
}

formula_errors = {
    name: maximum_absolute_error(
        p_term,
        reconstructed,
    )
    for name, reconstructed
    in candidate_formulas.items()
}

selected_formula = min(
    formula_errors,
    key=formula_errors.get,
)

selected_formula_error = formula_errors[
    selected_formula
]

# CSV round trips may introduce tiny differences, but not substantive ones.
if selected_formula_error > 1e-6:
    raise ValueError(
        "No candidate-confidence formula reproduced p_term within "
        f"tolerance. Best formula: {selected_formula}; "
        f"maximum error: {selected_formula_error}"
    )

# ---------------------------------------------------------------------
# Load development decisions
# ---------------------------------------------------------------------

with open(
    SELECTION_PATH,
    "r",
    encoding="utf-8",
) as file:
    selection = json.load(file)

with open(
    PERMUTATION_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as file:
    permutation_summary = json.load(file)

with open(
    ANCHOR_METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    anchor_metadata = json.load(file)

# ---------------------------------------------------------------------
# Lock protocol
# ---------------------------------------------------------------------

protocol = {
    "status": "LOCKED_BEFORE_TEST_INFERENCE",
    "test_data_used_for_parameter_selection": False,

    "neural_model": {
        "checkpoint": str(CHECKPOINT_PATH),
        "training_seed": 42,
        "candidate_minimum_boundary_probability": 0.05,
        "maximum_span_tokens": 16,
        "candidate_confidence_formula": selected_formula,
        "candidate_confidence_formula_maximum_error": (
            selected_formula_error
        ),
    },

    "nested_neural_baseline": {
        "policy": "retain all predicted occurrences including overlaps",
        "p_term_threshold": 0.97,
    },

    "nonoverlap_neural_baseline": {
        "policy": (
            "greedy highest-neural-confidence non-overlap selection"
        ),
        "p_term_threshold": 0.81,
    },

    "nested_plus_collapse": {
        "training_reference": "NESTED_HIGH_CONFIDENCE",
        "collapse_definition": (
            "exp(-assigned-centroid Euclidean distance / "
            "training-distance IQR)"
        ),
        "n_clusters_per_domain": 100,
        "p_keep": 0.97,
        "alpha": 0.50,
        "p_min": 0.75,
        "tau": 0.95,
        "development_effect": (
            "identical to nested neural baseline; no candidates added"
        ),
    },

    "nonoverlap_plus_collapse": {
        "training_reference": "NONOVERLAP_HIGH_CONFIDENCE",
        "collapse_definition": (
            "exp(-assigned-centroid Euclidean distance / "
            "training-distance IQR)"
        ),
        "n_clusters_per_domain": 100,
        "p_keep": 0.81,
        "alpha": 0.80,
        "p_min": 0.75,
        "tau": 0.65,
        "development_added_occurrences": 12,
        "development_exact_gold_additions": 3,
        "development_selection_score_change": (
            permutation_summary[
                "observed_selection_improvement"
            ]
        ),
        "development_permutation_p_one_sided": (
            permutation_summary[
                "permutation_p_value_one_sided"
            ]
        ),
    },

    "claim_calibration": {
        "development_conclusion": (
            "No reliable positive collapse contribution was detected."
        ),
        "test_interpretation": (
            "Test results will estimate generalization of the locked "
            "pipelines and will not be used for further tuning."
        ),
        "inverse_distance_status": (
            "Not part of the confirmatory collapse experiment; any "
            "future analysis must be labeled exploratory."
        ),
    },

    "artifact_fingerprints": {
        "development_candidate_pool_sha256": (
            file_sha256(DEV_CANDIDATE_PATH)
        ),
        "selection_json_sha256": (
            file_sha256(SELECTION_PATH)
        ),
        "permutation_summary_sha256": (
            file_sha256(PERMUTATION_SUMMARY_PATH)
        ),
        "anchor_metadata_sha256": (
            file_sha256(ANCHOR_METADATA_PATH)
        ),
    },
}

with open(
    LOCKED_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("CANDIDATE-CONFIDENCE FORMULA AUDIT")
print("=" * 78)

for name, error in sorted(
    formula_errors.items(),
    key=lambda item: item[1],
):
    print(
        f"{name:20s} maximum absolute error: {error:.12g}"
    )

print()
print("Selected formula:", selected_formula)
print(
    "Maximum reconstruction error:",
    selected_formula_error,
)

print()
print("=" * 78)
print("LOCKED TEST PROTOCOL")
print("=" * 78)
print("Neural checkpoint:", CHECKPOINT_PATH)
print("Maximum span tokens: 16")
print("Minimum boundary probability: 0.05")
print("Nested neural threshold: 0.97")
print("Non-overlap neural threshold: 0.81")
print()
print(
    "Nested + collapse:",
    "alpha=0.50, p_min=0.75, tau=0.95",
)
print(
    "Non-overlap + collapse:",
    "alpha=0.80, p_min=0.75, tau=0.65",
)
print(
    "Development permutation p-value:",
    permutation_summary[
        "permutation_p_value_one_sided"
    ],
)

print()
print("Saved locked protocol to:")
print(LOCKED_PROTOCOL_PATH)

print()
print("Cell 28 completed successfully.")

In [ ]:
# CELL 29 — Generate untouched test logits and candidate pool

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_from_disk
from scipy.special import expit
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# ---------------------------------------------------------------------
# Paths and locked parameters
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
MODEL_DIR = EXPERIMENT_DIR / "models"

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

DOCUMENT_PATH = (
    EXPERIMENT_DIR
    / "splits/taskA_documents_with_token_lengths.csv"
)

CHECKPOINT_PATH = (
    MODEL_DIR
    / "pubmedbert_boundary_seed42/checkpoint-104"
)

TEST_LOGITS_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_logits.npz"
)

TEST_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_full_pairing_max16_candidate_pool.csv"
)

TEST_ORACLE_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_candidate_oracle.csv"
)

TEST_METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_candidate_generation_metadata.json"
)

TEMP_INFERENCE_DIR = (
    EXPERIMENT_DIR
    / "cache/test_inference_seed42"
)

MIN_BOUNDARY_PROBABILITY = 0.05
MAX_SPAN_TOKENS = 16
EVAL_BATCH_SIZE = 16
SEED = 42

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_term_key(value):
    return " ".join(
        str(value).casefold().split()
    )


def infer_text_column(table):
    candidates = [
        "text",
        "abstract_text",
        "document_text",
        "abstract",
        "content",
    ]

    for column in candidates:
        if column in table.columns:
            return column

    raise ValueError(
        "Could not identify the document-text column. "
        f"Available columns: {list(table.columns)}"
    )


# ---------------------------------------------------------------------
# Load dataset and document text
# ---------------------------------------------------------------------

dataset = load_from_disk(
    str(DATASET_PATH)
)

test_dataset = dataset["test"]

documents = pd.read_csv(
    DOCUMENT_PATH
)

documents["num_article"] = normalize_article_id(
    documents["num_article"]
)

text_column = infer_text_column(
    documents
)

document_text_lookup = {
    (
        str(row.domain),
        str(row.num_article),
    ): str(getattr(row, text_column))
    for row in documents.itertuples(index=False)
}

print("=" * 78)
print("TEST INFERENCE SETUP")
print("=" * 78)
print("Checkpoint:", CHECKPOINT_PATH)
print("Test windows:", len(test_dataset))
print("Document text column:", text_column)
print("Minimum boundary probability:", MIN_BOUNDARY_PROBABILITY)
print("Maximum span tokens:", MAX_SPAN_TOKENS)

# ---------------------------------------------------------------------
# Load locked checkpoint
# ---------------------------------------------------------------------

model = AutoModelForTokenClassification.from_pretrained(
    str(CHECKPOINT_PATH)
)

model = AutoModelForTokenClassification.from_pretrained(
    str(CHECKPOINT_PATH)
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    use_fast=True,
)

inference_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

use_bf16 = bool(
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

use_fp16 = bool(
    torch.cuda.is_available()
    and not use_bf16
)

inference_arguments = TrainingArguments(
    output_dir=str(TEMP_INFERENCE_DIR),
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to=[],
    dataloader_num_workers=0,
    seed=SEED,
    remove_unused_columns=True,
)

inference_trainer = Trainer(
    model=model,
    args=inference_arguments,
    data_collator=inference_collator
)


# ---------------------------------------------------------------------
# Generate test logits
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("GENERATING UNTOUCHED TEST LOGITS")
print("=" * 78)

prediction_output = inference_trainer.predict(
    test_dataset
)

test_logits = prediction_output.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

if (
    test_logits.ndim != 3
    or test_logits.shape[-1] != 2
):
    raise ValueError(
        f"Unexpected logits shape: {test_logits.shape}"
    )

test_probabilities = expit(
    np.asarray(test_logits, dtype=np.float64)
)

np.savez_compressed(
    TEST_LOGITS_PATH,
    logits=np.asarray(
        test_logits,
        dtype=np.float32,
    ),
    probabilities=np.asarray(
        test_probabilities,
        dtype=np.float32,
    ),
)

print("Test logits shape:", test_logits.shape)
print(
    "Probability range:",
    float(test_probabilities.min()),
    "to",
    float(test_probabilities.max()),
)

# ---------------------------------------------------------------------
# Decode complete start/end cross-product
# ---------------------------------------------------------------------

candidate_rows = []

for row_index in tqdm(
    range(len(test_dataset)),
    desc="Decoding test windows",
):
    row = test_dataset[row_index]

    domain = str(row["domain"])
    article = str(row["num_article"])
    window_index = int(row["window_index"])

    lookup_key = (
        domain,
        article,
    )

    if lookup_key not in document_text_lookup:
        raise KeyError(
            f"No document text found for {lookup_key}"
        )

    document_text = document_text_lookup[
        lookup_key
    ]

    offsets = np.asarray(
        row["offset_mapping"],
        dtype=np.int64,
    )

    attention_mask = np.asarray(
        row["attention_mask"],
        dtype=np.int8,
    )

    label_mask = np.asarray(
        row["label_mask"],
        dtype=np.int8,
    )

    sequence_length = len(offsets)

    start_probabilities = test_probabilities[
        row_index,
        :sequence_length,
        0,
    ]

    end_probabilities = test_probabilities[
        row_index,
        :sequence_length,
        1,
    ]

    valid_token = (
        (attention_mask == 1)
        & (label_mask == 1)
        & (offsets[:, 1] > offsets[:, 0])
    )

    valid_start_indices = np.flatnonzero(
        valid_token
        & (
            start_probabilities
            >= MIN_BOUNDARY_PROBABILITY
        )
    )

    valid_end_mask = (
        valid_token
        & (
            end_probabilities
            >= MIN_BOUNDARY_PROBABILITY
        )
    )

    for start_token in valid_start_indices:
        maximum_end_token = min(
            sequence_length - 1,
            start_token + MAX_SPAN_TOKENS - 1,
        )

        possible_end_tokens = np.arange(
            start_token,
            maximum_end_token + 1,
        )

        possible_end_tokens = possible_end_tokens[
            valid_end_mask[possible_end_tokens]
        ]

        for end_token in possible_end_tokens:
            start_char = int(
                offsets[start_token, 0]
            )

            end_char = int(
                offsets[end_token, 1]
            )

            if end_char <= start_char:
                continue

            mention = document_text[
                start_char:end_char
            ]

            if not mention.strip():
                continue

            p_start = float(
                start_probabilities[start_token]
            )

            p_end = float(
                end_probabilities[end_token]
            )

            p_term = float(
                np.sqrt(p_start * p_end)
            )

            candidate_rows.append(
                {
                    "domain": domain,
                    "num_article": article,
                    "doi": str(row["doi"]),
                    "split": "test",
                    "window_index": window_index,
                    "start_token": int(start_token),
                    "end_token": int(end_token),
                    "span_token_count": int(
                        end_token - start_token + 1
                    ),
                    "start_char": start_char,
                    "end_char": end_char,
                    "mention": mention,
                    "term_key": normalize_term_key(
                        mention
                    ),
                    "p_start": p_start,
                    "p_end": p_end,
                    "p_term": p_term,
                }
            )

raw_candidates = pd.DataFrame(
    candidate_rows
)

if raw_candidates.empty:
    raise ValueError(
        "Test decoding produced no candidates."
    )

raw_count = len(raw_candidates)

# ---------------------------------------------------------------------
# Remove duplicate occurrence predictions created by overlapping windows
# ---------------------------------------------------------------------

raw_candidates = raw_candidates.sort_values(
    by=[
        "domain",
        "num_article",
        "start_char",
        "end_char",
        "p_term",
        "p_start",
        "p_end",
        "window_index",
    ],
    ascending=[
        True,
        True,
        True,
        True,
        False,
        False,
        False,
        True,
    ],
    kind="mergesort",
)

test_candidates = (
    raw_candidates.drop_duplicates(
        subset=[
            "domain",
            "num_article",
            "start_char",
            "end_char",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

test_candidates.insert(
    0,
    "candidate_id",
    np.arange(
        len(test_candidates),
        dtype=np.int64,
    ),
)

test_candidates.to_csv(
    TEST_POOL_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Construction checks
# ---------------------------------------------------------------------

if not np.allclose(
    test_candidates["p_term"],
    np.sqrt(
        test_candidates["p_start"]
        * test_candidates["p_end"]
    ),
    atol=1e-12,
    rtol=1e-12,
):
    raise ValueError(
        "p_term is inconsistent with the locked geometric mean."
    )

if (
    test_candidates["span_token_count"]
    > MAX_SPAN_TOKENS
).any():
    raise ValueError(
        "A decoded candidate exceeds the locked span limit."
    )

if (
    test_candidates[["start_char", "end_char"]]
    .duplicated()
    .all()
):
    raise ValueError(
        "Unexpected candidate duplication condition."
    )

duplicate_count = (
    raw_count - len(test_candidates)
)

metadata = {
    "split": "test",
    "gold_accessed_during_generation": False,
    "checkpoint": str(CHECKPOINT_PATH),
    "test_windows": int(len(test_dataset)),
    "minimum_boundary_probability": (
        MIN_BOUNDARY_PROBABILITY
    ),
    "maximum_span_tokens": MAX_SPAN_TOKENS,
    "candidate_confidence": (
        "sqrt(p_start * p_end)"
    ),
    "raw_window_candidates": int(raw_count),
    "unique_occurrence_candidates": int(
        len(test_candidates)
    ),
    "duplicate_window_candidates_removed": int(
        duplicate_count
    ),
    "bf16_inference": use_bf16,
    "fp16_inference": use_fp16,
    "logits_path": str(TEST_LOGITS_PATH),
    "candidate_pool_path": str(TEST_POOL_PATH),
}

with open(
    TEST_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("UNTOUCHED TEST CANDIDATE POOL")
print("=" * 78)
print("Raw window-level candidates:", raw_count)
print(
    "Unique occurrence-level candidates:",
    len(test_candidates),
)
print(
    "Duplicate window candidates removed:",
    duplicate_count,
)

print()
print("Candidates by domain:")
print(
    test_candidates.groupby("domain")
    .size()
    .to_string()
)

print()
print("Confidence percentiles:")
print(
    test_candidates["p_term"]
    .quantile(
        [
            0.00,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)

print()
print("Span-length percentiles:")
print(
    test_candidates["span_token_count"]
    .quantile(
        [
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            1.00,
        ]
    )
)

print()
print("Candidates with p_term >= 0.50:")
print(
    test_candidates.loc[
        test_candidates["p_term"] >= 0.50
    ]
    .groupby("domain")
    .size()
    .to_string()
)

print()
print("Saved test logits to:")
print(TEST_LOGITS_PATH)

print()
print("Saved test candidate pool to:")
print(TEST_POOL_PATH)

print()
print("Saved generation metadata to:")
print(TEST_METADATA_PATH)

print()
print("Cell 29 completed successfully.")

In [ ]:
# CELL 30 — Generate frozen contextual embeddings for eligible test candidates

import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_from_disk
from tqdm.auto import tqdm
from transformers import (
    AutoModel,
    AutoTokenizer,
)

# ---------------------------------------------------------------------
# Configuration and paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

TEST_POOL_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_full_pairing_max16_candidate_pool.csv"
)

TEST_ELIGIBLE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_candidates_p0.5.csv"
)

TEST_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_test_candidate_embeddings_p0.5.npy"
)

METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_embedding_metadata.json"
)

ENCODER_NAME = (
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)

ELIGIBILITY_FLOOR = 0.50
CONTEXT_BATCH_SIZE = 24

# ---------------------------------------------------------------------
# Load test windows and eligible candidates
# ---------------------------------------------------------------------

dataset = load_from_disk(
    str(DATASET_PATH)
)

test_windows = dataset["test"]

test_pool = pd.read_csv(
    TEST_POOL_PATH
)

test_pool["num_article"] = (
    test_pool["num_article"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

eligible = (
    test_pool.loc[
        test_pool["p_term"] >= ELIGIBILITY_FLOOR
    ]
    .reset_index(drop=True)
    .copy()
)

eligible.insert(
    0,
    "embedding_row",
    np.arange(
        len(eligible),
        dtype=np.int64,
    ),
)

print("=" * 78)
print("COLLAPSE-ELIGIBLE TEST CANDIDATES")
print("=" * 78)
print("Eligibility floor:", ELIGIBILITY_FLOOR)
print("Eligible test candidates:", len(eligible))

print()
print("Candidates by domain:")
print(
    eligible.groupby("domain")
    .size()
    .to_string()
)

# ---------------------------------------------------------------------
# Build an unambiguous window lookup
# ---------------------------------------------------------------------

window_lookup = {}

for row_index in range(len(test_windows)):
    row = test_windows[row_index]

    key = (
        str(row["domain"]),
        str(row["num_article"]),
        int(row["window_index"]),
    )

    if key in window_lookup:
        raise ValueError(
            f"Duplicate test-window key: {key}"
        )

    window_lookup[key] = row_index

candidate_window_rows = []

for row in eligible.itertuples(index=False):
    key = (
        str(row.domain),
        str(row.num_article),
        int(row.window_index),
    )

    if key not in window_lookup:
        raise KeyError(
            f"Candidate refers to an unknown window: {key}"
        )

    candidate_window_rows.append(
        window_lookup[key]
    )

eligible["_dataset_window_row"] = (
    np.asarray(
        candidate_window_rows,
        dtype=np.int64,
    )
)

# ---------------------------------------------------------------------
# Load frozen contextual encoder
# ---------------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_NAME,
    use_fast=True,
)

encoder = AutoModel.from_pretrained(
    ENCODER_NAME
)

encoder.to(device)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

hidden_size = int(
    encoder.config.hidden_size
)

use_bf16 = bool(
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print()
print("Contextual encoder:", ENCODER_NAME)
print("Device:", device)
print("Hidden size:", hidden_size)
print("Context batch size:", CONTEXT_BATCH_SIZE)
print("BF16 autocast:", use_bf16)

# ---------------------------------------------------------------------
# Allocate output matrix
# ---------------------------------------------------------------------

candidate_embeddings = np.empty(
    (
        len(eligible),
        hidden_size,
    ),
    dtype=np.float32,
)

written = np.zeros(
    len(eligible),
    dtype=bool,
)

# Map each test-window row to its eligible candidate rows
candidates_by_window = {
    int(window_row): group.index.to_numpy(dtype=np.int64)
    for window_row, group in eligible.groupby(
        "_dataset_window_row",
        sort=False,
    )
}

window_rows_with_candidates = sorted(
    candidates_by_window.keys()
)

# ---------------------------------------------------------------------
# Encode each required window once and mean-pool candidate spans
# ---------------------------------------------------------------------

for batch_start in tqdm(
    range(
        0,
        len(window_rows_with_candidates),
        CONTEXT_BATCH_SIZE,
    ),
    desc="Embedding eligible test candidates",
):
    batch_window_rows = window_rows_with_candidates[
        batch_start:
        batch_start + CONTEXT_BATCH_SIZE
    ]

    batch_input_ids = [
        test_windows[row_index]["input_ids"]
        for row_index in batch_window_rows
    ]

    batch_attention_masks = [
        test_windows[row_index]["attention_mask"]
        for row_index in batch_window_rows
    ]

    batch_token_type_ids = [
        test_windows[row_index]["token_type_ids"]
        for row_index in batch_window_rows
    ]

    batch = tokenizer.pad(
        {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_masks,
            "token_type_ids": batch_token_type_ids,
        },
        padding=True,
        return_tensors="pt",
    )

    batch = {
        key: value.to(device)
        for key, value in batch.items()
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=use_bf16,
        ):
            hidden_states = encoder(
                **batch
            ).last_hidden_state

    # Convert one batch at a time to float32 CPU data.
    hidden_states = (
        hidden_states
        .float()
        .cpu()
        .numpy()
    )

    for local_batch_index, window_row in enumerate(
        batch_window_rows
    ):
        candidate_rows = candidates_by_window[
            window_row
        ]

        window_hidden = hidden_states[
            local_batch_index
        ]

        sequence_length = len(
            test_windows[window_row]["input_ids"]
        )

        window_hidden = window_hidden[
            :sequence_length
        ]

        # Prefix sums allow efficient mean pooling of many spans.
        prefix = np.vstack(
            [
                np.zeros(
                    (1, hidden_size),
                    dtype=np.float32,
                ),
                np.cumsum(
                    window_hidden,
                    axis=0,
                    dtype=np.float32,
                ),
            ]
        )

        starts = eligible.loc[
            candidate_rows,
            "start_token",
        ].to_numpy(dtype=np.int64)

        ends_inclusive = eligible.loc[
            candidate_rows,
            "end_token",
        ].to_numpy(dtype=np.int64)

        ends_exclusive = ends_inclusive + 1

        if np.any(starts < 0):
            raise ValueError(
                "Negative candidate token index detected."
            )

        if np.any(ends_exclusive > sequence_length):
            raise ValueError(
                "Candidate token index exceeds its window."
            )

        span_sums = (
            prefix[ends_exclusive]
            - prefix[starts]
        )

        span_lengths = (
            ends_exclusive - starts
        ).astype(np.float32)

        pooled = (
            span_sums
            / span_lengths[:, None]
        )

        candidate_embeddings[
            candidate_rows
        ] = pooled

        written[candidate_rows] = True

# ---------------------------------------------------------------------
# Validate and save
# ---------------------------------------------------------------------

if not written.all():
    missing_rows = np.flatnonzero(
        ~written
    )

    raise ValueError(
        f"{len(missing_rows)} candidate embeddings were not written."
    )

if not np.isfinite(
    candidate_embeddings
).all():
    raise ValueError(
        "Non-finite values found in test embeddings."
    )

eligible_to_save = eligible.drop(
    columns=["_dataset_window_row"]
)

eligible_to_save.to_csv(
    TEST_ELIGIBLE_PATH,
    index=False,
)

np.save(
    TEST_EMBEDDING_PATH,
    candidate_embeddings,
)

metadata = {
    "split": "test",
    "gold_accessed": False,
    "encoder": ENCODER_NAME,
    "pooling": (
        "unweighted mean of final-layer contextual token vectors "
        "over the candidate span"
    ),
    "eligibility_floor": ELIGIBILITY_FLOOR,
    "candidate_count": int(len(eligible)),
    "embedding_shape": [
        int(value)
        for value in candidate_embeddings.shape
    ],
    "embedding_dtype": str(
        candidate_embeddings.dtype
    ),
    "bf16_autocast": use_bf16,
    "candidate_path": str(
        TEST_ELIGIBLE_PATH
    ),
    "embedding_path": str(
        TEST_EMBEDDING_PATH
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

print()
print("=" * 78)
print("TEST CONTEXTUAL EMBEDDINGS")
print("=" * 78)
print(
    "Embedding shape:",
    candidate_embeddings.shape,
)
print(
    "All values finite:",
    np.isfinite(candidate_embeddings).all(),
)
print(
    "Mean vector norm:",
    float(
        np.linalg.norm(
            candidate_embeddings.mean(axis=0)
        )
    ),
)

print()
print("Saved eligible test candidates to:")
print(TEST_ELIGIBLE_PATH)

print()
print("Saved test embeddings to:")
print(TEST_EMBEDDING_PATH)

print()
print("Saved embedding metadata to:")
print(METADATA_PATH)

print()
print("Cell 30 completed successfully.")

In [ ]:
# CELL 30A — Cross-split embedding consistency audit

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_from_disk

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

TRAIN_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_training_candidate_embeddings_p0.5.npy"
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_development_candidate_embeddings_p0.5.npy"
)

TEST_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_test_candidate_embeddings_p0.5.npy"
)

SAMPLE_SIZE = 100
AUDIT_SEED = 42

# ---------------------------------------------------------------------
# Load embeddings and compare distributions
# ---------------------------------------------------------------------

train_embeddings_saved = np.load(
    TRAIN_EMBEDDING_PATH,
    mmap_mode="r",
)

dev_embeddings_saved = np.load(
    DEV_EMBEDDING_PATH,
    mmap_mode="r",
)

test_embeddings_saved = np.load(
    TEST_EMBEDDING_PATH,
    mmap_mode="r",
)

def embedding_summary(name, matrix):
    vector_norms = np.linalg.norm(
        np.asarray(matrix),
        axis=1,
    )

    return {
        "split": name,
        "rows": int(matrix.shape[0]),
        "dimensions": int(matrix.shape[1]),
        "mean_vector_norm": float(
            np.linalg.norm(
                np.asarray(matrix).mean(axis=0)
            )
        ),
        "individual_norm_q05": float(
            np.quantile(vector_norms, 0.05)
        ),
        "individual_norm_median": float(
            np.median(vector_norms)
        ),
        "individual_norm_q95": float(
            np.quantile(vector_norms, 0.95)
        ),
        "global_mean": float(
            np.asarray(matrix).mean()
        ),
        "global_std": float(
            np.asarray(matrix).std()
        ),
    }

distribution_summary = pd.DataFrame(
    [
        embedding_summary(
            "training",
            train_embeddings_saved,
        ),
        embedding_summary(
            "development",
            dev_embeddings_saved,
        ),
        embedding_summary(
            "test",
            test_embeddings_saved,
        ),
    ]
)

print("=" * 78)
print("CROSS-SPLIT EMBEDDING DISTRIBUTIONS")
print("=" * 78)

print(
    distribution_summary
    .round(6)
    .to_string(index=False)
)

# ---------------------------------------------------------------------
# Re-encode a deterministic training sample with the current encoder
# ---------------------------------------------------------------------

training_candidates = pd.read_csv(
    TRAIN_CANDIDATE_PATH
)

training_candidates["num_article"] = (
    training_candidates["num_article"]
    .astype(str)
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

if len(training_candidates) != train_embeddings_saved.shape[0]:
    raise ValueError(
        "Current training candidate table does not match the stored "
        "training embedding matrix."
    )

dataset = load_from_disk(
    str(DATASET_PATH)
)

train_windows = dataset["train"]

window_lookup = {}

for window_row in range(len(train_windows)):
    row = train_windows[window_row]

    key = (
        str(row["domain"]),
        str(row["num_article"]),
        int(row["window_index"]),
    )

    window_lookup[key] = window_row

rng = np.random.default_rng(
    AUDIT_SEED
)

sample_rows = np.sort(
    rng.choice(
        len(training_candidates),
        size=min(
            SAMPLE_SIZE,
            len(training_candidates),
        ),
        replace=False,
    )
)

fresh_embeddings = []

encoder.eval()

for candidate_row in sample_rows:
    candidate = training_candidates.iloc[
        candidate_row
    ]

    key = (
        str(candidate["domain"]),
        str(candidate["num_article"]),
        int(candidate["window_index"]),
    )

    if key not in window_lookup:
        raise KeyError(
            f"Training window not found: {key}"
        )

    window = train_windows[
        window_lookup[key]
    ]

    inputs = {
        "input_ids": torch.tensor(
            [window["input_ids"]],
            dtype=torch.long,
            device=device,
        ),
        "attention_mask": torch.tensor(
            [window["attention_mask"]],
            dtype=torch.long,
            device=device,
        ),
        "token_type_ids": torch.tensor(
            [window["token_type_ids"]],
            dtype=torch.long,
            device=device,
        ),
    }

    with torch.inference_mode():
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=use_bf16,
        ):
            hidden = encoder(
                **inputs
            ).last_hidden_state[0]

    start_token = int(
        candidate["start_token"]
    )

    end_token = int(
        candidate["end_token"]
    )

    pooled = (
        hidden[
            start_token:end_token + 1
        ]
        .float()
        .mean(dim=0)
        .cpu()
        .numpy()
    )

    fresh_embeddings.append(
        pooled
    )

fresh_embeddings = np.stack(
    fresh_embeddings
).astype(np.float32)

saved_sample = np.asarray(
    train_embeddings_saved[sample_rows],
    dtype=np.float32,
)

# ---------------------------------------------------------------------
# Direct reproducibility measures
# ---------------------------------------------------------------------

dot_products = np.sum(
    saved_sample * fresh_embeddings,
    axis=1,
)

denominators = (
    np.linalg.norm(saved_sample, axis=1)
    * np.linalg.norm(fresh_embeddings, axis=1)
)

cosine_similarity = (
    dot_products
    / np.maximum(denominators, 1e-12)
)

absolute_difference = np.abs(
    saved_sample - fresh_embeddings
)

relative_l2_error = (
    np.linalg.norm(
        saved_sample - fresh_embeddings,
        axis=1,
    )
    / np.maximum(
        np.linalg.norm(saved_sample, axis=1),
        1e-12,
    )
)

print()
print("=" * 78)
print("TRAINING EMBEDDING REPRODUCTION")
print("=" * 78)
print("Candidates re-encoded:", len(sample_rows))
print(
    "Cosine similarity — minimum:",
    float(cosine_similarity.min()),
)
print(
    "Cosine similarity — median:",
    float(np.median(cosine_similarity)),
)
print(
    "Cosine similarity — mean:",
    float(cosine_similarity.mean()),
)
print(
    "Maximum absolute coordinate difference:",
    float(absolute_difference.max()),
)
print(
    "Median relative L2 error:",
    float(np.median(relative_l2_error)),
)
print(
    "Maximum relative L2 error:",
    float(relative_l2_error.max()),
)

CONSISTENT = bool(
    np.median(cosine_similarity) >= 0.999
    and np.median(relative_l2_error) <= 0.02
)

print()
print("Embedding procedure consistent:", CONSISTENT)

if not CONSISTENT:
    raise ValueError(
        "The current encoder/pooling procedure does not reproduce "
        "the stored training embeddings. Do not score the test set "
        "until the mismatch is resolved."
    )

print()
print("Cell 30A completed successfully.")

In [ ]:
# CELL 31 — Apply frozen collapse states to eligible test candidates

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

STATE_DIR = (
    EXPERIMENT_DIR
    / "models/collapse_anchor_comparison_seed42"
)

TEST_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_candidates_p0.5.csv"
)

TEST_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_test_candidate_embeddings_p0.5.npy"
)

DEV_SCORED_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_anchor_collapse_comparison.csv"
)

TEST_SCORED_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_candidates_with_locked_collapse.csv"
)

SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_collapse_distribution_summary.csv"
)

METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_collapse_scoring_metadata.json"
)

POLICIES = {
    "NESTED_HIGH_CONFIDENCE": {
        "score_column": "collapse_nested_anchor",
        "distance_column": "distance_nested_anchor",
    },
    "NONOVERLAP_HIGH_CONFIDENCE": {
        "score_column": "collapse_nonoverlap_anchor",
        "distance_column": "distance_nonoverlap_anchor",
    },
}

# ---------------------------------------------------------------------
# Load and align candidates with embeddings
# ---------------------------------------------------------------------

test_candidates = pd.read_csv(
    TEST_CANDIDATE_PATH
)

test_embeddings = np.load(
    TEST_EMBEDDING_PATH,
    mmap_mode="r",
)

if len(test_candidates) != test_embeddings.shape[0]:
    raise ValueError(
        "Test candidates and embeddings are misaligned: "
        f"{len(test_candidates)} rows versus "
        f"{test_embeddings.shape[0]} embeddings."
    )

expected_embedding_rows = np.arange(
    len(test_candidates),
    dtype=np.int64,
)

actual_embedding_rows = test_candidates[
    "embedding_row"
].to_numpy(dtype=np.int64)

if not np.array_equal(
    expected_embedding_rows,
    actual_embedding_rows,
):
    raise ValueError(
        "embedding_row is not identical to embedding matrix order."
    )

for policy in POLICIES:
    test_candidates[
        POLICIES[policy]["score_column"]
    ] = np.nan

    test_candidates[
        POLICIES[policy]["distance_column"]
    ] = np.nan

summary_rows = []
metadata_states = {}

print("=" * 78)
print("APPLYING FROZEN COLLAPSE STATES TO TEST")
print("=" * 78)
print("Test candidates:", len(test_candidates))
print("Embedding shape:", test_embeddings.shape)

# ---------------------------------------------------------------------
# Score separately by policy and domain
# ---------------------------------------------------------------------

for policy, columns in POLICIES.items():
    print()
    print("-" * 78)
    print("POLICY:", policy)
    print("-" * 78)

    metadata_states[policy] = {}

    for domain in sorted(
        test_candidates["domain"].unique()
    ):
        state_path = (
            STATE_DIR
            / f"{policy.lower()}_{domain}_K100.joblib"
        )

        if not state_path.exists():
            raise FileNotFoundError(
                f"Frozen state not found: {state_path}"
            )

        state = joblib.load(
            state_path
        )

        if state["policy"] != policy:
            raise ValueError(
                f"Policy mismatch in {state_path}"
            )

        if state["domain"] != domain:
            raise ValueError(
                f"Domain mismatch in {state_path}"
            )

        scaler = state["scaler"]
        kmeans = state["kmeans"]
        temperature = float(
            state["temperature"]
        )

        domain_positions = np.flatnonzero(
            test_candidates["domain"].to_numpy()
            == domain
        )

        X_test = np.asarray(
            test_embeddings[domain_positions],
            dtype=np.float32,
        )

        X_test_scaled = scaler.transform(
            X_test
        ).astype(
            np.float32,
            copy=False,
        )

        cluster_labels = kmeans.predict(
            X_test_scaled
        )

        centers = kmeans.cluster_centers_.astype(
            np.float32,
            copy=False,
        )

        distances = np.linalg.norm(
            X_test_scaled
            - centers[cluster_labels],
            axis=1,
        ).astype(np.float32)

        collapse = np.exp(
            -distances / temperature
        ).astype(np.float32)

        test_candidates.loc[
            test_candidates["domain"] == domain,
            columns["distance_column"],
        ] = distances

        test_candidates.loc[
            test_candidates["domain"] == domain,
            columns["score_column"],
        ] = collapse

        occupancy = np.bincount(
            cluster_labels,
            minlength=100,
        )

        summary_rows.append(
            {
                "policy": policy,
                "domain": domain,
                "n_candidates": int(
                    len(domain_positions)
                ),
                "temperature": temperature,
                "distance_q25": float(
                    np.quantile(distances, 0.25)
                ),
                "distance_median": float(
                    np.median(distances)
                ),
                "distance_q75": float(
                    np.quantile(distances, 0.75)
                ),
                "collapse_q05": float(
                    np.quantile(collapse, 0.05)
                ),
                "collapse_q25": float(
                    np.quantile(collapse, 0.25)
                ),
                "collapse_median": float(
                    np.median(collapse)
                ),
                "collapse_q75": float(
                    np.quantile(collapse, 0.75)
                ),
                "collapse_q95": float(
                    np.quantile(collapse, 0.95)
                ),
                "occupied_clusters": int(
                    np.sum(occupancy > 0)
                ),
                "minimum_cluster_size": int(
                    occupancy.min()
                ),
                "maximum_cluster_size": int(
                    occupancy.max()
                ),
            }
        )

        metadata_states[policy][domain] = {
            "state_path": str(state_path),
            "temperature": temperature,
            "test_candidates": int(
                len(domain_positions)
            ),
            "occupied_clusters": int(
                np.sum(occupancy > 0)
            ),
        }

        print()
        print("Domain:", domain)
        print("Candidates:", len(domain_positions))
        print("Temperature:", round(temperature, 6))
        print(
            "Collapse percentiles:",
            {
                "q05": round(
                    float(
                        np.quantile(
                            collapse,
                            0.05,
                        )
                    ),
                    4,
                ),
                "q25": round(
                    float(
                        np.quantile(
                            collapse,
                            0.25,
                        )
                    ),
                    4,
                ),
                "median": round(
                    float(
                        np.median(collapse)
                    ),
                    4,
                ),
                "q75": round(
                    float(
                        np.quantile(
                            collapse,
                            0.75,
                        )
                    ),
                    4,
                ),
                "q95": round(
                    float(
                        np.quantile(
                            collapse,
                            0.95,
                        )
                    ),
                    4,
                ),
            },
        )

# ---------------------------------------------------------------------
# Validate
# ---------------------------------------------------------------------

for policy, columns in POLICIES.items():
    for column in [
        columns["score_column"],
        columns["distance_column"],
    ]:
        values = test_candidates[
            column
        ].to_numpy(dtype=float)

        if not np.isfinite(values).all():
            raise ValueError(
                f"Non-finite values found in {column}."
            )

    if not test_candidates[
        columns["score_column"]
    ].between(0.0, 1.0).all():
        raise ValueError(
            f"Scores outside [0,1] for {policy}."
        )

# ---------------------------------------------------------------------
# Compare development and test distributions
# ---------------------------------------------------------------------

development = pd.read_csv(
    DEV_SCORED_PATH
)

for policy, columns in POLICIES.items():
    score_column = columns[
        "score_column"
    ]

    print()
    print("=" * 78)
    print(f"DEVELOPMENT–TEST COMPARISON: {policy}")
    print("=" * 78)

    comparison_rows = []

    for domain in sorted(
        test_candidates["domain"].unique()
    ):
        for split_name, table in [
            ("development", development),
            ("test", test_candidates),
        ]:
            values = table.loc[
                table["domain"] == domain,
                score_column,
            ].to_numpy(dtype=float)

            comparison_rows.append(
                {
                    "domain": domain,
                    "split": split_name,
                    "n": len(values),
                    "q25": np.quantile(
                        values,
                        0.25,
                    ),
                    "median": np.median(
                        values
                    ),
                    "q75": np.quantile(
                        values,
                        0.75,
                    ),
                    "q95": np.quantile(
                        values,
                        0.95,
                    ),
                }
            )

    print(
        pd.DataFrame(comparison_rows)
        .round(4)
        .to_string(index=False)
    )

# ---------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------

summary = pd.DataFrame(
    summary_rows
)

test_candidates.to_csv(
    TEST_SCORED_PATH,
    index=False,
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

metadata = {
    "split": "test",
    "gold_accessed": False,
    "method": (
        "Frozen training-only StandardScaler, KMeans centroids, "
        "and training-distance IQR temperatures"
    ),
    "test_candidates": int(
        len(test_candidates)
    ),
    "states": metadata_states,
    "scored_candidate_path": str(
        TEST_SCORED_PATH
    ),
    "summary_path": str(
        SUMMARY_PATH
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

print()
print("=" * 78)
print("TEST COLLAPSE SCORING COMPLETE")
print("=" * 78)

print(
    summary.round(4).to_string(
        index=False
    )
)

print()
print("Saved scored test candidates to:")
print(TEST_SCORED_PATH)

print()
print("Saved distribution summary to:")
print(SUMMARY_PATH)

print()
print("Saved scoring metadata to:")
print(METADATA_PATH)

print()
print("Cell 31 completed successfully.")

In [ ]:
# CELL 32 — First locked evaluation on the untouched test set

import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths and locked parameters
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

TEST_SCORED_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_test_candidates_with_locked_collapse.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

RESULT_TABLE_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_locked_test_results.csv"
)

RESULT_JSON_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_locked_test_results.json"
)

PREDICTION_OUTPUTS = {
    "NESTED_NEURAL": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nested_neural.csv"
    ),
    "NONOVERLAP_NEURAL": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nonoverlap_neural.csv"
    ),
    "NESTED_PLUS_COLLAPSE": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nested_plus_collapse.csv"
    ),
    "NONOVERLAP_PLUS_COLLAPSE": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nonoverlap_plus_collapse.csv"
    ),
}

# Locked development-selected parameters
NESTED_P_KEEP = 0.97
NESTED_ALPHA = 0.50
NESTED_P_MIN = 0.75
NESTED_TAU = 0.95

NONOVERLAP_P_KEEP = 0.81
NONOVERLAP_ALPHA = 0.80
NONOVERLAP_P_MIN = 0.75
NONOVERLAP_TAU = 0.65

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(
        str(value).casefold().split()
    )


def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_character_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_character_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted_indices = []
    accepted_intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        conflict = any(
            intervals_overlap(
                start,
                end,
                previous_start,
                previous_end,
            )
            for previous_start, previous_end
            in accepted_intervals
        )

        if not conflict:
            accepted_indices.append(row.Index)
            accepted_intervals.append(
                (start, end)
            )

    return group.loc[
        accepted_indices
    ].copy()


def calculate_prf(predicted, gold):
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def evaluate_variant(
    variant_name,
    predictions,
    gold_by_domain,
):
    result = {
        "variant": variant_name,
        "n_occurrence_predictions": int(
            len(predictions)
        ),
    }

    for domain, abbreviation in [
        ("parkinson", "PA"),
        ("mental_health", "MH"),
    ]:
        subset = predictions.loc[
            predictions["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                subset["num_article"],
                subset["mention"],
            )
        )

        predicted_types = {
            normalize_type(mention)
            for mention in subset["mention"]
            if str(mention).strip()
        }

        pair_metrics = calculate_prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_metrics = calculate_prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        result[
            f"{abbreviation}_n_unique_pairs"
        ] = len(predicted_pairs)

        result[
            f"{abbreviation}_n_unique_types"
        ] = len(predicted_types)

        for metric, value in pair_metrics.items():
            result[
                f"{abbreviation}_micro_{metric}"
            ] = value

        for metric, value in type_metrics.items():
            result[
                f"{abbreviation}_type_{metric}"
            ] = value

    result["avg_micro_f1"] = float(
        np.mean(
            [
                result["PA_micro_f1"],
                result["MH_micro_f1"],
            ]
        )
    )

    result["avg_type_f1"] = float(
        np.mean(
            [
                result["PA_type_f1"],
                result["MH_type_f1"],
            ]
        )
    )

    result["selection_score"] = float(
        np.mean(
            [
                result["avg_micro_f1"],
                result["avg_type_f1"],
            ]
        )
    )

    return result


# ---------------------------------------------------------------------
# Load eligible test candidates
# ---------------------------------------------------------------------

test_candidates = pd.read_csv(
    TEST_SCORED_PATH
)

test_candidates["num_article"] = normalize_article_id(
    test_candidates["num_article"]
)

test_candidates["mention"] = (
    test_candidates["mention"]
    .fillna("")
    .astype(str)
)

# ---------------------------------------------------------------------
# Load official test gold
# ---------------------------------------------------------------------

PA_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

MH_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(PA_GOLD_PATH)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(MH_GOLD_PATH)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

split_table = pd.read_csv(
    SPLIT_PATH
)

split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = test_gold.loc[
        test_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(term)
            for term in subset["term"]
            if str(term).strip()
        },
    }

print("=" * 78)
print("LOCKED TEST EVALUATION")
print("=" * 78)
print(
    "Parkinson test gold pairs:",
    len(gold_by_domain["parkinson"]["pairs"]),
)
print(
    "Mental-health test gold pairs:",
    len(gold_by_domain["mental_health"]["pairs"]),
)
print(
    "Eligible candidate occurrences:",
    len(test_candidates),
)

# ---------------------------------------------------------------------
# Construct the broad greedy non-overlap population
# ---------------------------------------------------------------------

nonoverlap_parts = []

for _, document_group in test_candidates.groupby(
    ["domain", "num_article"],
    sort=False,
):
    nonoverlap_parts.append(
        greedy_nonoverlap(
            document_group
        )
    )

test_nonoverlap = pd.concat(
    nonoverlap_parts,
    ignore_index=False,
).copy()

print(
    "Broad non-overlap candidate occurrences:",
    len(test_nonoverlap),
)

# ---------------------------------------------------------------------
# Apply four locked variants
# ---------------------------------------------------------------------

# A. Neural extraction retaining nested/overlapping predictions
nested_neural = test_candidates.loc[
    test_candidates["p_term"]
    >= NESTED_P_KEEP
].copy()

# B. Highest-confidence greedy non-overlap extraction
nonoverlap_neural = test_nonoverlap.loc[
    test_nonoverlap["p_term"]
    >= NONOVERLAP_P_KEEP
].copy()

# C. Nested neural extraction plus positive collapse signal
nested_combined = (
    NESTED_ALPHA
    * test_candidates["p_term"]
    + (1.0 - NESTED_ALPHA)
    * test_candidates["collapse_nested_anchor"]
)

nested_plus_collapse = test_candidates.loc[
    (
        test_candidates["p_term"]
        >= NESTED_P_KEEP
    )
    |
    (
        (
            test_candidates["p_term"]
            >= NESTED_P_MIN
        )
        & (
            nested_combined
            >= NESTED_TAU
        )
    )
].copy()

nested_plus_collapse[
    "combined_score"
] = (
    NESTED_ALPHA
    * nested_plus_collapse["p_term"]
    + (1.0 - NESTED_ALPHA)
    * nested_plus_collapse[
        "collapse_nested_anchor"
    ]
)

# D. Non-overlap neural extraction plus positive collapse signal
nonoverlap_combined = (
    NONOVERLAP_ALPHA
    * test_nonoverlap["p_term"]
    + (1.0 - NONOVERLAP_ALPHA)
    * test_nonoverlap[
        "collapse_nonoverlap_anchor"
    ]
)

nonoverlap_plus_collapse = test_nonoverlap.loc[
    (
        test_nonoverlap["p_term"]
        >= NONOVERLAP_P_KEEP
    )
    |
    (
        (
            test_nonoverlap["p_term"]
            >= NONOVERLAP_P_MIN
        )
        & (
            nonoverlap_combined
            >= NONOVERLAP_TAU
        )
    )
].copy()

nonoverlap_plus_collapse[
    "combined_score"
] = (
    NONOVERLAP_ALPHA
    * nonoverlap_plus_collapse["p_term"]
    + (1.0 - NONOVERLAP_ALPHA)
    * nonoverlap_plus_collapse[
        "collapse_nonoverlap_anchor"
    ]
)

variants = {
    "NESTED_NEURAL": nested_neural,
    "NONOVERLAP_NEURAL": nonoverlap_neural,
    "NESTED_PLUS_COLLAPSE": nested_plus_collapse,
    "NONOVERLAP_PLUS_COLLAPSE": (
        nonoverlap_plus_collapse
    ),
}

# ---------------------------------------------------------------------
# Evaluate and save
# ---------------------------------------------------------------------

result_rows = []

for variant_name, predictions in variants.items():
    predictions.to_csv(
        PREDICTION_OUTPUTS[variant_name],
        index=False,
    )

    result_rows.append(
        evaluate_variant(
            variant_name,
            predictions,
            gold_by_domain,
        )
    )

results = pd.DataFrame(
    result_rows
)

results.to_csv(
    RESULT_TABLE_PATH,
    index=False,
)

with open(
    RESULT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "protocol_status": (
                "locked before test inference"
            ),
            "test_used_for_tuning": False,
            "gold_pair_counts": {
                domain: len(
                    gold_by_domain[domain]["pairs"]
                )
                for domain in gold_by_domain
            },
            "results": results.to_dict(
                orient="records"
            ),
            "prediction_paths": {
                name: str(path)
                for name, path
                in PREDICTION_OUTPUTS.items()
            },
        },
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Added-candidate audit
# ---------------------------------------------------------------------

nested_added = set(
    nested_plus_collapse["candidate_id"]
) - set(
    nested_neural["candidate_id"]
)

nonoverlap_added = set(
    nonoverlap_plus_collapse["candidate_id"]
) - set(
    nonoverlap_neural["candidate_id"]
)

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("LOCKED TEST RESULTS")
print("=" * 78)

display_columns = [
    "variant",
    "n_occurrence_predictions",
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_precision",
    "PA_micro_recall",
    "PA_micro_f1",
    "PA_type_f1",
    "MH_micro_precision",
    "MH_micro_recall",
    "MH_micro_f1",
    "MH_type_f1",
]

print(
    results[display_columns]
    .round(4)
    .to_string(index=False)
)

print()
print("Collapse additions relative to matched baselines:")
print(
    "Nested + collapse:",
    len(nested_added),
)
print(
    "Non-overlap + collapse:",
    len(nonoverlap_added),
)

print()
print("Saved test result table to:")
print(RESULT_TABLE_PATH)

print()
print("Saved detailed test results to:")
print(RESULT_JSON_PATH)

print()
print("Cell 32 completed successfully.")

In [ ]:
# CELL 33 — Paired document-bootstrap confidence intervals

import json
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

PREDICTION_PATHS = {
    "NESTED_NEURAL": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nested_neural.csv"
    ),
    "NONOVERLAP_NEURAL": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nonoverlap_neural.csv"
    ),
    "NESTED_PLUS_COLLAPSE": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nested_plus_collapse.csv"
    ),
    "NONOVERLAP_PLUS_COLLAPSE": (
        PREDICTION_DIR
        / "pubmedbert_boundary_seed42_test_nonoverlap_plus_collapse.csv"
    ),
}

BOOTSTRAP_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_document_bootstrap.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_document_bootstrap_summary.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_document_bootstrap_metadata.json"
)

N_BOOTSTRAPS = 5000
BOOTSTRAP_SEED = 20260917

VARIANTS = list(
    PREDICTION_PATHS.keys()
)

CONTRASTS = {
    "NESTED_COLLAPSE_MINUS_NEURAL": (
        "NESTED_PLUS_COLLAPSE",
        "NESTED_NEURAL",
    ),
    "NONOVERLAP_COLLAPSE_MINUS_NEURAL": (
        "NONOVERLAP_PLUS_COLLAPSE",
        "NONOVERLAP_NEURAL",
    ),
}

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(
        str(value).casefold().split()
    )


def counts_to_metrics(tp, fp, fn):
    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def set_counts(predicted, gold):
    return (
        len(predicted & gold),
        len(predicted - gold),
        len(gold - predicted),
    )


def percentile_interval(values):
    values = np.asarray(
        values,
        dtype=float,
    )

    return {
        "estimate_mean": float(
            np.mean(values)
        ),
        "bootstrap_median": float(
            np.median(values)
        ),
        "ci_lower_95": float(
            np.quantile(values, 0.025)
        ),
        "ci_upper_95": float(
            np.quantile(values, 0.975)
        ),
    }


# ---------------------------------------------------------------------
# Load official test documents and gold
# ---------------------------------------------------------------------

split_table = pd.read_csv(
    SPLIT_PATH
)

split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

PA_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

MH_GOLD_PATH = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(PA_GOLD_PATH)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(MH_GOLD_PATH)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

# ---------------------------------------------------------------------
# Load predictions
# ---------------------------------------------------------------------

predictions = {}

for variant, path in PREDICTION_PATHS.items():
    table = pd.read_csv(path)

    table["num_article"] = normalize_article_id(
        table["num_article"]
    )

    table["mention"] = (
        table["mention"]
        .fillna("")
        .astype(str)
    )

    predictions[variant] = table

# ---------------------------------------------------------------------
# Precompute document-level sets and contingency counts
# ---------------------------------------------------------------------

domains = [
    "parkinson",
    "mental_health",
]

documents_by_domain = {
    domain: (
        test_documents.loc[
            test_documents["domain"] == domain,
            "num_article",
        ]
        .tolist()
    )
    for domain in domains
}

gold_pairs_by_document = {}
gold_types_by_document = {}

for domain in domains:
    for article in documents_by_domain[domain]:
        subset = test_gold.loc[
            (test_gold["domain"] == domain)
            & (
                test_gold["num_article"]
                == article
            )
        ]

        key = (
            domain,
            article,
        )

        gold_pairs_by_document[key] = set(
            subset["term"]
        )

        gold_types_by_document[key] = {
            normalize_type(term)
            for term in subset["term"]
            if str(term).strip()
        }

prediction_pairs_by_document = {}
prediction_types_by_document = {}
document_pair_counts = {}

for variant, table in predictions.items():
    for domain in domains:
        for article in documents_by_domain[domain]:
            subset = table.loc[
                (table["domain"] == domain)
                & (
                    table["num_article"]
                    == article
                )
            ]

            key = (
                variant,
                domain,
                article,
            )

            predicted_pairs = set(
                subset["mention"]
            )

            predicted_types = {
                normalize_type(mention)
                for mention in subset["mention"]
                if str(mention).strip()
            }

            prediction_pairs_by_document[
                key
            ] = predicted_pairs

            prediction_types_by_document[
                key
            ] = predicted_types

            document_pair_counts[key] = set_counts(
                predicted_pairs,
                gold_pairs_by_document[
                    (domain, article)
                ],
            )

# ---------------------------------------------------------------------
# Bootstrap
# ---------------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for bootstrap_id in tqdm(
    range(N_BOOTSTRAPS),
    desc="Document bootstrap",
):
    sampled_documents = {
        domain: rng.choice(
            documents_by_domain[domain],
            size=len(
                documents_by_domain[domain]
            ),
            replace=True,
        ).tolist()
        for domain in domains
    }

    replicate_results = {}

    for variant in VARIANTS:
        domain_micro_f1 = []
        domain_type_f1 = []

        row = {
            "bootstrap": bootstrap_id,
            "variant": variant,
        }

        for domain, abbreviation in [
            ("parkinson", "PA"),
            ("mental_health", "MH"),
        ]:
            sampled = sampled_documents[
                domain
            ]

            # Document-pair counts retain bootstrap multiplicity.
            tp = 0
            fp = 0
            fn = 0

            # Type inventories use the union within the sampled corpus.
            predicted_type_union = set()
            gold_type_union = set()

            for article in sampled:
                counts = document_pair_counts[
                    (
                        variant,
                        domain,
                        article,
                    )
                ]

                tp += counts[0]
                fp += counts[1]
                fn += counts[2]

                predicted_type_union.update(
                    prediction_types_by_document[
                        (
                            variant,
                            domain,
                            article,
                        )
                    ]
                )

                gold_type_union.update(
                    gold_types_by_document[
                        (
                            domain,
                            article,
                        )
                    ]
                )

            micro_precision, micro_recall, micro_f1 = (
                counts_to_metrics(
                    tp,
                    fp,
                    fn,
                )
            )

            type_tp, type_fp, type_fn = set_counts(
                predicted_type_union,
                gold_type_union,
            )

            type_precision, type_recall, type_f1 = (
                counts_to_metrics(
                    type_tp,
                    type_fp,
                    type_fn,
                )
            )

            row[
                f"{abbreviation}_micro_f1"
            ] = micro_f1

            row[
                f"{abbreviation}_type_f1"
            ] = type_f1

            domain_micro_f1.append(
                micro_f1
            )

            domain_type_f1.append(
                type_f1
            )

        row["avg_micro_f1"] = float(
            np.mean(domain_micro_f1)
        )

        row["avg_type_f1"] = float(
            np.mean(domain_type_f1)
        )

        row["selection_score"] = float(
            (
                row["avg_micro_f1"]
                + row["avg_type_f1"]
            )
            / 2
        )

        replicate_results[variant] = row
        bootstrap_rows.append(row)

bootstrap = pd.DataFrame(
    bootstrap_rows
)

bootstrap.to_csv(
    BOOTSTRAP_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Absolute metric intervals and paired contrasts
# ---------------------------------------------------------------------

summary_rows = []

metric_columns = [
    "PA_micro_f1",
    "MH_micro_f1",
    "avg_micro_f1",
    "PA_type_f1",
    "MH_type_f1",
    "avg_type_f1",
    "selection_score",
]

for variant in VARIANTS:
    subset = bootstrap.loc[
        bootstrap["variant"] == variant
    ]

    for metric in metric_columns:
        interval = percentile_interval(
            subset[metric]
        )

        summary_rows.append(
            {
                "result_type": "absolute",
                "name": variant,
                "metric": metric,
                **interval,
                "probability_greater_than_zero": np.nan,
                "probability_less_than_zero": np.nan,
            }
        )

for contrast_name, (
    treatment,
    baseline,
) in CONTRASTS.items():
    treatment_rows = (
        bootstrap.loc[
            bootstrap["variant"] == treatment
        ]
        .sort_values("bootstrap")
        .reset_index(drop=True)
    )

    baseline_rows = (
        bootstrap.loc[
            bootstrap["variant"] == baseline
        ]
        .sort_values("bootstrap")
        .reset_index(drop=True)
    )

    for metric in metric_columns:
        difference = (
            treatment_rows[metric].to_numpy()
            - baseline_rows[metric].to_numpy()
        )

        interval = percentile_interval(
            difference
        )

        summary_rows.append(
            {
                "result_type": "paired_difference",
                "name": contrast_name,
                "metric": metric,
                **interval,
                "probability_greater_than_zero": float(
                    np.mean(difference > 0)
                ),
                "probability_less_than_zero": float(
                    np.mean(difference < 0)
                ),
            }
        )

summary = pd.DataFrame(
    summary_rows
)

summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

metadata = {
    "bootstrap_unit": "document",
    "domain_stratified": True,
    "paired_across_variants": True,
    "n_bootstrap_replicates": N_BOOTSTRAPS,
    "random_seed": BOOTSTRAP_SEED,
    "parkinson_documents_per_replicate": len(
        documents_by_domain["parkinson"]
    ),
    "mental_health_documents_per_replicate": len(
        documents_by_domain["mental_health"]
    ),
    "micro_f1_bootstrap_method": (
        "Document-level TP, FP, and FN counts summed with bootstrap "
        "multiplicity."
    ),
    "type_f1_bootstrap_method": (
        "Unique normalized type inventories reconstructed from the "
        "documents sampled in each replicate."
    ),
    "confidence_interval": (
        "2.5th and 97.5th percentile interval"
    ),
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display the matched contrasts
# ---------------------------------------------------------------------

print("=" * 78)
print("PAIRED DOCUMENT-BOOTSTRAP CONTRASTS")
print("=" * 78)

contrast_display = summary.loc[
    (summary["result_type"] == "paired_difference")
    & (
        summary["metric"].isin(
            [
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
            ]
        )
    )
].copy()

print(
    contrast_display[
        [
            "name",
            "metric",
            "estimate_mean",
            "bootstrap_median",
            "ci_lower_95",
            "ci_upper_95",
            "probability_greater_than_zero",
            "probability_less_than_zero",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print()
print("=" * 78)
print("ABSOLUTE 95% INTERVALS")
print("=" * 78)

absolute_display = summary.loc[
    (summary["result_type"] == "absolute")
    & (
        summary["metric"].isin(
            [
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
            ]
        )
    )
]

print(
    absolute_display[
        [
            "name",
            "metric",
            "estimate_mean",
            "ci_lower_95",
            "ci_upper_95",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print()
print("Saved bootstrap replicates to:")
print(BOOTSTRAP_OUTPUT_PATH)

print()
print("Saved bootstrap summary to:")
print(SUMMARY_OUTPUT_PATH)

print()
print("Saved bootstrap metadata to:")
print(METADATA_OUTPUT_PATH)

print()
print("Cell 33 completed successfully.")

In [ ]:
# CELL 34 — Central post-processing ablation contrast

import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

RESULTS_DIR = (
    BASE_DIR
    / "journal_experiments/taskA_holdout/results"
)

BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_document_bootstrap.csv"
)

POINT_RESULTS_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_locked_test_results.csv"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_postprocessing_contrast.csv"
)

OUTPUT_JSON_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_postprocessing_contrast.json"
)

# ---------------------------------------------------------------------
# Load results
# ---------------------------------------------------------------------

bootstrap = pd.read_csv(
    BOOTSTRAP_PATH
)

point_results = pd.read_csv(
    POINT_RESULTS_PATH
).set_index("variant")

nested_bootstrap = (
    bootstrap.loc[
        bootstrap["variant"]
        == "NESTED_NEURAL"
    ]
    .sort_values("bootstrap")
    .reset_index(drop=True)
)

nonoverlap_bootstrap = (
    bootstrap.loc[
        bootstrap["variant"]
        == "NONOVERLAP_NEURAL"
    ]
    .sort_values("bootstrap")
    .reset_index(drop=True)
)

if not np.array_equal(
    nested_bootstrap["bootstrap"],
    nonoverlap_bootstrap["bootstrap"],
):
    raise ValueError(
        "Bootstrap replicate IDs are not aligned."
    )

metrics = [
    "PA_micro_f1",
    "MH_micro_f1",
    "avg_micro_f1",
    "PA_type_f1",
    "MH_type_f1",
    "avg_type_f1",
    "selection_score",
]

rows = []

for metric in metrics:
    differences = (
        nonoverlap_bootstrap[
            metric
        ].to_numpy(dtype=float)
        - nested_bootstrap[
            metric
        ].to_numpy(dtype=float)
    )

    point_difference = float(
        point_results.loc[
            "NONOVERLAP_NEURAL",
            metric,
        ]
        - point_results.loc[
            "NESTED_NEURAL",
            metric,
        ]
    )

    rows.append(
        {
            "contrast": (
                "NONOVERLAP_NEURAL_MINUS_NESTED_NEURAL"
            ),
            "metric": metric,
            "observed_point_difference": point_difference,
            "bootstrap_mean_difference": float(
                differences.mean()
            ),
            "bootstrap_median_difference": float(
                np.median(differences)
            ),
            "ci_lower_95": float(
                np.quantile(
                    differences,
                    0.025,
                )
            ),
            "ci_upper_95": float(
                np.quantile(
                    differences,
                    0.975,
                )
            ),
            "probability_greater_than_zero": float(
                np.mean(differences > 0)
            ),
            "probability_less_than_or_equal_zero": float(
                np.mean(differences <= 0)
            ),
        }
    )

contrast = pd.DataFrame(
    rows
)

contrast.to_csv(
    OUTPUT_PATH,
    index=False,
)

metadata = {
    "contrast": (
        "NONOVERLAP_NEURAL minus NESTED_NEURAL"
    ),
    "interpretation": (
        "Positive values favor greedy highest-confidence "
        "non-overlap post-processing."
    ),
    "bootstrap_replicates": int(
        len(nested_bootstrap)
    ),
    "paired": True,
    "domain_stratified": True,
    "point_estimates_source": str(
        POINT_RESULTS_PATH
    ),
    "bootstrap_source": str(
        BOOTSTRAP_PATH
    ),
    "results": contrast.to_dict(
        orient="records"
    ),
}

with open(
    OUTPUT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

print("=" * 78)
print("POST-PROCESSING ABLATION")
print("=" * 78)
print(
    "Contrast: NONOVERLAP_NEURAL − NESTED_NEURAL"
)

print()
print(
    contrast[
        [
            "metric",
            "observed_point_difference",
            "bootstrap_mean_difference",
            "ci_lower_95",
            "ci_upper_95",
            "probability_greater_than_zero",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print()
print("Saved contrast table to:")
print(OUTPUT_PATH)

print()
print("Saved contrast metadata to:")
print(OUTPUT_JSON_PATH)

print()
print("Cell 34 completed successfully.")

In [ ]:
# CELL 35 — Lock multi-seed robustness protocol before additional runs

import json
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

RESULTS_DIR = (
    BASE_DIR
    / "journal_experiments/taskA_holdout/results"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "taskA_multiseed_robustness_protocol.json"
)

protocol = {
    "status": "LOCKED_BEFORE_ADDITIONAL_SEED_TRAINING",

    "important_timing_disclosure": (
        "The seed-42 test result was inspected before training the "
        "additional robustness seeds. No model architecture, threshold, "
        "collapse parameter, evaluation definition, or seed was selected "
        "using the additional test results."
    ),

    "seeds": [
        13,
        21,
        42,
        77,
        101,
    ],

    "completed_seed": 42,

    "additional_seeds_to_train": [
        13,
        21,
        77,
        101,
    ],

    "neural_training": {
        "base_model": (
            "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
        ),
        "epochs": 5,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.10,
        "effective_batch_size": 16,
        "checkpoint_criterion": "development loss",
        "start_positive_weight": 6.4669974356115505,
        "end_positive_weight": 6.446241939070491,
    },

    "candidate_decoding": {
        "minimum_boundary_probability": 0.05,
        "maximum_span_tokens": 16,
        "term_confidence": "sqrt(p_start * p_end)",
        "collapse_eligibility_floor": 0.50,
    },

    "locked_neural_policies": {
        "nested_threshold": 0.97,
        "nonoverlap_threshold": 0.81,
    },

    "locked_collapse_policies": {
        "n_clusters_per_domain": 100,

        "nested_plus_collapse": {
            "anchor_policy": "NESTED_HIGH_CONFIDENCE",
            "alpha": 0.50,
            "p_keep": 0.97,
            "p_min": 0.75,
            "tau": 0.95,
        },

        "nonoverlap_plus_collapse": {
            "anchor_policy": "NONOVERLAP_HIGH_CONFIDENCE",
            "alpha": 0.80,
            "p_keep": 0.81,
            "p_min": 0.75,
            "tau": 0.65,
        },
    },

    "evaluation": {
        "primary_unit": (
            "unique exact case-sensitive document–surface-form pair"
        ),
        "secondary_unit": (
            "unique case-folded whitespace-normalized term type"
        ),
        "domains": [
            "parkinson",
            "mental_health",
        ],
        "primary_report": (
            "mean and standard deviation across five training seeds, "
            "with seed-42 document-bootstrap intervals retained as the "
            "prespecified primary-run uncertainty analysis"
        ),
    },

    "claim_rules": {
        "postprocessing_claim": (
            "Supported only if the non-overlap advantage remains "
            "positive across all or nearly all seeds."
        ),
        "collapse_claim": (
            "Supported only if collapse improves its matched baseline "
            "consistently across seeds. Otherwise report a null or "
            "negative result."
        ),
        "no_additional_test_tuning": True,
    },
}

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol,
        file,
        indent=2,
    )

print("=" * 78)
print("MULTI-SEED ROBUSTNESS PROTOCOL LOCKED")
print("=" * 78)
print("Seeds:", protocol["seeds"])
print(
    "Additional seeds:",
    protocol["additional_seeds_to_train"],
)
print("No additional test tuning:", True)

print()
print("Saved protocol to:")
print(OUTPUT_PATH)

print()
print("Cell 35 completed successfully.")

In [ ]:
# CELL 36 — Train the four additional locked neural seeds

import gc
import inspect
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from datasets import load_from_disk
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

# ---------------------------------------------------------------------
# Locked configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
MODEL_ROOT = EXPERIMENT_DIR / "models"
RESULTS_DIR = EXPERIMENT_DIR / "results"

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

MODEL_NAME = (
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)

ADDITIONAL_SEEDS = [
    13,
    21,
    77,
    101,
]

START_POS_WEIGHT = 6.4669974356115505
END_POS_WEIGHT = 6.446241939070491

EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

TRAIN_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 2
EVAL_BATCH_SIZE = 16

SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_additional_seed_training_summary.csv"
)

# ---------------------------------------------------------------------
# Load tokenizer and dataset
# ---------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

dataset = load_from_disk(
    str(DATASET_PATH)
)

model_columns = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "start_labels",
    "end_labels",
    "label_mask",
]

train_dataset = dataset[
    "train"
].select_columns(model_columns)

development_dataset = dataset[
    "development"
].select_columns(model_columns)

# ---------------------------------------------------------------------
# Dynamic-padding collator for inputs and three aligned label arrays
# ---------------------------------------------------------------------

class BoundaryDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        input_features = [
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"],
                "token_type_ids": item["token_type_ids"],
            }
            for item in features
        ]

        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            return_tensors="pt",
        )

        maximum_length = batch[
            "input_ids"
        ].shape[1]

        def pad_aligned(values, padding_value=0):
            padded = []

            for value in values:
                value = list(value)

                padded.append(
                    value
                    + [padding_value]
                    * (
                        maximum_length
                        - len(value)
                    )
                )

            return torch.tensor(
                padded,
                dtype=torch.float32,
            )

        batch["start_labels"] = pad_aligned(
            [
                item["start_labels"]
                for item in features
            ]
        )

        batch["end_labels"] = pad_aligned(
            [
                item["end_labels"]
                for item in features
            ]
        )

        batch["label_mask"] = pad_aligned(
            [
                item["label_mask"]
                for item in features
            ]
        )

        return batch


boundary_collator = BoundaryDataCollator(
    tokenizer
)

# ---------------------------------------------------------------------
# Weighted masked boundary-loss Trainer
# ---------------------------------------------------------------------

class BoundaryTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        start_labels = inputs.pop(
            "start_labels"
        )

        end_labels = inputs.pop(
            "end_labels"
        )

        label_mask = inputs.pop(
            "label_mask"
        )

        outputs = model(
            **inputs
        )

        logits = outputs.logits

        start_logits = logits[
            ...,
            0,
        ]

        end_logits = logits[
            ...,
            1,
        ]

        start_weight = torch.tensor(
            START_POS_WEIGHT,
            dtype=start_logits.dtype,
            device=start_logits.device,
        )

        end_weight = torch.tensor(
            END_POS_WEIGHT,
            dtype=end_logits.dtype,
            device=end_logits.device,
        )

        start_loss = (
            F.binary_cross_entropy_with_logits(
                start_logits,
                start_labels.to(
                    start_logits.dtype
                ),
                pos_weight=start_weight,
                reduction="none",
            )
        )

        end_loss = (
            F.binary_cross_entropy_with_logits(
                end_logits,
                end_labels.to(
                    end_logits.dtype
                ),
                pos_weight=end_weight,
                reduction="none",
            )
        )

        mask = label_mask.to(
            start_logits.dtype
        )

        denominator = torch.clamp(
            2.0 * mask.sum(),
            min=1.0,
        )

        loss = (
            (
                start_loss
                + end_loss
            )
            * mask
        ).sum() / denominator

        if return_outputs:
            return loss, outputs

        return loss


# ---------------------------------------------------------------------
# Construct version-compatible TrainingArguments
# ---------------------------------------------------------------------

def make_training_arguments(
    output_directory,
    seed,
):
    parameters = inspect.signature(
        TrainingArguments.__init__
    ).parameters

    arguments = {
        "output_dir": str(
            output_directory
        ),
        "num_train_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_steps": 13,
        "per_device_train_batch_size": (
            TRAIN_BATCH_SIZE
        ),
        "gradient_accumulation_steps": (
            GRADIENT_ACCUMULATION
        ),
        "per_device_eval_batch_size": (
            EVAL_BATCH_SIZE
        ),
        "save_strategy": "epoch",
        "logging_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "eval_loss",
        "greater_is_better": False,
        "save_total_limit": 2,
        "report_to": [],
        "seed": seed,
        "data_seed": seed,
        "dataloader_num_workers": 0,
        "remove_unused_columns": False,
        "bf16": bool(
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        ),
        "fp16": bool(
            torch.cuda.is_available()
            and not torch.cuda.is_bf16_supported()
        ),
    }

    if "eval_strategy" in parameters:
        arguments[
            "eval_strategy"
        ] = "epoch"
    else:
        arguments[
            "evaluation_strategy"
        ] = "epoch"
    if "label_names" in parameters:
        arguments["label_names"] = [
            "start_labels",
            "end_labels",
            "label_mask",
        ]


    return TrainingArguments(
        **arguments
    )


# ---------------------------------------------------------------------
# Train each additional seed
# ---------------------------------------------------------------------

summary_rows = []

print("=" * 78)
print("LOCKED MULTI-SEED TRAINING")
print("=" * 78)
print("Additional seeds:", ADDITIONAL_SEEDS)
print("Training windows:", len(train_dataset))
print(
    "Development windows:",
    len(development_dataset),
)
print(
    "Effective batch size:",
    TRAIN_BATCH_SIZE
    * GRADIENT_ACCUMULATION,
)
print(
    "GPU:",
    (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else "CPU"
    ),
)

for seed in ADDITIONAL_SEEDS:
    print()
    print("=" * 78)
    print(f"TRAINING SEED {seed}")
    print("=" * 78)

    run_directory = (
        MODEL_ROOT
        / f"pubmedbert_boundary_seed{seed}"
    )

    best_model_directory = (
        run_directory
        / "best_model"
    )

    metadata_path = (
        RESULTS_DIR
        / f"pubmedbert_boundary_seed{seed}_metadata.json"
    )

    # Safely skip an already completed run.
    if (
        metadata_path.exists()
        and best_model_directory.exists()
    ):
        print(
            "Completed artifacts already exist; "
            "skipping this seed."
        )

        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as file:
            existing_metadata = json.load(
                file
            )

        summary_rows.append(
            existing_metadata[
                "summary_row"
            ]
        )

        continue

    set_seed(seed)

    model = (
        AutoModelForTokenClassification
        .from_pretrained(
            MODEL_NAME,
            num_labels=2,
        )
    )

    training_arguments = (
        make_training_arguments(
            run_directory,
            seed,
        )
    )

    trainer = BoundaryTrainer(
        model=model,
        args=training_arguments,
        train_dataset=train_dataset,
        eval_dataset=development_dataset,
        data_collator=boundary_collator,
    )

    trainer.label_names = [
        "start_labels",
        "end_labels",
        "label_mask",
    ]

    last_checkpoint = None

    if run_directory.exists():
        last_checkpoint = get_last_checkpoint(
            str(run_directory)
        )

    if last_checkpoint:
        print(
            "Resuming from:",
            last_checkpoint,
        )
    else:
        print(
            "Starting from pretrained PubMedBERT."
        )

    start_time = time.time()

    training_result = trainer.train(
        resume_from_checkpoint=last_checkpoint
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    best_checkpoint = (
        trainer.state.best_model_checkpoint
    )

    best_eval_loss = (
        trainer.state.best_metric
    )

    final_development_metrics = (
        trainer.evaluate(
            eval_dataset=development_dataset,
            metric_key_prefix="development",
        )
    )

    trainer.save_model(
        str(best_model_directory)
    )

    tokenizer.save_pretrained(
        str(best_model_directory)
    )

    history = pd.DataFrame(
        trainer.state.log_history
    )

    history_path = (
        RESULTS_DIR
        / f"pubmedbert_boundary_seed{seed}_training_history.csv"
    )

    history.to_csv(
        history_path,
        index=False,
    )

    summary_row = {
        "seed": seed,
        "best_checkpoint": str(
            best_checkpoint
        ),
        "best_eval_loss": float(
            best_eval_loss
        ),
        "final_loaded_development_loss": float(
            final_development_metrics[
                "development_loss"
            ]
        ),
        "elapsed_minutes": float(
            elapsed_minutes
        ),
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "effective_batch_size": (
            TRAIN_BATCH_SIZE
            * GRADIENT_ACCUMULATION
        ),
    }

    metadata = {
        "seed": seed,
        "locked_protocol": True,
        "base_model": MODEL_NAME,
        "best_checkpoint": str(
            best_checkpoint
        ),
        "best_model_directory": str(
            best_model_directory
        ),
        "best_development_loss": float(
            best_eval_loss
        ),
        "final_loaded_development_metrics": {
            key: float(value)
            for key, value
            in final_development_metrics.items()
        },
        "elapsed_minutes": float(
            elapsed_minutes
        ),
        "training_history_path": str(
            history_path
        ),
        "summary_row": summary_row,
    }

    with open(
        metadata_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata,
            file,
            indent=2,
        )

    summary_rows.append(
        summary_row
    )

    print()
    print("Best checkpoint:", best_checkpoint)
    print(
        "Best development loss:",
        best_eval_loss,
    )
    print(
        "Loaded-best development loss:",
        final_development_metrics[
            "development_loss"
        ],
    )
    print(
        "Elapsed minutes:",
        round(elapsed_minutes, 2),
    )

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------------------
# Save summary
# ---------------------------------------------------------------------

summary = (
    pd.DataFrame(summary_rows)
    .sort_values("seed")
    .reset_index(drop=True)
)

summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print()
print("=" * 78)
print("ADDITIONAL-SEED TRAINING SUMMARY")
print("=" * 78)

print(
    summary.round(6).to_string(
        index=False
    )
)

print()
print("Saved training summary to:")
print(SUMMARY_PATH)

print()
print("Cell 36 completed successfully.")

In [ ]:
# CELL 37 — Generate and evaluate locked neural predictions for added seeds

import gc
import json
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_from_disk
from scipy.special import expit
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
CACHE_DIR = EXPERIMENT_DIR / "cache"
MODEL_ROOT = EXPERIMENT_DIR / "models"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

DOCUMENT_PATH = (
    SPLIT_DIR
    / "taskA_documents_with_token_lengths.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

SEEDS = [13, 21, 77, 101]

MIN_BOUNDARY_PROBABILITY = 0.05
MAX_SPAN_TOKENS = 16
NESTED_THRESHOLD = 0.97
NONOVERLAP_THRESHOLD = 0.81

SUMMARY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_neural_test_results.csv"
)

METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_additional_seed_neural_test_metadata.json"
)

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(str(value).casefold().split())


def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted = []
    intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        if not any(
            intervals_overlap(start, end, old_start, old_end)
            for old_start, old_end in intervals
        ):
            accepted.append(row.Index)
            intervals.append((start, end))

    return group.loc[accepted].copy()


def prf(predicted, gold):
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def evaluate(seed, variant, table, gold_by_domain):
    result = {
        "seed": seed,
        "variant": variant,
        "n_occurrence_predictions": int(len(table)),
    }

    for domain, prefix in [
        ("parkinson", "PA"),
        ("mental_health", "MH"),
    ]:
        subset = table.loc[
            table["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                subset["num_article"],
                subset["mention"],
            )
        )

        predicted_types = {
            normalize_type(value)
            for value in subset["mention"]
            if str(value).strip()
        }

        pair_scores = prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_scores = prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        result[f"{prefix}_micro_precision"] = pair_scores[0]
        result[f"{prefix}_micro_recall"] = pair_scores[1]
        result[f"{prefix}_micro_f1"] = pair_scores[2]

        result[f"{prefix}_type_precision"] = type_scores[0]
        result[f"{prefix}_type_recall"] = type_scores[1]
        result[f"{prefix}_type_f1"] = type_scores[2]

    result["avg_micro_f1"] = np.mean(
        [
            result["PA_micro_f1"],
            result["MH_micro_f1"],
        ]
    )

    result["avg_type_f1"] = np.mean(
        [
            result["PA_type_f1"],
            result["MH_type_f1"],
        ]
    )

    result["selection_score"] = np.mean(
        [
            result["avg_micro_f1"],
            result["avg_type_f1"],
        ]
    )

    return result


# ---------------------------------------------------------------------
# Load datasets and document texts
# ---------------------------------------------------------------------

dataset = load_from_disk(
    str(DATASET_PATH)
)

test_windows = dataset["test"]

test_inputs = test_windows.select_columns(
    [
        "input_ids",
        "attention_mask",
        "token_type_ids",
    ]
)

documents = pd.read_csv(
    DOCUMENT_PATH
)

documents["num_article"] = normalize_article_id(
    documents["num_article"]
)

document_lookup = {
    (
        str(row.domain),
        str(row.num_article),
    ): str(row.text)
    for row in documents.itertuples(index=False)
}

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
    use_fast=True,
)

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# ---------------------------------------------------------------------
# Official test gold
# ---------------------------------------------------------------------

pa_path = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

mh_path = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(pa_path)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(mh_path)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = gold["term"].fillna("").astype(str)

split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = test_gold.loc[
        test_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(value)
            for value in subset["term"]
            if str(value).strip()
        },
    }

# ---------------------------------------------------------------------
# Decode helper
# ---------------------------------------------------------------------

def decode_candidate_pool(probabilities):
    rows = []

    for row_index in range(len(test_windows)):
        row = test_windows[row_index]

        domain = str(row["domain"])
        article = str(row["num_article"])
        window_index = int(row["window_index"])

        text = document_lookup[
            (domain, article)
        ]

        offsets = np.asarray(
            row["offset_mapping"],
            dtype=np.int64,
        )

        attention = np.asarray(
            row["attention_mask"],
            dtype=np.int8,
        )

        label_mask = np.asarray(
            row["label_mask"],
            dtype=np.int8,
        )

        length = len(offsets)

        p_start = probabilities[
            row_index,
            :length,
            0,
        ]

        p_end = probabilities[
            row_index,
            :length,
            1,
        ]

        valid = (
            (attention == 1)
            & (label_mask == 1)
            & (offsets[:, 1] > offsets[:, 0])
        )

        starts = np.flatnonzero(
            valid
            & (
                p_start
                >= MIN_BOUNDARY_PROBABILITY
            )
        )

        valid_ends = (
            valid
            & (
                p_end
                >= MIN_BOUNDARY_PROBABILITY
            )
        )

        for start_token in starts:
            possible_ends = np.arange(
                start_token,
                min(
                    length,
                    start_token
                    + MAX_SPAN_TOKENS,
                ),
            )

            possible_ends = possible_ends[
                valid_ends[possible_ends]
            ]

            for end_token in possible_ends:
                start_char = int(
                    offsets[start_token, 0]
                )

                end_char = int(
                    offsets[end_token, 1]
                )

                if end_char <= start_char:
                    continue

                mention = text[
                    start_char:end_char
                ]

                if not mention.strip():
                    continue

                start_probability = float(
                    p_start[start_token]
                )

                end_probability = float(
                    p_end[end_token]
                )

                rows.append(
                    {
                        "domain": domain,
                        "num_article": article,
                        "doi": str(row["doi"]),
                        "split": "test",
                        "window_index": window_index,
                        "start_token": int(start_token),
                        "end_token": int(end_token),
                        "span_token_count": int(
                            end_token
                            - start_token
                            + 1
                        ),
                        "start_char": start_char,
                        "end_char": end_char,
                        "mention": mention,
                        "term_key": normalize_type(
                            mention
                        ),
                        "p_start": start_probability,
                        "p_end": end_probability,
                        "p_term": float(
                            np.sqrt(
                                start_probability
                                * end_probability
                            )
                        ),
                    }
                )

    candidates = pd.DataFrame(rows)

    candidates = candidates.sort_values(
        [
            "domain",
            "num_article",
            "start_char",
            "end_char",
            "p_term",
            "p_start",
            "p_end",
            "window_index",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            False,
            False,
            True,
        ],
        kind="mergesort",
    )

    candidates = (
        candidates.drop_duplicates(
            [
                "domain",
                "num_article",
                "start_char",
                "end_char",
            ],
            keep="first",
        )
        .reset_index(drop=True)
    )

    candidates.insert(
        0,
        "candidate_id",
        np.arange(
            len(candidates),
            dtype=np.int64,
        ),
    )

    return candidates


# ---------------------------------------------------------------------
# Evaluate added seeds
# ---------------------------------------------------------------------

result_rows = []
generation_metadata = {}

for seed in SEEDS:
    print()
    print("=" * 78)
    print(f"TEST INFERENCE — SEED {seed}")
    print("=" * 78)

    model_path = (
        MODEL_ROOT
        / f"pubmedbert_boundary_seed{seed}/best_model"
    )

    if not model_path.exists():
        raise FileNotFoundError(
            f"Best model not found: {model_path}"
        )

    model = (
        AutoModelForTokenClassification
        .from_pretrained(
            str(model_path)
        )
    )

    inference_arguments = TrainingArguments(
        output_dir=str(
            CACHE_DIR
            / f"inference_seed{seed}"
        ),
        per_device_eval_batch_size=16,
        bf16=bool(
            torch.cuda.is_available()
            and torch.cuda.is_bf16_supported()
        ),
        fp16=bool(
            torch.cuda.is_available()
            and not torch.cuda.is_bf16_supported()
        ),
        report_to=[],
        dataloader_num_workers=0,
    )

    inference_trainer = Trainer(
        model=model,
        args=inference_arguments,
        data_collator=collator,
    )

    prediction_output = inference_trainer.predict(
        test_inputs
    )

    logits = prediction_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilities = expit(
        np.asarray(
            logits,
            dtype=np.float64,
        )
    )

    logits_path = (
        PREDICTION_DIR
        / f"pubmedbert_boundary_seed{seed}_test_logits.npz"
    )

    np.savez_compressed(
        logits_path,
        logits=np.asarray(
            logits,
            dtype=np.float32,
        ),
    )

    candidates = decode_candidate_pool(
        probabilities
    )

    pool_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_full_pairing_max16_candidate_pool.csv"
        )
    )

    candidates.to_csv(
        pool_path,
        index=False,
    )

    nested = candidates.loc[
        candidates["p_term"]
        >= NESTED_THRESHOLD
    ].copy()

    nonoverlap_parts = []

    eligible_nonoverlap_source = candidates.loc[
        candidates["p_term"] >= 0.50
    ]

    for _, document_group in (
        eligible_nonoverlap_source.groupby(
            ["domain", "num_article"],
            sort=False,
        )
    ):
        nonoverlap_parts.append(
            greedy_nonoverlap(
                document_group
            )
        )

    broad_nonoverlap = pd.concat(
        nonoverlap_parts,
        ignore_index=False,
    )

    nonoverlap = broad_nonoverlap.loc[
        broad_nonoverlap["p_term"]
        >= NONOVERLAP_THRESHOLD
    ].copy()

    nested_path = (
        PREDICTION_DIR
        / f"pubmedbert_boundary_seed{seed}_test_nested_neural.csv"
    )

    nonoverlap_path = (
        PREDICTION_DIR
        / f"pubmedbert_boundary_seed{seed}_test_nonoverlap_neural.csv"
    )

    nested.to_csv(
        nested_path,
        index=False,
    )

    nonoverlap.to_csv(
        nonoverlap_path,
        index=False,
    )

    result_rows.append(
        evaluate(
            seed,
            "NESTED_NEURAL",
            nested,
            gold_by_domain,
        )
    )

    result_rows.append(
        evaluate(
            seed,
            "NONOVERLAP_NEURAL",
            nonoverlap,
            gold_by_domain,
        )
    )

    generation_metadata[str(seed)] = {
        "candidate_pool": str(pool_path),
        "candidate_count": int(
            len(candidates)
        ),
        "nested_predictions": int(
            len(nested)
        ),
        "nonoverlap_predictions": int(
            len(nonoverlap)
        ),
        "logits": str(logits_path),
    }

    print("Candidate pool:", len(candidates))
    print("Nested predictions:", len(nested))
    print(
        "Non-overlap predictions:",
        len(nonoverlap),
    )

    del inference_trainer
    del model
    del logits
    del probabilities

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ---------------------------------------------------------------------
# Add existing seed-42 results
# ---------------------------------------------------------------------

seed42_path = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_locked_test_results.csv"
)

seed42 = pd.read_csv(
    seed42_path
)

seed42 = seed42.loc[
    seed42["variant"].isin(
        [
            "NESTED_NEURAL",
            "NONOVERLAP_NEURAL",
        ]
    )
].copy()

seed42.insert(
    0,
    "seed",
    42,
)

common_columns = sorted(
    set(seed42.columns)
    & set(pd.DataFrame(result_rows).columns)
)

all_results = pd.concat(
    [
        pd.DataFrame(result_rows),
        seed42[common_columns],
    ],
    ignore_index=True,
    sort=False,
)

all_results = all_results.sort_values(
    ["variant", "seed"]
).reset_index(drop=True)

all_results.to_csv(
    SUMMARY_PATH,
    index=False,
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "locked_thresholds": {
                "nested": NESTED_THRESHOLD,
                "nonoverlap": NONOVERLAP_THRESHOLD,
            },
            "test_used_for_additional_tuning": False,
            "additional_seed_artifacts": generation_metadata,
            "summary_path": str(SUMMARY_PATH),
        },
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("FIVE-SEED NEURAL TEST RESULTS")
print("=" * 78)

display_columns = [
    "seed",
    "variant",
    "n_occurrence_predictions",
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_f1",
    "MH_micro_f1",
]

print(
    all_results[display_columns]
    .round(4)
    .to_string(index=False)
)

print()
print("=" * 78)
print("MEAN ± STANDARD DEVIATION ACROSS SEEDS")
print("=" * 78)

aggregate = (
    all_results.groupby("variant")[
        [
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
            "PA_micro_f1",
            "MH_micro_f1",
        ]
    ]
    .agg(["mean", "std", "min", "max"])
)

print(
    aggregate.round(4).to_string()
)

print()
print("Saved five-seed results to:")
print(SUMMARY_PATH)

print()
print("Cell 37 completed successfully.")

In [ ]:
# CELL 38 — Score additional-seed test candidates with the locked collapse reference

import gc
import joblib
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_from_disk
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"

STATE_DIR = (
    EXPERIMENT_DIR
    / "models/collapse_anchor_comparison_seed42"
)

DATASET_PATH = (
    CACHE_DIR
    / "pubmedbert_start_end_windows_v1"
)

ENCODER_NAME = (
    "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"
)

SEEDS = [13, 21, 77, 101]
ELIGIBILITY_FLOOR = 0.50
BATCH_SIZE = 24

POLICIES = {
    "NESTED_HIGH_CONFIDENCE": {
        "score_column": "collapse_nested_anchor",
        "distance_column": "distance_nested_anchor",
    },
    "NONOVERLAP_HIGH_CONFIDENCE": {
        "score_column": "collapse_nonoverlap_anchor",
        "distance_column": "distance_nonoverlap_anchor",
    },
}

# ---------------------------------------------------------------------
# Load windows and frozen encoder
# ---------------------------------------------------------------------

dataset = load_from_disk(
    str(DATASET_PATH)
)

test_windows = dataset["test"]

window_lookup = {}

for window_row in range(len(test_windows)):
    row = test_windows[window_row]

    key = (
        str(row["domain"]),
        str(row["num_article"]),
        int(row["window_index"]),
    )

    if key in window_lookup:
        raise ValueError(
            f"Duplicate test-window key: {key}"
        )

    window_lookup[key] = window_row

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(
    ENCODER_NAME,
    use_fast=True,
)

encoder = AutoModel.from_pretrained(
    ENCODER_NAME
)

encoder.to(device)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

hidden_size = int(
    encoder.config.hidden_size
)

use_bf16 = bool(
    device.type == "cuda"
    and torch.cuda.is_bf16_supported()
)

print("=" * 78)
print("LOCKED COLLAPSE SCORING — ADDITIONAL SEEDS")
print("=" * 78)
print("Seeds:", SEEDS)
print("Encoder:", ENCODER_NAME)
print("Device:", device)
print("BF16:", use_bf16)

# ---------------------------------------------------------------------
# Encode and score each seed
# ---------------------------------------------------------------------

for seed in SEEDS:
    print()
    print("=" * 78)
    print(f"SEED {seed}")
    print("=" * 78)

    pool_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_full_pairing_max16_candidate_pool.csv"
        )
    )

    output_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_candidates_with_locked_collapse.csv"
        )
    )

    candidates = pd.read_csv(
        pool_path
    )

    candidates["num_article"] = (
        candidates["num_article"]
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    eligible = (
        candidates.loc[
            candidates["p_term"]
            >= ELIGIBILITY_FLOOR
        ]
        .reset_index(drop=True)
        .copy()
    )

    eligible.insert(
        0,
        "embedding_row",
        np.arange(
            len(eligible),
            dtype=np.int64,
        ),
    )

    candidate_window_rows = []

    for row in eligible.itertuples(index=False):
        key = (
            str(row.domain),
            str(row.num_article),
            int(row.window_index),
        )

        if key not in window_lookup:
            raise KeyError(
                f"Unknown test window: {key}"
            )

        candidate_window_rows.append(
            window_lookup[key]
        )

    eligible["_window_row"] = np.asarray(
        candidate_window_rows,
        dtype=np.int64,
    )

    candidates_by_window = {
        int(window_row): group.index.to_numpy(
            dtype=np.int64
        )
        for window_row, group in eligible.groupby(
            "_window_row",
            sort=False,
        )
    }

    embeddings = np.empty(
        (
            len(eligible),
            hidden_size,
        ),
        dtype=np.float32,
    )

    written = np.zeros(
        len(eligible),
        dtype=bool,
    )

    required_windows = sorted(
        candidates_by_window.keys()
    )

    for batch_start in tqdm(
        range(
            0,
            len(required_windows),
            BATCH_SIZE,
        ),
        desc=f"Embedding seed {seed}",
    ):
        batch_window_rows = required_windows[
            batch_start:
            batch_start + BATCH_SIZE
        ]

        batch = tokenizer.pad(
            {
                "input_ids": [
                    test_windows[index][
                        "input_ids"
                    ]
                    for index in batch_window_rows
                ],
                "attention_mask": [
                    test_windows[index][
                        "attention_mask"
                    ]
                    for index in batch_window_rows
                ],
                "token_type_ids": [
                    test_windows[index][
                        "token_type_ids"
                    ]
                    for index in batch_window_rows
                ],
            },
            padding=True,
            return_tensors="pt",
        )

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        with torch.inference_mode():
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=use_bf16,
            ):
                hidden = encoder(
                    **batch
                ).last_hidden_state

        hidden = (
            hidden.float()
            .cpu()
            .numpy()
        )

        for local_index, window_row in enumerate(
            batch_window_rows
        ):
            candidate_rows = candidates_by_window[
                window_row
            ]

            sequence_length = len(
                test_windows[
                    window_row
                ]["input_ids"]
            )

            window_hidden = hidden[
                local_index,
                :sequence_length,
            ]

            prefix = np.vstack(
                [
                    np.zeros(
                        (1, hidden_size),
                        dtype=np.float32,
                    ),
                    np.cumsum(
                        window_hidden,
                        axis=0,
                        dtype=np.float32,
                    ),
                ]
            )

            starts = eligible.loc[
                candidate_rows,
                "start_token",
            ].to_numpy(dtype=np.int64)

            ends = (
                eligible.loc[
                    candidate_rows,
                    "end_token",
                ].to_numpy(dtype=np.int64)
                + 1
            )

            span_sums = (
                prefix[ends]
                - prefix[starts]
            )

            span_lengths = (
                ends - starts
            ).astype(np.float32)

            embeddings[
                candidate_rows
            ] = (
                span_sums
                / span_lengths[:, None]
            )

            written[
                candidate_rows
            ] = True

    if not written.all():
        raise ValueError(
            f"Seed {seed}: not all embeddings were written."
        )

    if not np.isfinite(embeddings).all():
        raise ValueError(
            f"Seed {seed}: non-finite embeddings detected."
        )

    # Apply both frozen seed-42 training-derived states
    for policy, columns in POLICIES.items():
        eligible[
            columns["score_column"]
        ] = np.nan

        eligible[
            columns["distance_column"]
        ] = np.nan

        for domain in sorted(
            eligible["domain"].unique()
        ):
            state_path = (
                STATE_DIR
                / f"{policy.lower()}_{domain}_K100.joblib"
            )

            state = joblib.load(
                state_path
            )

            positions = np.flatnonzero(
                eligible["domain"].to_numpy()
                == domain
            )

            X = np.asarray(
                embeddings[positions],
                dtype=np.float32,
            )

            X_scaled = (
                state["scaler"]
                .transform(X)
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            labels = state[
                "kmeans"
            ].predict(X_scaled)

            centers = state[
                "kmeans"
            ].cluster_centers_.astype(
                np.float32,
                copy=False,
            )

            distances = np.linalg.norm(
                X_scaled - centers[labels],
                axis=1,
            ).astype(np.float32)

            temperature = float(
                state["temperature"]
            )

            collapse = np.exp(
                -distances / temperature
            ).astype(np.float32)

            eligible.loc[
                eligible["domain"] == domain,
                columns["distance_column"],
            ] = distances

            eligible.loc[
                eligible["domain"] == domain,
                columns["score_column"],
            ] = collapse

    eligible = eligible.drop(
        columns=["_window_row"]
    )

    required_score_columns = [
        "collapse_nested_anchor",
        "collapse_nonoverlap_anchor",
    ]

    if not np.isfinite(
        eligible[
            required_score_columns
        ].to_numpy(dtype=float)
    ).all():
        raise ValueError(
            f"Seed {seed}: invalid collapse values."
        )

    eligible.to_csv(
        output_path,
        index=False,
    )

    print("Eligible candidates:", len(eligible))
    print(
        "Candidates by domain:"
    )
    print(
        eligible.groupby("domain")
        .size()
        .to_string()
    )

    print(
        "Nested collapse median:",
        round(
            float(
                eligible[
                    "collapse_nested_anchor"
                ].median()
            ),
            4,
        ),
    )

    print(
        "Non-overlap collapse median:",
        round(
            float(
                eligible[
                    "collapse_nonoverlap_anchor"
                ].median()
            ),
            4,
        ),
    )

    print("Saved:", output_path)

    del embeddings
    del eligible
    del candidates

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print("Cell 38 completed successfully.")

In [ ]:
# CELL 39 — Five-seed evaluation of all four locked variants

import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

ADDITIONAL_SEEDS = [13, 21, 77, 101]

OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_all_variants_test_results.csv"
)

AGGREGATE_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_all_variants_summary.csv"
)

METADATA_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_all_variants_metadata.json"
)

# Locked parameters
NESTED_P_KEEP = 0.97
NESTED_ALPHA = 0.50
NESTED_P_MIN = 0.75
NESTED_TAU = 0.95

NONOVERLAP_P_KEEP = 0.81
NONOVERLAP_ALPHA = 0.80
NONOVERLAP_P_MIN = 0.75
NONOVERLAP_TAU = 0.65

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(str(value).casefold().split())


def overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    selected = []
    intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        if not any(
            overlap(start, end, old_start, old_end)
            for old_start, old_end in intervals
        ):
            selected.append(row.Index)
            intervals.append((start, end))

    return group.loc[selected].copy()


def prf(predicted, gold):
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def evaluate(seed, variant, predictions, gold_by_domain):
    result = {
        "seed": seed,
        "variant": variant,
        "n_occurrence_predictions": int(
            len(predictions)
        ),
    }

    for domain, prefix in [
        ("parkinson", "PA"),
        ("mental_health", "MH"),
    ]:
        subset = predictions.loc[
            predictions["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                subset["num_article"],
                subset["mention"],
            )
        )

        predicted_types = {
            normalize_type(value)
            for value in subset["mention"]
            if str(value).strip()
        }

        pair_metrics = prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_metrics = prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        result[f"{prefix}_n_unique_pairs"] = len(
            predicted_pairs
        )

        result[f"{prefix}_n_unique_types"] = len(
            predicted_types
        )

        for name, value in pair_metrics.items():
            result[
                f"{prefix}_micro_{name}"
            ] = value

        for name, value in type_metrics.items():
            result[
                f"{prefix}_type_{name}"
            ] = value

    result["avg_micro_f1"] = np.mean(
        [
            result["PA_micro_f1"],
            result["MH_micro_f1"],
        ]
    )

    result["avg_type_f1"] = np.mean(
        [
            result["PA_type_f1"],
            result["MH_type_f1"],
        ]
    )

    result["selection_score"] = np.mean(
        [
            result["avg_micro_f1"],
            result["avg_type_f1"],
        ]
    )

    return result


# ---------------------------------------------------------------------
# Official test gold
# ---------------------------------------------------------------------

pa_path = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

mh_path = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(pa_path)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(mh_path)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = gold["term"].fillna("").astype(str)

split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = test_gold.loc[
        test_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(value)
            for value in subset["term"]
            if str(value).strip()
        },
    }

# ---------------------------------------------------------------------
# Evaluate four variants for each additional seed
# ---------------------------------------------------------------------

result_rows = []
prediction_counts = {}

for seed in ADDITIONAL_SEEDS:
    print()
    print("=" * 78)
    print(f"LOCKED FOUR-VARIANT EVALUATION — SEED {seed}")
    print("=" * 78)

    scored_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_candidates_with_locked_collapse.csv"
        )
    )

    candidates = pd.read_csv(
        scored_path
    )

    candidates["num_article"] = normalize_article_id(
        candidates["num_article"]
    )

    candidates["mention"] = (
        candidates["mention"]
        .fillna("")
        .astype(str)
    )

    # Broad non-overlap set is constructed before threshold gating.
    nonoverlap_parts = []

    for _, document_group in candidates.groupby(
        ["domain", "num_article"],
        sort=False,
    ):
        nonoverlap_parts.append(
            greedy_nonoverlap(document_group)
        )

    broad_nonoverlap = pd.concat(
        nonoverlap_parts,
        ignore_index=False,
    ).copy()

    nested_neural = candidates.loc[
        candidates["p_term"]
        >= NESTED_P_KEEP
    ].copy()

    nonoverlap_neural = broad_nonoverlap.loc[
        broad_nonoverlap["p_term"]
        >= NONOVERLAP_P_KEEP
    ].copy()

    nested_combined = (
        NESTED_ALPHA
        * candidates["p_term"]
        + (1.0 - NESTED_ALPHA)
        * candidates["collapse_nested_anchor"]
    )

    nested_plus_collapse = candidates.loc[
        (
            candidates["p_term"]
            >= NESTED_P_KEEP
        )
        |
        (
            (
                candidates["p_term"]
                >= NESTED_P_MIN
            )
            & (
                nested_combined
                >= NESTED_TAU
            )
        )
    ].copy()

    nonoverlap_combined = (
        NONOVERLAP_ALPHA
        * broad_nonoverlap["p_term"]
        + (1.0 - NONOVERLAP_ALPHA)
        * broad_nonoverlap[
            "collapse_nonoverlap_anchor"
        ]
    )

    nonoverlap_plus_collapse = broad_nonoverlap.loc[
        (
            broad_nonoverlap["p_term"]
            >= NONOVERLAP_P_KEEP
        )
        |
        (
            (
                broad_nonoverlap["p_term"]
                >= NONOVERLAP_P_MIN
            )
            & (
                nonoverlap_combined
                >= NONOVERLAP_TAU
            )
        )
    ].copy()

    variants = {
        "NESTED_NEURAL": nested_neural,
        "NONOVERLAP_NEURAL": nonoverlap_neural,
        "NESTED_PLUS_COLLAPSE": nested_plus_collapse,
        "NONOVERLAP_PLUS_COLLAPSE": (
            nonoverlap_plus_collapse
        ),
    }

    prediction_counts[str(seed)] = {}

    for variant, predictions in variants.items():
        output_path = (
            PREDICTION_DIR
            / (
                f"pubmedbert_boundary_seed{seed}_"
                f"test_{variant.lower()}.csv"
            )
        )

        predictions.to_csv(
            output_path,
            index=False,
        )

        result_rows.append(
            evaluate(
                seed,
                variant,
                predictions,
                gold_by_domain,
            )
        )

        prediction_counts[str(seed)][variant] = int(
            len(predictions)
        )

        print(
            f"{variant:28s}",
            len(predictions),
        )

# ---------------------------------------------------------------------
# Add the existing locked seed-42 results
# ---------------------------------------------------------------------

seed42_path = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_locked_test_results.csv"
)

seed42 = pd.read_csv(
    seed42_path
)

seed42.insert(
    0,
    "seed",
    42,
)

additional_results = pd.DataFrame(
    result_rows
)

all_columns = sorted(
    set(additional_results.columns)
    | set(seed42.columns)
)

additional_results = additional_results.reindex(
    columns=all_columns
)

seed42 = seed42.reindex(
    columns=all_columns
)

all_results = pd.concat(
    [
        additional_results,
        seed42,
    ],
    ignore_index=True,
)

all_results = all_results.sort_values(
    ["variant", "seed"]
).reset_index(drop=True)

all_results.to_csv(
    OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Aggregate across seeds
# ---------------------------------------------------------------------

metric_columns = [
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_f1",
    "MH_micro_f1",
    "PA_type_f1",
    "MH_type_f1",
]

aggregate_rows = []

for variant in sorted(
    all_results["variant"].unique()
):
    subset = all_results.loc[
        all_results["variant"] == variant
    ]

    for metric in metric_columns:
        values = subset[
            metric
        ].to_numpy(dtype=float)

        aggregate_rows.append(
            {
                "variant": variant,
                "metric": metric,
                "n_seeds": len(values),
                "mean": float(values.mean()),
                "standard_deviation": float(
                    values.std(ddof=1)
                ),
                "minimum": float(values.min()),
                "maximum": float(values.max()),
            }
        )

aggregate = pd.DataFrame(
    aggregate_rows
)

aggregate.to_csv(
    AGGREGATE_PATH,
    index=False,
)

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "seeds": [13, 21, 42, 77, 101],
            "test_used_for_additional_tuning": False,
            "collapse_reference": (
                "Common frozen seed-42 training-derived "
                "StandardScaler/KMeans states"
            ),
            "locked_parameters": {
                "nested": {
                    "p_keep": NESTED_P_KEEP,
                    "alpha": NESTED_ALPHA,
                    "p_min": NESTED_P_MIN,
                    "tau": NESTED_TAU,
                },
                "nonoverlap": {
                    "p_keep": NONOVERLAP_P_KEEP,
                    "alpha": NONOVERLAP_ALPHA,
                    "p_min": NONOVERLAP_P_MIN,
                    "tau": NONOVERLAP_TAU,
                },
            },
            "prediction_counts": prediction_counts,
            "result_path": str(OUTPUT_PATH),
            "aggregate_path": str(AGGREGATE_PATH),
        },
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("FIVE-SEED RESULTS — ALL VARIANTS")
print("=" * 78)

print(
    all_results[
        [
            "seed",
            "variant",
            "n_occurrence_predictions",
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("=" * 78)
print("FIVE-SEED MEAN ± STANDARD DEVIATION")
print("=" * 78)

for variant in sorted(
    all_results["variant"].unique()
):
    print()
    print(variant)

    subset = aggregate.loc[
        (aggregate["variant"] == variant)
        & aggregate["metric"].isin(
            [
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
            ]
        )
    ]

    for row in subset.itertuples():
        print(
            f"  {row.metric}: "
            f"{row.mean:.4f} ± "
            f"{row.standard_deviation:.4f} "
            f"[{row.minimum:.4f}, {row.maximum:.4f}]"
        )

print()
print("Saved complete results to:")
print(OUTPUT_PATH)

print()
print("Saved aggregate summary to:")
print(AGGREGATE_PATH)

print()
print("Cell 39 completed successfully.")

In [ ]:
# CELL 40 — Paired multi-seed contrasts and added-candidate audit

import itertools
import json
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import t

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

RESULT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_all_variants_test_results.csv"
)

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

CONTRAST_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_paired_contrasts.csv"
)

ADDITION_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_collapse_added_candidates.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_statistical_summary.json"
)

SEEDS = [13, 21, 42, 77, 101]

CONTRASTS = {
    "POSTPROCESSING": (
        "NONOVERLAP_NEURAL",
        "NESTED_NEURAL",
    ),
    "NESTED_COLLAPSE": (
        "NESTED_PLUS_COLLAPSE",
        "NESTED_NEURAL",
    ),
    "NONOVERLAP_COLLAPSE": (
        "NONOVERLAP_PLUS_COLLAPSE",
        "NONOVERLAP_NEURAL",
    ),
}

METRICS = [
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_f1",
    "MH_micro_f1",
    "PA_type_f1",
    "MH_type_f1",
]

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def exact_sign_flip_pvalue(differences):
    """
    Exact two-sided randomization test over all 2^n sign assignments.
    """
    differences = np.asarray(
        differences,
        dtype=float,
    )

    if np.allclose(differences, 0.0):
        return 1.0

    observed = abs(
        differences.mean()
    )

    permuted_statistics = []

    for signs in itertools.product(
        [-1.0, 1.0],
        repeat=len(differences),
    ):
        signs = np.asarray(
            signs,
            dtype=float,
        )

        permuted_statistics.append(
            abs(
                np.mean(
                    differences * signs
                )
            )
        )

    permuted_statistics = np.asarray(
        permuted_statistics
    )

    return float(
        np.mean(
            permuted_statistics
            >= observed - 1e-15
        )
    )


# ---------------------------------------------------------------------
# Load five-seed results
# ---------------------------------------------------------------------

results = pd.read_csv(
    RESULT_PATH
)

contrast_rows = []
difference_records = {}

for contrast_name, (
    treatment,
    baseline,
) in CONTRASTS.items():
    treatment_rows = (
        results.loc[
            results["variant"] == treatment
        ]
        .set_index("seed")
        .loc[SEEDS]
    )

    baseline_rows = (
        results.loc[
            results["variant"] == baseline
        ]
        .set_index("seed")
        .loc[SEEDS]
    )

    difference_records[
        contrast_name
    ] = {}

    for metric in METRICS:
        differences = (
            treatment_rows[metric]
            - baseline_rows[metric]
        ).to_numpy(dtype=float)

        n = len(differences)
        mean_difference = float(
            differences.mean()
        )

        standard_deviation = float(
            differences.std(ddof=1)
        )

        standard_error = (
            standard_deviation
            / np.sqrt(n)
        )

        critical_value = float(
            t.ppf(
                0.975,
                df=n - 1,
            )
        )

        ci_lower = (
            mean_difference
            - critical_value
            * standard_error
        )

        ci_upper = (
            mean_difference
            + critical_value
            * standard_error
        )

        sign_flip_p = exact_sign_flip_pvalue(
            differences
        )

        contrast_rows.append(
            {
                "contrast": contrast_name,
                "treatment": treatment,
                "baseline": baseline,
                "metric": metric,
                "n_seeds": n,
                "mean_difference": mean_difference,
                "standard_deviation": standard_deviation,
                "ci_lower_95_t": ci_lower,
                "ci_upper_95_t": ci_upper,
                "minimum_difference": float(
                    differences.min()
                ),
                "maximum_difference": float(
                    differences.max()
                ),
                "positive_seeds": int(
                    np.sum(differences > 0)
                ),
                "negative_seeds": int(
                    np.sum(differences < 0)
                ),
                "zero_seeds": int(
                    np.sum(
                        np.isclose(
                            differences,
                            0.0,
                            atol=1e-12,
                        )
                    )
                ),
                "exact_sign_flip_p_two_sided": (
                    sign_flip_p
                ),
            }
        )

        difference_records[
            contrast_name
        ][metric] = {
            str(seed): float(value)
            for seed, value in zip(
                SEEDS,
                differences,
            )
        }

contrasts = pd.DataFrame(
    contrast_rows
)

contrasts.to_csv(
    CONTRAST_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Audit candidates added by the non-overlap collapse gate
# ---------------------------------------------------------------------

pa_path = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

mh_path = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(pa_path)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(mh_path)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = gold["term"].fillna("").astype(str)

split_table = pd.read_csv(
    SPLIT_PATH
)

split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_pair_lookup = set(
    zip(
        test_gold["domain"],
        test_gold["num_article"],
        test_gold["term"],
    )
)

addition_rows = []

for seed in SEEDS:
    neural_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_nonoverlap_neural.csv"
        )
    )

    collapse_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_nonoverlap_plus_collapse.csv"
        )
    )

    neural = pd.read_csv(
        neural_path
    )

    collapse = pd.read_csv(
        collapse_path
    )

    neural["num_article"] = normalize_article_id(
        neural["num_article"]
    )

    collapse["num_article"] = normalize_article_id(
        collapse["num_article"]
    )

    neural_ids = set(
        neural["candidate_id"]
    )

    added = collapse.loc[
        ~collapse["candidate_id"].isin(
            neural_ids
        )
    ].copy()

    added["seed"] = seed

    added["exact_gold_pair"] = [
        (
            str(domain),
            str(article),
            str(mention),
        )
        in gold_pair_lookup
        for domain, article, mention in zip(
            added["domain"],
            added["num_article"],
            added["mention"],
        )
    ]

    addition_rows.append(
        added
    )

added_candidates = pd.concat(
    addition_rows,
    ignore_index=True,
)

added_candidates.to_csv(
    ADDITION_OUTPUT_PATH,
    index=False,
)

addition_summary = (
    added_candidates.groupby("seed")
    .agg(
        added_candidates=(
            "candidate_id",
            "size",
        ),
        correct_additions=(
            "exact_gold_pair",
            "sum",
        ),
    )
    .reset_index()
)

addition_summary["incorrect_additions"] = (
    addition_summary["added_candidates"]
    - addition_summary["correct_additions"]
)

addition_summary["addition_precision"] = (
    addition_summary["correct_additions"]
    / addition_summary["added_candidates"]
)

# ---------------------------------------------------------------------
# Save metadata
# ---------------------------------------------------------------------

metadata = {
    "seeds": SEEDS,
    "paired_unit": "neural initialization seed",
    "confidence_interval": (
        "Two-sided 95% Student-t interval for the mean paired "
        "difference across five seeds"
    ),
    "randomization_test": (
        "Exact two-sided sign-flip test over all 2^5 assignments"
    ),
    "minimum_attainable_nonzero_two_sided_p": 0.0625,
    "differences_by_seed": difference_records,
    "collapse_addition_summary": (
        addition_summary.to_dict(
            orient="records"
        )
    ),
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("PAIRED FIVE-SEED CONTRASTS")
print("=" * 78)

display = contrasts.loc[
    contrasts["metric"].isin(
        [
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    )
]

print(
    display[
        [
            "contrast",
            "metric",
            "mean_difference",
            "standard_deviation",
            "ci_lower_95_t",
            "ci_upper_95_t",
            "positive_seeds",
            "negative_seeds",
            "zero_seeds",
            "exact_sign_flip_p_two_sided",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print()
print("=" * 78)
print("NON-OVERLAP COLLAPSE ADDITIONS")
print("=" * 78)

print(
    addition_summary
    .round(4)
    .to_string(index=False)
)

print()
print("Saved paired contrasts to:")
print(CONTRAST_OUTPUT_PATH)

print()
print("Saved added-candidate audit to:")
print(ADDITION_OUTPUT_PATH)

print()
print("Saved statistical metadata to:")
print(METADATA_OUTPUT_PATH)

print()
print("Cell 40 completed successfully.")

In [ ]:
# CELL 41 — Length and nestedness subgroup analysis across five seeds

import re
import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

DETAIL_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_test_subgroup_results.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_test_subgroup_summary.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_test_subgroup_metadata.json"
)

SEEDS = [13, 21, 42, 77, 101]

VARIANTS = [
    "NESTED_NEURAL",
    "NONOVERLAP_NEURAL",
    "NESTED_PLUS_COLLAPSE",
    "NONOVERLAP_PLUS_COLLAPSE",
]

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def word_count(value):
    return len(
        str(value).strip().split()
    )


def boundary_contains(shorter, longer):
    """
    Case-sensitive lexical containment with alphanumeric boundaries.
    This matches the official exact-surface convention while avoiding
    substring artifacts such as 'AD' inside an unrelated word.
    """
    pattern = (
        r"(?<!\w)"
        + re.escape(str(shorter))
        + r"(?!\w)"
    )

    return bool(
        re.search(
            pattern,
            str(longer),
        )
    )


def safe_ratio(numerator, denominator):
    return (
        numerator / denominator
        if denominator
        else np.nan
    )


# ---------------------------------------------------------------------
# Load official test gold
# ---------------------------------------------------------------------

pa_path = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

mh_path = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(pa_path)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(mh_path)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = (
    gold["term"]
    .fillna("")
    .astype(str)
)

split_table = pd.read_csv(
    SPLIT_PATH
)

split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

test_gold["word_count"] = (
    test_gold["term"]
    .map(word_count)
)

test_gold["length_group"] = np.where(
    test_gold["word_count"] == 1,
    "single_word",
    "multiword",
)

# ---------------------------------------------------------------------
# Identify lexical nestedness within each document inventory
# ---------------------------------------------------------------------

test_gold["lexically_nested"] = False

for (_, _), group in test_gold.groupby(
    ["domain", "num_article"],
    sort=False,
):
    indices = group.index.to_list()
    terms = group["term"].to_dict()

    for index in indices:
        candidate = terms[index]

        nested = any(
            (
                other_index != index
                and len(terms[other_index]) > len(candidate)
                and boundary_contains(
                    candidate,
                    terms[other_index],
                )
            )
            for other_index in indices
        )

        test_gold.loc[
            index,
            "lexically_nested",
        ] = nested

test_gold["nestedness_group"] = np.where(
    test_gold["lexically_nested"],
    "lexically_nested",
    "not_lexically_nested",
)

print("=" * 78)
print("TEST GOLD SUBGROUPS")
print("=" * 78)

print(
    test_gold.groupby(
        [
            "domain",
            "length_group",
        ]
    )
    .size()
    .to_string()
)

print()
print(
    test_gold.groupby(
        [
            "domain",
            "nestedness_group",
        ]
    )
    .size()
    .to_string()
)

# ---------------------------------------------------------------------
# Evaluate subgroup recall and length-specific precision
# ---------------------------------------------------------------------

rows = []

for seed in SEEDS:
    for variant in VARIANTS:
        prediction_path = (
            PREDICTION_DIR
            / (
                f"pubmedbert_boundary_seed{seed}_"
                f"test_{variant.lower()}.csv"
            )
        )

        predictions = pd.read_csv(
            prediction_path
        )

        predictions["num_article"] = (
            normalize_article_id(
                predictions["num_article"]
            )
        )

        predictions["mention"] = (
            predictions["mention"]
            .fillna("")
            .astype(str)
        )

        predictions["word_count"] = (
            predictions["mention"]
            .map(word_count)
        )

        predictions["length_group"] = np.where(
            predictions["word_count"] == 1,
            "single_word",
            "multiword",
        )

        for domain in [
            "parkinson",
            "mental_health",
        ]:
            domain_gold = test_gold.loc[
                test_gold["domain"] == domain
            ]

            domain_predictions = predictions.loc[
                predictions["domain"] == domain
            ]

            predicted_pairs = set(
                zip(
                    domain_predictions["num_article"],
                    domain_predictions["mention"],
                )
            )

            # Gold-based recall subgroups
            for grouping_variable, groups in [
                (
                    "length_group",
                    [
                        "single_word",
                        "multiword",
                    ],
                ),
                (
                    "nestedness_group",
                    [
                        "lexically_nested",
                        "not_lexically_nested",
                    ],
                ),
            ]:
                for subgroup in groups:
                    gold_subset = domain_gold.loc[
                        domain_gold[grouping_variable]
                        == subgroup
                    ]

                    gold_pairs = set(
                        zip(
                            gold_subset["num_article"],
                            gold_subset["term"],
                        )
                    )

                    true_positives = len(
                        predicted_pairs
                        & gold_pairs
                    )

                    rows.append(
                        {
                            "seed": seed,
                            "variant": variant,
                            "domain": domain,
                            "analysis": "gold_recall",
                            "grouping_variable": (
                                grouping_variable
                            ),
                            "subgroup": subgroup,
                            "numerator": true_positives,
                            "denominator": len(
                                gold_pairs
                            ),
                            "metric_value": safe_ratio(
                                true_positives,
                                len(gold_pairs),
                            ),
                        }
                    )

            # Prediction-based precision by term length
            for subgroup in [
                "single_word",
                "multiword",
            ]:
                prediction_subset = (
                    domain_predictions.loc[
                        domain_predictions[
                            "length_group"
                        ] == subgroup
                    ]
                )

                subgroup_predictions = set(
                    zip(
                        prediction_subset[
                            "num_article"
                        ],
                        prediction_subset[
                            "mention"
                        ],
                    )
                )

                matching_gold = set(
                    zip(
                        domain_gold.loc[
                            domain_gold[
                                "length_group"
                            ] == subgroup,
                            "num_article",
                        ],
                        domain_gold.loc[
                            domain_gold[
                                "length_group"
                            ] == subgroup,
                            "term",
                        ],
                    )
                )

                true_positives = len(
                    subgroup_predictions
                    & matching_gold
                )

                rows.append(
                    {
                        "seed": seed,
                        "variant": variant,
                        "domain": domain,
                        "analysis": (
                            "prediction_precision"
                        ),
                        "grouping_variable": (
                            "length_group"
                        ),
                        "subgroup": subgroup,
                        "numerator": true_positives,
                        "denominator": len(
                            subgroup_predictions
                        ),
                        "metric_value": safe_ratio(
                            true_positives,
                            len(
                                subgroup_predictions
                            ),
                        ),
                    }
                )

detail = pd.DataFrame(rows)

detail.to_csv(
    DETAIL_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Aggregate across domains within seed, then across seeds
# ---------------------------------------------------------------------

seed_level = (
    detail.groupby(
        [
            "seed",
            "variant",
            "analysis",
            "grouping_variable",
            "subgroup",
        ],
        as_index=False,
    )
    .agg(
        numerator=("numerator", "sum"),
        denominator=("denominator", "sum"),
    )
)

seed_level["metric_value"] = (
    seed_level["numerator"]
    / seed_level["denominator"]
)

summary = (
    seed_level.groupby(
        [
            "variant",
            "analysis",
            "grouping_variable",
            "subgroup",
        ],
        as_index=False,
    )
    .agg(
        mean=("metric_value", "mean"),
        standard_deviation=(
            "metric_value",
            "std",
        ),
        minimum=("metric_value", "min"),
        maximum=("metric_value", "max"),
    )
)

summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

metadata = {
    "lexical_nestedness_definition": (
        "A gold surface form is lexically nested when it occurs with "
        "alphanumeric boundaries inside a longer official gold surface "
        "form from the same document."
    ),
    "important_limitation": (
        "Because official DETECH Task A provides document-level term "
        "inventories rather than adjudicated occurrence offsets, this "
        "is lexical inventory nestedness, not definitive occurrence-level "
        "span nestedness."
    ),
    "seeds": SEEDS,
    "variants": VARIANTS,
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display central neural-policy comparison
# ---------------------------------------------------------------------

print()
print("=" * 78)
print("FIVE-SEED SUBGROUP SUMMARY")
print("=" * 78)

display = summary.loc[
    summary["variant"].isin(
        [
            "NESTED_NEURAL",
            "NONOVERLAP_NEURAL",
        ]
    )
]

print(
    display[
        [
            "variant",
            "analysis",
            "subgroup",
            "mean",
            "standard_deviation",
            "minimum",
            "maximum",
        ]
    ]
    .sort_values(
        [
            "analysis",
            "subgroup",
            "variant",
        ]
    )
    .round(4)
    .to_string(index=False)
)

print()
print("Saved detailed subgroup results to:")
print(DETAIL_OUTPUT_PATH)

print()
print("Saved subgroup summary to:")
print(SUMMARY_OUTPUT_PATH)

print()
print("Saved subgroup metadata to:")
print(METADATA_OUTPUT_PATH)

print()
print("Cell 41 completed successfully.")

In [ ]:
# CELL 42 — Matched-threshold post-processing sensitivity analysis

import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
DATA_ROOT = BASE_DIR / "data/taskA"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"
SPLIT_DIR = EXPERIMENT_DIR / "splits"

SPLIT_PATH = (
    SPLIT_DIR
    / "taskA_document_holdout_grouped_70_15_15.csv"
)

DETAIL_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_matched_threshold_sensitivity.csv"
)

SUMMARY_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_matched_threshold_summary.csv"
)

METADATA_OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_matched_threshold_metadata.json"
)

SEEDS = [13, 21, 42, 77, 101]
THRESHOLDS = [0.81, 0.97]
NONOVERLAP_SOURCE_FLOOR = 0.50

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def find_unique_file(root, filename):
    matches = list(root.rglob(filename))

    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename} under {root}"
        )

    return matches[0]


def normalize_article_id(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_type(value):
    return " ".join(str(value).casefold().split())


def intervals_overlap(a_start, a_end, b_start, b_end):
    return a_start < b_end and b_start < a_end


def greedy_nonoverlap(group):
    working = group.copy()

    working["_boundary_min"] = working[
        ["p_start", "p_end"]
    ].min(axis=1)

    working["_length"] = (
        working["end_char"] - working["start_char"]
    )

    working = working.sort_values(
        [
            "p_term",
            "_boundary_min",
            "_length",
            "start_char",
            "end_char",
            "candidate_id",
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True,
            True,
        ],
        kind="mergesort",
    )

    accepted = []
    accepted_intervals = []

    for row in working.itertuples():
        start = int(row.start_char)
        end = int(row.end_char)

        conflict = any(
            intervals_overlap(
                start,
                end,
                previous_start,
                previous_end,
            )
            for previous_start, previous_end
            in accepted_intervals
        )

        if not conflict:
            accepted.append(row.Index)
            accepted_intervals.append(
                (start, end)
            )

    return group.loc[accepted].copy()


def prf(predicted, gold):
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0.0
    )

    return precision, recall, f1


def evaluate(seed, policy, threshold, table, gold_by_domain):
    result = {
        "seed": seed,
        "policy": policy,
        "threshold": threshold,
        "n_occurrence_predictions": int(len(table)),
    }

    for domain, prefix in [
        ("parkinson", "PA"),
        ("mental_health", "MH"),
    ]:
        subset = table.loc[
            table["domain"] == domain
        ]

        predicted_pairs = set(
            zip(
                subset["num_article"],
                subset["mention"],
            )
        )

        predicted_types = {
            normalize_type(value)
            for value in subset["mention"]
            if str(value).strip()
        }

        pair_scores = prf(
            predicted_pairs,
            gold_by_domain[domain]["pairs"],
        )

        type_scores = prf(
            predicted_types,
            gold_by_domain[domain]["types"],
        )

        result[f"{prefix}_micro_precision"] = pair_scores[0]
        result[f"{prefix}_micro_recall"] = pair_scores[1]
        result[f"{prefix}_micro_f1"] = pair_scores[2]

        result[f"{prefix}_type_precision"] = type_scores[0]
        result[f"{prefix}_type_recall"] = type_scores[1]
        result[f"{prefix}_type_f1"] = type_scores[2]

    result["avg_micro_f1"] = np.mean(
        [
            result["PA_micro_f1"],
            result["MH_micro_f1"],
        ]
    )

    result["avg_type_f1"] = np.mean(
        [
            result["PA_type_f1"],
            result["MH_type_f1"],
        ]
    )

    result["selection_score"] = np.mean(
        [
            result["avg_micro_f1"],
            result["avg_type_f1"],
        ]
    )

    return result


# ---------------------------------------------------------------------
# Load official test gold
# ---------------------------------------------------------------------

pa_path = find_unique_file(
    DATA_ROOT,
    "taskA_parkinson_terms_train.csv",
)

mh_path = find_unique_file(
    DATA_ROOT,
    "taskA_mental_health_terms_train.csv",
)

pa_gold = pd.read_csv(pa_path)
pa_gold["domain"] = "parkinson"

mh_gold = pd.read_csv(mh_path)
mh_gold["domain"] = "mental_health"

gold = pd.concat(
    [pa_gold, mh_gold],
    ignore_index=True,
)

gold["num_article"] = normalize_article_id(
    gold["num_article"]
)

gold["term"] = gold["term"].fillna("").astype(str)

split_table = pd.read_csv(SPLIT_PATH)
split_table["num_article"] = normalize_article_id(
    split_table["num_article"]
)

test_documents = (
    split_table.loc[
        split_table["split"] == "test",
        ["domain", "num_article"],
    ]
    .drop_duplicates()
)

test_gold = (
    gold.merge(
        test_documents,
        on=["domain", "num_article"],
        how="inner",
    )
    [["domain", "num_article", "term"]]
    .drop_duplicates()
)

gold_by_domain = {}

for domain in ["parkinson", "mental_health"]:
    subset = test_gold.loc[
        test_gold["domain"] == domain
    ]

    gold_by_domain[domain] = {
        "pairs": set(
            zip(
                subset["num_article"],
                subset["term"],
            )
        ),
        "types": {
            normalize_type(value)
            for value in subset["term"]
            if str(value).strip()
        },
    }

# ---------------------------------------------------------------------
# Apply both policies at both thresholds
# ---------------------------------------------------------------------

rows = []

for seed in SEEDS:
    pool_path = (
        PREDICTION_DIR
        / (
            f"pubmedbert_boundary_seed{seed}_"
            "test_full_pairing_max16_candidate_pool.csv"
        )
    )

    candidates = pd.read_csv(
        pool_path
    )

    candidates["num_article"] = normalize_article_id(
        candidates["num_article"]
    )

    candidates["mention"] = (
        candidates["mention"]
        .fillna("")
        .astype(str)
    )

    nonoverlap_source = candidates.loc[
        candidates["p_term"]
        >= NONOVERLAP_SOURCE_FLOOR
    ].copy()

    nonoverlap_parts = []

    for _, document_group in nonoverlap_source.groupby(
        ["domain", "num_article"],
        sort=False,
    ):
        nonoverlap_parts.append(
            greedy_nonoverlap(document_group)
        )

    broad_nonoverlap = pd.concat(
        nonoverlap_parts,
        ignore_index=False,
    )

    for threshold in THRESHOLDS:
        nested = candidates.loc[
            candidates["p_term"]
            >= threshold
        ].copy()

        nonoverlap = broad_nonoverlap.loc[
            broad_nonoverlap["p_term"]
            >= threshold
        ].copy()

        rows.append(
            evaluate(
                seed,
                "NESTED",
                threshold,
                nested,
                gold_by_domain,
            )
        )

        rows.append(
            evaluate(
                seed,
                "NONOVERLAP",
                threshold,
                nonoverlap,
                gold_by_domain,
            )
        )

detail = pd.DataFrame(rows)

detail.to_csv(
    DETAIL_OUTPUT_PATH,
    index=False,
)

# ---------------------------------------------------------------------
# Aggregate and calculate paired differences
# ---------------------------------------------------------------------

metrics = [
    "avg_micro_f1",
    "avg_type_f1",
    "selection_score",
    "PA_micro_f1",
    "MH_micro_f1",
]

summary_rows = []

for threshold in THRESHOLDS:
    nested = (
        detail.loc[
            (detail["policy"] == "NESTED")
            & (
                detail["threshold"]
                == threshold
            )
        ]
        .set_index("seed")
        .loc[SEEDS]
    )

    nonoverlap = (
        detail.loc[
            (detail["policy"] == "NONOVERLAP")
            & (
                detail["threshold"]
                == threshold
            )
        ]
        .set_index("seed")
        .loc[SEEDS]
    )

    for metric in metrics:
        nested_values = nested[
            metric
        ].to_numpy(dtype=float)

        nonoverlap_values = nonoverlap[
            metric
        ].to_numpy(dtype=float)

        differences = (
            nonoverlap_values
            - nested_values
        )

        summary_rows.append(
            {
                "threshold": threshold,
                "metric": metric,
                "nested_mean": float(
                    nested_values.mean()
                ),
                "nested_sd": float(
                    nested_values.std(ddof=1)
                ),
                "nonoverlap_mean": float(
                    nonoverlap_values.mean()
                ),
                "nonoverlap_sd": float(
                    nonoverlap_values.std(ddof=1)
                ),
                "mean_paired_difference": float(
                    differences.mean()
                ),
                "difference_sd": float(
                    differences.std(ddof=1)
                ),
                "minimum_difference": float(
                    differences.min()
                ),
                "maximum_difference": float(
                    differences.max()
                ),
                "positive_seeds": int(
                    np.sum(differences > 0)
                ),
                "negative_seeds": int(
                    np.sum(differences < 0)
                ),
            }
        )

summary = pd.DataFrame(
    summary_rows
)

summary.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
)

metadata = {
    "analysis_status": (
        "Post hoc matched-threshold sensitivity analysis conducted "
        "after inspection of the primary seed-42 test result."
    ),
    "purpose": (
        "Separate the effect of greedy non-overlap post-processing "
        "from the effect of independently tuned neural thresholds."
    ),
    "thresholds": THRESHOLDS,
    "seeds": SEEDS,
    "test_used_for_threshold_selection": False,
    "threshold_origin": (
        "The two thresholds were previously selected on development "
        "for the nested and non-overlap pipeline variants."
    ),
}

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("MATCHED-THRESHOLD POLICY RESULTS")
print("=" * 78)

print(
    detail[
        [
            "seed",
            "policy",
            "threshold",
            "n_occurrence_predictions",
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("=" * 78)
print("NONOVERLAP − NESTED AT MATCHED THRESHOLDS")
print("=" * 78)

print(
    summary.loc[
        summary["metric"].isin(
            [
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
            ]
        )
    ][
        [
            "threshold",
            "metric",
            "nested_mean",
            "nonoverlap_mean",
            "mean_paired_difference",
            "difference_sd",
            "minimum_difference",
            "maximum_difference",
            "positive_seeds",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("Saved detailed sensitivity results to:")
print(DETAIL_OUTPUT_PATH)

print()
print("Saved sensitivity summary to:")
print(SUMMARY_OUTPUT_PATH)

print()
print("Cell 42 completed successfully.")

In [ ]:
# CELL 43 — Development-only collapse sensitivity to cluster count

import time
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Paths and configuration
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

EXPERIMENT_DIR = BASE_DIR / "journal_experiments/taskA_holdout"
CACHE_DIR = EXPERIMENT_DIR / "cache"
PREDICTION_DIR = EXPERIMENT_DIR / "predictions"
RESULTS_DIR = EXPERIMENT_DIR / "results"

TRAIN_CANDIDATE_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_training_candidates_with_collapse.csv"
)

DEV_DIAGNOSTIC_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_diagnostic.csv"
)

DEV_K100_PATH = (
    PREDICTION_DIR
    / "pubmedbert_boundary_seed42_development_anchor_collapse_comparison.csv"
)

TRAIN_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_training_candidate_embeddings_p0.5.npy"
)

DEV_EMBEDDING_PATH = (
    CACHE_DIR
    / "pubmedbert_boundary_seed42_development_candidate_embeddings_p0.5.npy"
)

ANCHOR_MEMBERSHIP_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_training_collapse_anchor_membership.csv"
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_K_sensitivity.csv"
)

K_VALUES_TO_FIT = [50, 200]
SEED = 42

POLICIES = {
    "NESTED_HIGH_CONFIDENCE": (
        "collapse_nested_anchor"
    ),
    "NONOVERLAP_HIGH_CONFIDENCE": (
        "collapse_nonoverlap_anchor"
    ),
}

# ---------------------------------------------------------------------
# Load and validate
# ---------------------------------------------------------------------

training = pd.read_csv(
    TRAIN_CANDIDATE_PATH
)

development_labels = pd.read_csv(
    DEV_DIAGNOSTIC_PATH
)[
    ["candidate_id", "domain", "gold_compatible"]
].drop_duplicates(
    subset=["candidate_id"]
)

development_k100 = pd.read_csv(
    DEV_K100_PATH
)

membership = pd.read_csv(
    ANCHOR_MEMBERSHIP_PATH
)

train_embeddings = np.load(
    TRAIN_EMBEDDING_PATH,
    mmap_mode="r",
)

dev_embeddings = np.load(
    DEV_EMBEDDING_PATH,
    mmap_mode="r",
)

if len(training) != train_embeddings.shape[0]:
    raise ValueError(
        "Training candidates and embeddings are misaligned."
    )

if len(development_k100) != dev_embeddings.shape[0]:
    raise ValueError(
        "Development candidates and embeddings are misaligned."
    )

development = (
    development_k100.drop(
        columns=["gold_compatible"],
        errors="ignore",
    )
    .merge(
        development_labels,
        on=[
            "candidate_id",
            "domain",
        ],
        how="left",
        validate="one_to_one",
    )
)

if development["gold_compatible"].isna().any():
    raise ValueError(
        "Some development candidates lack gold-compatibility labels."
    )

development["gold_compatible"] = (
    development["gold_compatible"]
    .astype(int)
)

# ---------------------------------------------------------------------
# Evaluation helper
# ---------------------------------------------------------------------

def discrimination_row(
    policy,
    k,
    domain,
    labels,
    scores,
    anchors,
    temperature,
    fit_minutes,
):
    gold_scores = scores[
        labels == 1
    ]

    nongold_scores = scores[
        labels == 0
    ]

    return {
        "policy": policy,
        "K": k,
        "domain": domain,
        "training_anchors": anchors,
        "development_candidates": len(scores),
        "temperature": temperature,
        "fit_minutes": fit_minutes,
        "roc_auc": float(
            roc_auc_score(
                labels,
                scores,
            )
        ),
        "average_precision": float(
            average_precision_score(
                labels,
                scores,
            )
        ),
        "gold_median": float(
            np.median(gold_scores)
        ),
        "nongold_median": float(
            np.median(nongold_scores)
        ),
        "gold_minus_nongold_median": float(
            np.median(gold_scores)
            - np.median(nongold_scores)
        ),
    }


rows = []

# ---------------------------------------------------------------------
# Add already-computed K=100 results
# ---------------------------------------------------------------------

for policy, score_column in POLICIES.items():
    for domain in [
        "mental_health",
        "parkinson",
    ]:
        mask = (
            development["domain"]
            == domain
        )

        labels = development.loc[
            mask,
            "gold_compatible",
        ].to_numpy(dtype=int)

        scores = development.loc[
            mask,
            score_column,
        ].to_numpy(dtype=float)

        anchor_count = (
            membership.loc[
                (membership["anchor_policy"] == policy)
                & (
                    membership["domain"]
                    == domain
                ),
                "candidate_id",
            ]
            .nunique()
        )

        rows.append(
            discrimination_row(
                policy=policy,
                k=100,
                domain=domain,
                labels=labels,
                scores=scores,
                anchors=int(anchor_count),
                temperature=np.nan,
                fit_minutes=0.0,
            )
        )

    labels_all = development[
        "gold_compatible"
    ].to_numpy(dtype=int)

    scores_all = development[
        score_column
    ].to_numpy(dtype=float)

    rows.append(
        discrimination_row(
            policy=policy,
            k=100,
            domain="ALL",
            labels=labels_all,
            scores=scores_all,
            anchors=int(
                membership.loc[
                    membership[
                        "anchor_policy"
                    ] == policy,
                    "candidate_id",
                ].nunique()
            ),
            temperature=np.nan,
            fit_minutes=0.0,
        )
    )

# ---------------------------------------------------------------------
# Fit K=50 and K=200 using training anchors only
# ---------------------------------------------------------------------

for policy in POLICIES:
    policy_membership = membership.loc[
        membership["anchor_policy"]
        == policy
    ]

    for k in K_VALUES_TO_FIT:
        print()
        print("=" * 78)
        print(f"POLICY: {policy} | K={k}")
        print("=" * 78)

        combined_labels = []
        combined_scores = []
        total_anchors = 0
        total_minutes = 0.0

        for domain in [
            "mental_health",
            "parkinson",
        ]:
            anchor_ids = (
                policy_membership.loc[
                    policy_membership[
                        "domain"
                    ] == domain,
                    "candidate_id",
                ]
                .drop_duplicates()
                .astype(int)
                .to_numpy()
            )

            dev_positions = np.flatnonzero(
                development[
                    "domain"
                ].to_numpy() == domain
            )

            X_anchor = np.asarray(
                train_embeddings[
                    anchor_ids
                ],
                dtype=np.float32,
            )

            X_dev = np.asarray(
                dev_embeddings[
                    dev_positions
                ],
                dtype=np.float32,
            )

            scaler = StandardScaler()

            X_anchor_scaled = (
                scaler.fit_transform(
                    X_anchor
                )
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            X_dev_scaled = (
                scaler.transform(
                    X_dev
                )
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            kmeans = KMeans(
                n_clusters=k,
                init="k-means++",
                n_init=10,
                max_iter=300,
                tol=1e-4,
                random_state=SEED,
                algorithm="lloyd",
            )

            start_time = time.time()

            anchor_labels = (
                kmeans.fit_predict(
                    X_anchor_scaled
                )
            )

            fit_minutes = (
                time.time()
                - start_time
            ) / 60

            dev_labels = kmeans.predict(
                X_dev_scaled
            )

            centers = (
                kmeans.cluster_centers_
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            anchor_distances = np.linalg.norm(
                X_anchor_scaled
                - centers[anchor_labels],
                axis=1,
            )

            dev_distances = np.linalg.norm(
                X_dev_scaled
                - centers[dev_labels],
                axis=1,
            )

            q25, q75 = np.quantile(
                anchor_distances,
                [0.25, 0.75],
            )

            temperature = float(
                q75 - q25
            )

            collapse = np.exp(
                -dev_distances
                / temperature
            )

            labels = development.loc[
                development["domain"]
                == domain,
                "gold_compatible",
            ].to_numpy(dtype=int)

            rows.append(
                discrimination_row(
                    policy=policy,
                    k=k,
                    domain=domain,
                    labels=labels,
                    scores=collapse,
                    anchors=len(anchor_ids),
                    temperature=temperature,
                    fit_minutes=fit_minutes,
                )
            )

            combined_labels.append(
                labels
            )

            combined_scores.append(
                collapse
            )

            total_anchors += len(
                anchor_ids
            )

            total_minutes += (
                fit_minutes
            )

            print(
                f"{domain}: "
                f"AUC={roc_auc_score(labels, collapse):.4f}, "
                f"AP={average_precision_score(labels, collapse):.4f}, "
                f"minutes={fit_minutes:.2f}"
            )

        combined_labels = np.concatenate(
            combined_labels
        )

        combined_scores = np.concatenate(
            combined_scores
        )

        rows.append(
            discrimination_row(
                policy=policy,
                k=k,
                domain="ALL",
                labels=combined_labels,
                scores=combined_scores,
                anchors=total_anchors,
                temperature=np.nan,
                fit_minutes=total_minutes,
            )
        )

# ---------------------------------------------------------------------
# Save and display
# ---------------------------------------------------------------------

results = pd.DataFrame(
    rows
).sort_values(
    [
        "policy",
        "K",
        "domain",
    ]
)

results.to_csv(
    OUTPUT_PATH,
    index=False,
)

print()
print("=" * 78)
print("COLLAPSE K-SENSITIVITY — COMBINED DOMAINS")
print("=" * 78)

combined = results.loc[
    results["domain"] == "ALL"
]

print(
    combined[
        [
            "policy",
            "K",
            "training_anchors",
            "roc_auc",
            "average_precision",
            "gold_median",
            "nongold_median",
            "gold_minus_nongold_median",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("Saved K-sensitivity results to:")
print(OUTPUT_PATH)

print()
print("Cell 43 completed successfully.")

In [ ]:
# CELL 44 — Consolidate publication-ready Task A results

import json
import numpy as np
import pandas as pd

from pathlib import Path

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/MDPI Information/DETECH2026"
)

RESULTS_DIR = (
    BASE_DIR
    / "journal_experiments/taskA_holdout/results"
)

FIVE_SEED_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_all_variants_test_results.csv"
)

CONTRAST_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_paired_contrasts.csv"
)

BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_test_document_bootstrap_summary.csv"
)

MATCHED_THRESHOLD_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_matched_threshold_summary.csv"
)

SUBGROUP_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_test_subgroup_summary.csv"
)

K_SENSITIVITY_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_seed42_development_collapse_K_sensitivity.csv"
)

ADDITIONS_PATH = (
    RESULTS_DIR
    / "pubmedbert_boundary_five_seed_collapse_added_candidates.csv"
)

WORKBOOK_PATH = (
    RESULTS_DIR
    / "DETECH_TaskA_holdout_publication_results.xlsx"
)

JSON_PATH = (
    RESULTS_DIR
    / "DETECH_TaskA_holdout_publication_summary.json"
)

# ---------------------------------------------------------------------
# Load source tables
# ---------------------------------------------------------------------

five_seed = pd.read_csv(
    FIVE_SEED_PATH
)

contrasts = pd.read_csv(
    CONTRAST_PATH
)

bootstrap = pd.read_csv(
    BOOTSTRAP_PATH
)

matched = pd.read_csv(
    MATCHED_THRESHOLD_PATH
)

subgroups = pd.read_csv(
    SUBGROUP_PATH
)

k_sensitivity = pd.read_csv(
    K_SENSITIVITY_PATH
)

additions = pd.read_csv(
    ADDITIONS_PATH
)

# ---------------------------------------------------------------------
# Main five-seed table
# ---------------------------------------------------------------------

metric_columns = [
    "PA_micro_precision",
    "PA_micro_recall",
    "PA_micro_f1",
    "MH_micro_precision",
    "MH_micro_recall",
    "MH_micro_f1",
    "avg_micro_f1",
    "PA_type_f1",
    "MH_type_f1",
    "avg_type_f1",
    "selection_score",
]

main_rows = []

variant_order = [
    "NESTED_NEURAL",
    "NONOVERLAP_NEURAL",
    "NESTED_PLUS_COLLAPSE",
    "NONOVERLAP_PLUS_COLLAPSE",
]

for variant in variant_order:
    subset = five_seed.loc[
        five_seed["variant"] == variant
    ]

    row = {
        "variant": variant,
        "n_seeds": int(
            subset["seed"].nunique()
        ),
        "mean_occurrence_predictions": float(
            subset[
                "n_occurrence_predictions"
            ].mean()
        ),
    }

    for metric in metric_columns:
        row[f"{metric}_mean"] = float(
            subset[metric].mean()
        )

        row[f"{metric}_sd"] = float(
            subset[metric].std(ddof=1)
        )

    main_rows.append(row)

main_table = pd.DataFrame(
    main_rows
)

# ---------------------------------------------------------------------
# Central paired contrasts
# ---------------------------------------------------------------------

central_contrasts = contrasts.loc[
    contrasts["metric"].isin(
        [
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    )
].copy()

# ---------------------------------------------------------------------
# Seed-42 document-bootstrap matched contrasts
# ---------------------------------------------------------------------

bootstrap_contrasts = bootstrap.loc[
    (bootstrap["result_type"] == "paired_difference")
    & bootstrap["metric"].isin(
        [
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    )
].copy()

# ---------------------------------------------------------------------
# Matched-threshold sensitivity
# ---------------------------------------------------------------------

matched_main = matched.loc[
    matched["metric"].isin(
        [
            "avg_micro_f1",
            "avg_type_f1",
            "selection_score",
        ]
    )
].copy()

# ---------------------------------------------------------------------
# Subgroup results for the two neural policies
# ---------------------------------------------------------------------

subgroup_main = subgroups.loc[
    subgroups["variant"].isin(
        [
            "NESTED_NEURAL",
            "NONOVERLAP_NEURAL",
        ]
    )
].copy()

# ---------------------------------------------------------------------
# Collapse K-sensitivity combined-domain rows
# ---------------------------------------------------------------------

k_main = k_sensitivity.loc[
    k_sensitivity["domain"] == "ALL"
].copy()

# ---------------------------------------------------------------------
# Added-candidate summary
# ---------------------------------------------------------------------

addition_summary = (
    additions.groupby("seed")
    .agg(
        added_candidates=(
            "candidate_id",
            "size",
        ),
        correct_additions=(
            "exact_gold_pair",
            "sum",
        ),
    )
    .reset_index()
)

addition_summary["incorrect_additions"] = (
    addition_summary["added_candidates"]
    - addition_summary["correct_additions"]
)

addition_summary["addition_precision"] = (
    addition_summary["correct_additions"]
    / addition_summary["added_candidates"]
)

addition_total = pd.DataFrame(
    [
        {
            "seed": "ALL",
            "added_candidates": int(
                addition_summary[
                    "added_candidates"
                ].sum()
            ),
            "correct_additions": int(
                addition_summary[
                    "correct_additions"
                ].sum()
            ),
            "incorrect_additions": int(
                addition_summary[
                    "incorrect_additions"
                ].sum()
            ),
            "addition_precision": float(
                addition_summary[
                    "correct_additions"
                ].sum()
                / addition_summary[
                    "added_candidates"
                ].sum()
            ),
        }
    ]
)

addition_summary = pd.concat(
    [
        addition_summary,
        addition_total,
    ],
    ignore_index=True,
)

# ---------------------------------------------------------------------
# Save Excel workbook
# ---------------------------------------------------------------------

with pd.ExcelWriter(
    WORKBOOK_PATH,
    engine="openpyxl",
) as writer:
    main_table.to_excel(
        writer,
        sheet_name="Main five-seed results",
        index=False,
    )

    five_seed.to_excel(
        writer,
        sheet_name="Seed-level results",
        index=False,
    )

    central_contrasts.to_excel(
        writer,
        sheet_name="Paired seed contrasts",
        index=False,
    )

    bootstrap_contrasts.to_excel(
        writer,
        sheet_name="Document bootstrap",
        index=False,
    )

    matched_main.to_excel(
        writer,
        sheet_name="Matched thresholds",
        index=False,
    )

    subgroup_main.to_excel(
        writer,
        sheet_name="Subgroup analysis",
        index=False,
    )

    k_main.to_excel(
        writer,
        sheet_name="K sensitivity",
        index=False,
    )

    addition_summary.to_excel(
        writer,
        sheet_name="Collapse additions",
        index=False,
    )

# ---------------------------------------------------------------------
# Machine-readable summary and calibrated conclusions
# ---------------------------------------------------------------------

def variant_metric(variant, metric):
    subset = five_seed.loc[
        five_seed["variant"] == variant,
        metric,
    ]

    return {
        "mean": float(subset.mean()),
        "sd": float(subset.std(ddof=1)),
        "minimum": float(subset.min()),
        "maximum": float(subset.max()),
    }


summary = {
    "study": "DETECH Task A holdout ablation",
    "seeds": [13, 21, 42, 77, 101],

    "primary_variants": {
        variant: {
            metric: variant_metric(
                variant,
                metric,
            )
            for metric in [
                "avg_micro_f1",
                "avg_type_f1",
                "selection_score",
            ]
        }
        for variant in variant_order
    },

    "calibrated_findings": {
        "nonoverlap_postprocessing": {
            "finding": (
                "Large and consistent improvement over nested extraction."
            ),
            "mean_avg_micro_f1_difference": 0.231692,
            "mean_avg_type_f1_difference": 0.275413,
            "mean_selection_score_difference": 0.253552,
            "all_five_seeds_positive": True,
            "matched_thresholds_also_positive": True,
        },

        "nested_collapse": {
            "finding": (
                "No operational effect under the locked decision rule."
            ),
            "effect": 0.0,
            "identical_to_neural_baseline_in_all_seeds": True,
        },

        "nonoverlap_collapse": {
            "finding": (
                "No reliable improvement across neural seeds."
            ),
            "mean_avg_micro_f1_difference": 0.000058,
            "mean_avg_type_f1_difference": 0.000204,
            "mean_selection_score_difference": 0.000131,
            "selection_score_ci_95": [
                -0.001294,
                0.001556,
            ],
            "exact_sign_flip_p": 0.875,
            "added_candidates": 63,
            "correct_additions": 21,
            "addition_precision": 21 / 63,
        },

        "collapse_discrimination": {
            "finding": (
                "Centroid proximity was inversely associated with official "
                "gold compatibility on development."
            ),
            "tested_K": [50, 100, 200],
            "all_auc_below_chance": True,
            "interpretation": (
                "Dense embedding regions were dominated by generic or "
                "combinatorial candidate spans rather than valid terms."
            ),
        },

        "nestedness_tradeoff": {
            "nested_neural_recall": 0.8144,
            "nonoverlap_neural_recall": 0.6730,
            "interpretation": (
                "Non-overlap filtering improves precision substantially "
                "but sacrifices legitimate nested gold terms."
            ),
        },
    },

    "reporting_constraints": [
        (
            "Describe nested and non-overlap primary systems as "
            "development-optimized pipeline variants because they use "
            "different neural thresholds."
        ),
        (
            "Use the matched-threshold analysis as post hoc sensitivity "
            "evidence that overlap suppression itself is beneficial."
        ),
        (
            "Do not claim that the tested collapse signal improved Task A."
        ),
        (
            "Do not reinterpret inverse centroid distance as collapse; "
            "that would constitute a new exploratory mechanism."
        ),
        (
            "Use Cell 32 values as seed-42 point estimates and the "
            "document bootstrap only for uncertainty."
        ),
    ],
}

with open(
    JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
    )

# ---------------------------------------------------------------------
# Display
# ---------------------------------------------------------------------

print("=" * 78)
print("PUBLICATION RESULTS CONSOLIDATED")
print("=" * 78)

print()
print("Main five-seed results:")
print(
    main_table[
        [
            "variant",
            "avg_micro_f1_mean",
            "avg_micro_f1_sd",
            "avg_type_f1_mean",
            "avg_type_f1_sd",
            "selection_score_mean",
            "selection_score_sd",
        ]
    ]
    .round(4)
    .to_string(index=False)
)

print()
print("Collapse additions:")
print(
    addition_summary
    .round(4)
    .to_string(index=False)
)

print()
print("Saved publication workbook to:")
print(WORKBOOK_PATH)

print()
print("Saved machine-readable summary to:")
print(JSON_PATH)

print()
print("Cell 44 completed successfully.")